In [1]:
# ============================================================
# CELL 0 — Structured + Fold-Safe Prior OOF
# Environment, paths, and artifact discovery
# ============================================================

from pathlib import Path
import json
import hashlib
import platform
import sys

import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

MODERNBERT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local\modernbert_outputs"
    r"\modernbert_outputs_full\modernbert_outputs"
)

if not MODERNBERT_ROOT.exists():
    raise FileNotFoundError(
        f"ModernBERT output directory not found:\n{MODERNBERT_ROOT}"
    )

# Expected project root:
# D:\Competition\Trace-the-race-local
PROJECT_ROOT = MODERNBERT_ROOT.parents[2]

SCRATCH_ROOT = PROJECT_ROOT / "scratch_mastery_outputs"

CANONICAL_ROOT = (
    SCRATCH_ROOT
    / "01_data_foundation"
    / "03_integrity"
    / "canonical"
)

RESPONSES_PATH = CANONICAL_ROOT / "responses.parquet"
TURNS_PATH = CANONICAL_ROOT / "turns.parquet"
SESSIONS_PATH = CANONICAL_ROOT / "sessions.parquet"
OBJECTIVES_PATH = CANONICAL_ROOT / "objectives.parquet"


# ------------------------------------------------------------
# 2. REQUIRED PATH VALIDATION
# ------------------------------------------------------------

required_paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "MODERNBERT_ROOT": MODERNBERT_ROOT,
    "RESPONSES_PATH": RESPONSES_PATH,
    "TURNS_PATH": TURNS_PATH,
    "SESSIONS_PATH": SESSIONS_PATH,
    "OBJECTIVES_PATH": OBJECTIVES_PATH,
}

print("=" * 70)
print("TRACE THE RACE — CELL 0")
print("Structured + Fold-Safe Prior OOF")
print("=" * 70)

for name, path in required_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:20s}: {status}")
    print(f"  {path}")

missing = [
    str(path)
    for path in required_paths.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "\nMissing required artifacts:\n"
        + "\n".join(missing)
    )


# ------------------------------------------------------------
# 3. DISCOVER MODERNBERT OUTPUTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODERNBERT ARTIFACT DISCOVERY")
print("=" * 70)

all_modernbert_files = sorted(
    p for p in MODERNBERT_ROOT.rglob("*")
    if p.is_file()
)

print(f"Files discovered: {len(all_modernbert_files)}")

for p in all_modernbert_files[:100]:
    print(f"  {p.relative_to(MODERNBERT_ROOT)}")

if len(all_modernbert_files) > 100:
    print(f"  ... and {len(all_modernbert_files) - 100} more")


# ------------------------------------------------------------
# 4. LIKELY OOF ARTIFACTS
# ------------------------------------------------------------

oof_candidates = [
    p for p in all_modernbert_files
    if any(
        key in p.name.lower()
        for key in [
            "oof",
            "out_of_fold",
            "prediction",
            "pred",
        ]
    )
    and p.suffix.lower() in {
        ".parquet",
        ".csv",
        ".json",
        ".jsonl",
    }
]

print("\n" + "=" * 70)
print("POSSIBLE OOF / PREDICTION ARTIFACTS")
print("=" * 70)

if not oof_candidates:
    print("No OOF/prediction candidate found automatically.")
else:
    for p in oof_candidates:
        print(f"  {p.relative_to(MODERNBERT_ROOT)}")


# ------------------------------------------------------------
# 5. CANONICAL DATA SCHEMA CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CANONICAL SCHEMA CHECK")
print("=" * 70)

responses = pd.read_parquet(RESPONSES_PATH)

print(f"responses.parquet shape: {responses.shape}")

required_response_cols = {
    "response_id",
    "session_id",
    "objective_uid",
    "target",
    "fold",
}

missing_response_cols = (
    required_response_cols - set(responses.columns)
)

if missing_response_cols:
    raise ValueError(
        "responses.parquet missing columns:\n"
        f"{sorted(missing_response_cols)}"
    )

print("Required response columns: PASS")

print("\nFold distribution:")
print(
    responses["fold"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ------------------------------------------------------------
# 6. BASIC RESPONSE INTEGRITY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESPONSE INTEGRITY")
print("=" * 70)

assert responses["response_id"].is_unique, (
    "response_id is not unique"
)

assert responses["session_id"].notna().all(), (
    "session_id contains nulls"
)

assert responses["objective_uid"].notna().all(), (
    "objective_uid contains nulls"
)

assert responses["target"].isin([0, 1]).all(), (
    "target contains values outside {0,1}"
)

assert responses["fold"].isin([0, 1, 2, 3, 4]).all(), (
    "Unexpected fold values"
)

print(f"Rows              : {len(responses):,}")
print(f"Unique responses  : {responses['response_id'].nunique():,}")
print(f"Unique sessions   : {responses['session_id'].nunique():,}")
print(f"Unique objectives : {responses['objective_uid'].nunique():,}")
print(f"Positive labels   : {int(responses['target'].sum()):,}")
print(f"Negative labels   : {int((1 - responses['target']).sum()):,}")
print("Integrity          : PASS")


# ------------------------------------------------------------
# 7. SESSION-FOLD LEAKAGE CHECK
# ------------------------------------------------------------

session_fold_counts = (
    responses.groupby("session_id")["fold"]
    .nunique()
)

leaky_sessions = session_fold_counts[
    session_fold_counts > 1
]

print("\n" + "=" * 70)
print("SESSION / FOLD LEAKAGE CHECK")
print("=" * 70)

print(f"Sessions checked : {len(session_fold_counts):,}")
print(f"Leaky sessions   : {len(leaky_sessions):,}")

assert len(leaky_sessions) == 0, (
    "Session-fold leakage detected."
)

print("Session grouping : PASS")


# ------------------------------------------------------------
# 8. RUNTIME INFO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RUNTIME")
print("=" * 70)

print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()}")
print(f"Pandas   : {pd.__version__}")


# ------------------------------------------------------------
# 9. CELL-0 CONFIG
# ------------------------------------------------------------

CONFIG = {
    "project_root": str(PROJECT_ROOT),
    "modernbert_root": str(MODERNBERT_ROOT),
    "canonical_root": str(CANONICAL_ROOT),

    "responses_path": str(RESPONSES_PATH),
    "turns_path": str(TURNS_PATH),
    "sessions_path": str(SESSIONS_PATH),
    "objectives_path": str(OBJECTIVES_PATH),

    "n_rows": int(len(responses)),
    "n_sessions": int(responses["session_id"].nunique()),
    "n_objectives": int(responses["objective_uid"].nunique()),
    "n_folds": int(responses["fold"].nunique()),

    "cell_0_status": "PASS",
}

print("\n" + "=" * 70)
print("CELL 0 STATUS")
print("=" * 70)
print(json.dumps(CONFIG, indent=2))

print("\nCELL 0 COMPLETE — PASS")

TRACE THE RACE — CELL 0
Structured + Fold-Safe Prior OOF
PROJECT_ROOT        : FOUND
  D:\Competition\Trace-the-race-local
MODERNBERT_ROOT     : FOUND
  D:\Competition\Trace-the-race-local\modernbert_outputs\modernbert_outputs_full\modernbert_outputs
RESPONSES_PATH      : FOUND
  D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\responses.parquet
TURNS_PATH          : FOUND
  D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\turns.parquet
SESSIONS_PATH       : FOUND
  D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\sessions.parquet
OBJECTIVES_PATH     : FOUND
  D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\objectives.parquet

MODERNBERT ARTIFACT DISCOVERY
Files discovered: 81
  audit\cell0_gpu_bootstrap.json
  audit\cell1_dataset_contract.json
  audit\cell1_fold_summary.parqu

In [2]:
# ============================================================
# CELL 1 — ModernBERT OOF Artifact Discovery
# Identify and validate the exact prediction artifact
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. DISCOVER ALL TABULAR / JSON ARTIFACTS
# ------------------------------------------------------------

candidate_files = sorted(
    p for p in MODERNBERT_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {
        ".parquet",
        ".csv",
        ".json",
        ".jsonl",
    }
)

print("=" * 70)
print("TRACE THE RACE — CELL 1")
print("ModernBERT OOF Artifact Discovery")
print("=" * 70)

print(f"Candidate files: {len(candidate_files)}")

for i, p in enumerate(candidate_files, start=1):
    print(f"[{i:02d}] {p.relative_to(MODERNBERT_ROOT)}")


# ------------------------------------------------------------
# 2. SCORE FILES BY NAME
# ------------------------------------------------------------

prediction_keywords = {
    "oof": 10,
    "out_of_fold": 10,
    "prediction": 8,
    "predictions": 8,
    "pred": 5,
    "prob": 5,
    "probability": 5,
    "logit": 3,
}

negative_keywords = {
    "history": -5,
    "metric": -3,
    "config": -5,
    "contract": -5,
    "manifest": -5,
    "audit": -5,
    "summary": -3,
}

ranked_candidates = []

for p in candidate_files:
    name = p.stem.lower()

    score = 0

    for keyword, weight in prediction_keywords.items():
        if keyword in name:
            score += weight

    for keyword, weight in negative_keywords.items():
        if keyword in name:
            score += weight

    ranked_candidates.append(
        {
            "path": p,
            "score": score,
        }
    )

ranked_candidates = sorted(
    ranked_candidates,
    key=lambda x: (-x["score"], str(x["path"])),
)


# ------------------------------------------------------------
# 3. SHOW TOP CANDIDATES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP PREDICTION-LIKE CANDIDATES")
print("=" * 70)

for i, item in enumerate(ranked_candidates[:20], start=1):
    print(
        f"[{i:02d}] score={item['score']:>3} | "
        f"{item['path'].relative_to(MODERNBERT_ROOT)}"
    )


# ------------------------------------------------------------
# 4. INSPECT TABULAR SCHEMAS
# ------------------------------------------------------------

def inspect_tabular_file(path):
    """
    Return lightweight schema/sample information.
    Does NOT load the full dataset into memory unnecessarily.
    """

    suffix = path.suffix.lower()

    result = {
        "path": str(path),
        "suffix": suffix,
        "shape": None,
        "columns": None,
        "dtypes": None,
        "sample": None,
        "error": None,
    }

    try:
        if suffix == ".parquet":
            df = pd.read_parquet(path)

        elif suffix == ".csv":
            df = pd.read_csv(path, nrows=10_000)

        else:
            return result

        result["shape"] = df.shape
        result["columns"] = list(df.columns)
        result["dtypes"] = {
            c: str(df[c].dtype)
            for c in df.columns
        }

        result["sample"] = df.head(3)

    except Exception as exc:
        result["error"] = repr(exc)

    return result


# ------------------------------------------------------------
# 5. INSPECT TOP TABULAR CANDIDATES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CANDIDATE SCHEMA INSPECTION")
print("=" * 70)

candidate_inspections = []

for item in ranked_candidates[:15]:

    p = item["path"]

    if p.suffix.lower() not in {".parquet", ".csv"}:
        continue

    info = inspect_tabular_file(p)

    candidate_inspections.append(
        {
            "path": p,
            "score": item["score"],
            "info": info,
        }
    )

    print("\n" + "-" * 70)
    print(f"PATH: {p.relative_to(MODERNBERT_ROOT)}")
    print(f"SCORE: {item['score']}")

    if info["error"] is not None:
        print(f"ERROR: {info['error']}")
        continue

    print(f"SHAPE: {info['shape']}")
    print("COLUMNS:")

    for col in info["columns"]:
        print(f"  - {col}")

    print("\nSAMPLE:")
    print(info["sample"].to_string(index=False))


# ------------------------------------------------------------
# 6. DETECT POSSIBLE OOF CONTRACTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OOF CONTRACT DETECTION")
print("=" * 70)


def detect_oof_contract(columns, n_rows):
    """
    Detect whether a table structurally looks like an OOF
    response-level prediction table.

    This is intentionally conservative.
    """

    cols = {str(c).lower() for c in columns}

    id_candidates = {
        "response_id",
        "response",
        "id",
    }

    session_candidates = {
        "session_id",
        "session",
    }

    prediction_candidates = {
        "oof_pred",
        "oof_probability",
        "oof_prob",
        "prediction",
        "pred",
        "probability",
        "prob",
        "p",
    }

    target_candidates = {
        "target",
        "label",
        "is_correct",
        "y",
    }

    has_id = bool(cols & id_candidates)
    has_session = bool(cols & session_candidates)
    has_prediction = bool(cols & prediction_candidates)
    has_target = bool(cols & target_candidates)

    score = (
        3 * has_id
        + 2 * has_session
        + 4 * has_prediction
        + 1 * has_target
    )

    # Expected OOF population should be close to the
    # response-level training population.
    population_match = (
        abs(n_rows - len(responses)) <= 0
    )

    return {
        "contract_score": score,
        "has_response_id_like": has_id,
        "has_session_id": has_session,
        "has_prediction_like": has_prediction,
        "has_target_like": has_target,
        "row_count_matches_responses": population_match,
    }


contract_candidates = []

for item in candidate_inspections:

    info = item["info"]

    if info["error"] is not None:
        continue

    shape = info["shape"]

    if shape is None:
        continue

    n_rows = shape[0]

    contract = detect_oof_contract(
        info["columns"],
        n_rows,
    )

    contract_candidates.append(
        {
            "path": item["path"],
            "name_score": item["score"],
            **contract,
        }
    )


contract_candidates = sorted(
    contract_candidates,
    key=lambda x: (
        -x["contract_score"],
        -x["name_score"],
        str(x["path"]),
    ),
)


for i, item in enumerate(contract_candidates, start=1):

    print(
        f"[{i:02d}] "
        f"contract={item['contract_score']} "
        f"name={item['name_score']} | "
        f"rows_match={item['row_count_matches_responses']} | "
        f"{item['path'].relative_to(MODERNBERT_ROOT)}"
    )


# ------------------------------------------------------------
# 7. DO NOT AUTO-SELECT YET
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 1 DECISION")
print("=" * 70)

print(
    "No OOF artifact has been automatically locked."
)

print(
    "\nThe exact artifact must satisfy ALL of:"
)

print(
    "  1. Response-level population = 35,072"
)

print(
    "  2. Response/session identity available"
)

print(
    "  3. Fold information or reproducible fold linkage"
)

print(
    "  4. Prediction/probability or logit column"
)

print(
    "  5. Predictions are genuinely OOF, not training predictions"
)

print(
    "  6. No target-derived prediction artifact leakage"
)

print(
    "\nCELL 1 COMPLETE — DISCOVERY ONLY"
)

TRACE THE RACE — CELL 1
ModernBERT OOF Artifact Discovery
Candidate files: 71
[01] audit\cell0_gpu_bootstrap.json
[02] audit\cell1_dataset_contract.json
[03] audit\cell1_fold_summary.parquet
[04] audit\cell2_tokenization_contract.json
[05] audit\cell2_tokenization_sample.parquet
[06] audit\modernbert_5fold_fold_metrics.parquet
[07] engineering_fold_0\checkpoints\best\config.json
[08] engineering_fold_0\checkpoints\best\special_tokens_map.json
[09] engineering_fold_0\checkpoints\best\tokenizer.json
[10] engineering_fold_0\checkpoints\best\tokenizer_config.json
[11] engineering_fold_0\checkpoints\epoch_1\config.json
[12] engineering_fold_0\checkpoints\epoch_1\special_tokens_map.json
[13] engineering_fold_0\checkpoints\epoch_1\tokenizer.json
[14] engineering_fold_0\checkpoints\epoch_1\tokenizer_config.json
[15] engineering_fold_0\checkpoints\epoch_2\config.json
[16] engineering_fold_0\checkpoints\epoch_2\special_tokens_map.json
[17] engineering_fold_0\checkpoints\epoch_2\tokenizer.json
[1

In [3]:
# ==============================================================================
# TRACE THE RACE — CELL 2
# MODERNBERT OOF LOCK + EXACT RESPONSE / FOLD / TARGET ALIGNMENT
# ==============================================================================

from pathlib import Path
import hashlib
import json
import gc

import numpy as np
import pandas as pd


print("=" * 100)
print(
    "TRACE THE RACE — CELL 2"
)
print(
    "ModernBERT OOF Lock + Exact Alignment"
)
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "MODERNBERT_ROOT" in globals()
), (
    "Cell 0 dependency missing: MODERNBERT_ROOT."
)

assert (
    "RESPONSES_PATH" in globals()
), (
    "Cell 0 dependency missing: RESPONSES_PATH."
)

assert (
    Path(MODERNBERT_ROOT).exists()
), (
    f"ModernBERT root missing:\n{MODERNBERT_ROOT}"
)

assert (
    Path(RESPONSES_PATH).exists()
), (
    f"Responses artifact missing:\n{RESPONSES_PATH}"
)

print("\nCell 0 dependency : PASS")


# ==============================================================================
# 2. CANONICAL MODERNBERT OOF PATH
# ==============================================================================

MODERNBERT_OOF_PATH = (
    Path(MODERNBERT_ROOT)
    / "oof"
    / "modernbert_5fold_oof.parquet"
)

MODERNBERT_METRICS_PATH = (
    Path(MODERNBERT_ROOT)
    / "metrics"
    / "modernbert_5fold_oof_metrics.json"
)

MODERNBERT_MANIFEST_PATH = (
    Path(MODERNBERT_ROOT)
    / "manifests"
    / "modernbert_5fold_oof_manifest.json"
)


print("\n" + "=" * 100)
print("LOCKED MODERNBERT ARTIFACT")
print("=" * 100)

print(
    "OOF parquet :",
    MODERNBERT_OOF_PATH,
)

print(
    "Metrics     :",
    MODERNBERT_METRICS_PATH,
)

print(
    "Manifest    :",
    MODERNBERT_MANIFEST_PATH,
)


assert MODERNBERT_OOF_PATH.exists(), (
    "Canonical ModernBERT OOF parquet not found:\n"
    f"{MODERNBERT_OOF_PATH}"
)

assert MODERNBERT_METRICS_PATH.exists(), (
    "ModernBERT metrics JSON not found:\n"
    f"{MODERNBERT_METRICS_PATH}"
)

assert MODERNBERT_MANIFEST_PATH.exists(), (
    "ModernBERT OOF manifest not found:\n"
    f"{MODERNBERT_MANIFEST_PATH}"
)

print("OOF parquet : PASS")
print("Metrics     : PASS")
print("Manifest    : PASS")


# ==============================================================================
# 3. LOAD OOF
# ==============================================================================

modernbert_oof = pd.read_parquet(
    MODERNBERT_OOF_PATH,
)

print("\n" + "=" * 100)
print("MODERNBERT OOF SCHEMA")
print("=" * 100)

print(
    "Rows    :",
    len(modernbert_oof),
)

print(
    "Columns :",
    list(modernbert_oof.columns),
)


# ==============================================================================
# 4. REQUIRED OOF CONTRACT
# ==============================================================================

REQUIRED_OOF_COLUMNS = [
    "response_id",
    "session_id",
    "fold",
    "target",
    "prediction",
]

missing_oof_columns = [
    column
    for column in REQUIRED_OOF_COLUMNS
    if column not in modernbert_oof.columns
]

assert not missing_oof_columns, (
    "ModernBERT OOF missing required columns:\n"
    + "\n".join(missing_oof_columns)
)

print(
    "\nRequired OOF columns : PASS"
)


# ==============================================================================
# 5. EXACT POPULATION
# ==============================================================================

EXPECTED_ROWS = 35_072

assert (
    len(modernbert_oof)
    ==
    EXPECTED_ROWS
), (
    "Unexpected ModernBERT OOF population:\n"
    f"Expected: {EXPECTED_ROWS:,}\n"
    f"Observed: {len(modernbert_oof):,}"
)

print(
    "OOF population : PASS"
)


# ==============================================================================
# 6. RESPONSE IDENTITY
# ==============================================================================

assert (
    modernbert_oof["response_id"].notna().all()
), (
    "ModernBERT OOF contains null response_id."
)

assert (
    modernbert_oof["response_id"].is_unique
), (
    "ModernBERT OOF response_id is not unique."
)

print(
    "OOF response identity : PASS"
)


# ==============================================================================
# 7. SESSION IDENTITY
# ==============================================================================

assert (
    modernbert_oof["session_id"].notna().all()
), (
    "ModernBERT OOF contains null session_id."
)

print(
    "OOF session identity : PASS"
)


# ==============================================================================
# 8. FOLD CONTRACT
# ==============================================================================

assert (
    modernbert_oof["fold"]
    .notna()
    .all()
), (
    "ModernBERT OOF contains null fold."
)

assert (
    modernbert_oof["fold"]
    .isin([0, 1, 2, 3, 4])
    .all()
), (
    "ModernBERT OOF contains invalid fold values."
)

oof_fold_counts = (
    modernbert_oof["fold"]
    .value_counts()
    .sort_index()
)

print("\nOOF fold counts:")

for fold, count in oof_fold_counts.items():
    print(
        f"Fold {int(fold)} : {int(count):,}"
    )

print(
    "Five-fold contract : PASS"
)


# ==============================================================================
# 9. TARGET CONTRACT
# ==============================================================================

assert (
    modernbert_oof["target"]
    .notna()
    .all()
), (
    "ModernBERT OOF contains null targets."
)

assert (
    modernbert_oof["target"]
    .isin([0, 1])
    .all()
), (
    "ModernBERT OOF contains invalid target values."
)

print(
    "Target contract : PASS"
)


# ==============================================================================
# 10. PREDICTION CONTRACT
# ==============================================================================

prediction = pd.to_numeric(
    modernbert_oof["prediction"],
    errors="coerce",
)

assert (
    prediction.notna().all()
), (
    "ModernBERT OOF contains non-numeric predictions."
)

prediction_values = prediction.to_numpy(
    dtype=np.float64,
)

assert (
    np.isfinite(prediction_values).all()
), (
    "ModernBERT OOF contains non-finite predictions."
)

assert (
    (prediction_values >= 0.0).all()
), (
    "ModernBERT OOF contains prediction < 0."
)

assert (
    (prediction_values <= 1.0).all()
), (
    "ModernBERT OOF contains prediction > 1."
)

print(
    "Prediction numerical contract : PASS"
)

print(
    "Prediction min :",
    float(prediction_values.min()),
)

print(
    "Prediction max :",
    float(prediction_values.max()),
)


# ==============================================================================
# 11. LOAD CANONICAL RESPONSE TABLE
# ==============================================================================

responses_alignment = pd.read_parquet(
    RESPONSES_PATH,
    columns=[
        "response_id",
        "session_id",
        "objective_uid",
        "fold",
        "target",
    ],
)

print("\n" + "=" * 100)
print("CANONICAL RESPONSE ALIGNMENT TABLE")
print("=" * 100)

print(
    "Canonical rows :",
    len(responses_alignment),
)

assert (
    len(responses_alignment)
    ==
    EXPECTED_ROWS
), (
    "Canonical response population mismatch."
)

assert (
    responses_alignment["response_id"].is_unique
), (
    "Canonical response_id is not unique."
)

print(
    "Canonical response population : PASS"
)


# ==============================================================================
# 12. EXACT RESPONSE-ID SET MATCH
# ==============================================================================

oof_ids = set(
    modernbert_oof["response_id"]
)

canonical_ids = set(
    responses_alignment["response_id"]
)

missing_from_oof = canonical_ids - oof_ids
extra_in_oof = oof_ids - canonical_ids

assert not missing_from_oof, (
    "Canonical responses missing from ModernBERT OOF:\n"
    f"{list(missing_from_oof)[:20]}"
)

assert not extra_in_oof, (
    "ModernBERT OOF contains unknown response_ids:\n"
    f"{list(extra_in_oof)[:20]}"
)

print(
    "Exact response_id set match : PASS"
)


# ==============================================================================
# 13. EXACT SESSION / FOLD / TARGET ALIGNMENT
# ==============================================================================

canonical_lookup = (
    responses_alignment
    .set_index("response_id")[
        [
            "session_id",
            "fold",
            "target",
        ]
    ]
)

oof_check = (
    modernbert_oof
    .set_index("response_id")[
        [
            "session_id",
            "fold",
            "target",
        ]
    ]
    .loc[canonical_lookup.index]
)

session_mismatch = (
    oof_check["session_id"].astype(str)
    !=
    canonical_lookup["session_id"].astype(str)
)

fold_mismatch = (
    oof_check["fold"].astype(int)
    !=
    canonical_lookup["fold"].astype(int)
)

target_mismatch = (
    oof_check["target"].astype(int)
    !=
    canonical_lookup["target"].astype(int)
)

assert (
    int(session_mismatch.sum()) == 0
), (
    "Session alignment mismatch detected."
)

assert (
    int(fold_mismatch.sum()) == 0
), (
    "Fold alignment mismatch detected."
)

assert (
    int(target_mismatch.sum()) == 0
), (
    "Target alignment mismatch detected."
)

print(
    "Session alignment : PASS"
)

print(
    "Fold alignment    : PASS"
)

print(
    "Target alignment  : PASS"
)


# ==============================================================================
# 14. OOF METRICS CROSS-CHECK
# ==============================================================================

with open(
    MODERNBERT_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:

    modernbert_metrics = json.load(f)

print("\n" + "=" * 100)
print("MODERNBERT METRICS CROSS-CHECK")
print("=" * 100)

recorded_logloss = modernbert_metrics.get(
    "oof_log_loss"
)

recorded_auc = modernbert_metrics.get(
    "oof_roc_auc"
)

print(
    "Recorded OOF Log Loss :",
    recorded_logloss,
)

print(
    "Recorded OOF ROC-AUC  :",
    recorded_auc,
)

assert recorded_logloss is not None, (
    "oof_log_loss missing from metrics JSON."
)

assert recorded_auc is not None, (
    "oof_roc_auc missing from metrics JSON."
)

print(
    "Metrics artifact : PASS"
)


# ==============================================================================
# 15. SHA-256
# ==============================================================================

EXPECTED_OOF_SHA256 = (
    "d3b9fd29452d391fd938f00927cd81cb885afeb25116abbc688650d39ac1dfb3"
)

print("\n" + "=" * 100)
print("MODERNBERT OOF SHA-256")
print("=" * 100)

sha256 = hashlib.sha256()

with open(
    MODERNBERT_OOF_PATH,
    "rb",
) as f:

    while True:

        chunk = f.read(
            16 * 1024 * 1024
        )

        if not chunk:
            break

        sha256.update(chunk)

observed_oof_sha256 = sha256.hexdigest()

print(
    "Observed :",
    observed_oof_sha256,
)

print(
    "Expected :",
    EXPECTED_OOF_SHA256,
)

assert (
    observed_oof_sha256
    ==
    EXPECTED_OOF_SHA256
), (
    "ModernBERT OOF SHA-256 mismatch."
)

print(
    "OOF SHA-256 integrity : PASS"
)


# ==============================================================================
# 16. LOCKED MASTER OOF TABLE
# ==============================================================================

MODERNBERT_OOF_LOCKED = True

MODERNBERT_OOF_CONTRACT = {
    "path": str(MODERNBERT_OOF_PATH),
    "rows": int(len(modernbert_oof)),
    "unique_response_ids": int(
        modernbert_oof["response_id"].nunique()
    ),
    "sha256": observed_oof_sha256,
    "oof_log_loss": float(recorded_logloss),
    "oof_roc_auc": float(recorded_auc),
    "status": "LOCKED",
}

print("\n" + "=" * 100)
print("CELL 2 — FINAL STATUS")
print("=" * 100)

print(
    "ModernBERT OOF artifact : LOCKED"
)

print(
    "Rows                    : 35,072"
)

print(
    "Unique response IDs     : 35,072"
)

print(
    "Response alignment      : PASS"
)

print(
    "Session alignment       : PASS"
)

print(
    "Fold alignment          : PASS"
)

print(
    "Target alignment        : PASS"
)

print(
    "Prediction contract     : PASS"
)

print(
    "SHA-256 integrity       : PASS"
)

print(
    "OOF Log Loss            :",
    recorded_logloss,
)

print(
    "OOF ROC-AUC             :",
    recorded_auc,
)

print(
    "\n09B ModernBERT OOF import/lock : PASS"
)


# ==============================================================================
# 17. MEMORY CLEANUP
# ==============================================================================

del responses_alignment
del canonical_lookup
del oof_check
del prediction
del prediction_values
del oof_ids
del canonical_ids
del missing_from_oof
del extra_in_oof
del session_mismatch
del fold_mismatch
del target_mismatch
del sha256

gc.collect()

print(
    "Cell 2 memory cleanup : PASS"
)

TRACE THE RACE — CELL 2
ModernBERT OOF Lock + Exact Alignment

Cell 0 dependency : PASS

LOCKED MODERNBERT ARTIFACT
OOF parquet : D:\Competition\Trace-the-race-local\modernbert_outputs\modernbert_outputs_full\modernbert_outputs\oof\modernbert_5fold_oof.parquet
Metrics     : D:\Competition\Trace-the-race-local\modernbert_outputs\modernbert_outputs_full\modernbert_outputs\metrics\modernbert_5fold_oof_metrics.json
Manifest    : D:\Competition\Trace-the-race-local\modernbert_outputs\modernbert_outputs_full\modernbert_outputs\manifests\modernbert_5fold_oof_manifest.json
OOF parquet : PASS
Metrics     : PASS
Manifest    : PASS

MODERNBERT OOF SCHEMA
Rows    : 35072
Columns : ['response_id', 'session_id', 'fold', 'target', 'prediction']

Required OOF columns : PASS
OOF population : PASS
OOF response identity : PASS
OOF session identity : PASS

OOF fold counts:
Fold 0 : 6,958
Fold 1 : 7,050
Fold 2 : 7,023
Fold 3 : 7,081
Fold 4 : 6,960
Five-fold contract : PASS
Target contract : PASS
Prediction

In [4]:
# ==============================================================================
# TRACE THE RACE — CELL 3
# STRUCTURED FEATURE ARTIFACT DISCOVERY
# ==============================================================================

from pathlib import Path
import pandas as pd
import json


print("=" * 100)
print("TRACE THE RACE — CELL 3")
print("Structured Feature Artifact Discovery")
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    globals().get("MODERNBERT_OOF_LOCKED", False)
    is True
), (
    "ModernBERT OOF is not locked. "
    "Run Cell 2 successfully first."
)

print("\nModernBERT OOF dependency : PASS")


# ==============================================================================
# 2. STRUCTURED SEARCH ROOTS
# ==============================================================================

SEARCH_ROOTS = [
    PROJECT_ROOT / "scratch_mastery_outputs",
    PROJECT_ROOT / "scratch" / "_mastery_outputs",
]

SEARCH_ROOTS = [
    p for p in SEARCH_ROOTS
    if p.exists()
]

assert SEARCH_ROOTS, (
    "No scratch_mastery_outputs root found."
)

print("\n" + "=" * 100)
print("SEARCH ROOTS")
print("=" * 100)

for root in SEARCH_ROOTS:
    print("FOUND :", root)


# ==============================================================================
# 3. DISCOVER TABULAR ARTIFACTS
# ==============================================================================

tabular_files = []

for root in SEARCH_ROOTS:

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in {
            ".parquet",
            ".csv",
        }:
            continue

        tabular_files.append(path)


# Remove duplicates
tabular_files = sorted(
    set(tabular_files),
    key=lambda p: str(p).lower(),
)

print("\n" + "=" * 100)
print("TABULAR ARTIFACT DISCOVERY")
print("=" * 100)

print(
    "Total tabular artifacts:",
    len(tabular_files),
)


# ==============================================================================
# 4. NAME-BASED STRUCTURED FEATURE CANDIDATES
# ==============================================================================

STRUCTURED_KEYWORDS = [
    "feature",
    "features",
    "structured",
    "metadata",
    "response_feature",
    "response_features",
    "behavior",
    "behavioral",
    "linguistic",
    "style",
    "signal",
    "baseline",
]

EXCLUDE_KEYWORDS = [
    "modernbert",
    "oof",
    "tfidf",
    "retrieval",
    "dense",
    "sparse",
    "cross_encoder",
    "evidence_pack",
    "session_turn_index",
]

structured_candidates = []

for path in tabular_files:

    name = path.name.lower()
    full_name = str(path).lower()

    positive_hits = [
        keyword
        for keyword in STRUCTURED_KEYWORDS
        if keyword in name
    ]

    negative_hits = [
        keyword
        for keyword in EXCLUDE_KEYWORDS
        if keyword in full_name
    ]

    if positive_hits and not negative_hits:

        structured_candidates.append(
            {
                "path": path,
                "positive_hits": positive_hits,
                "negative_hits": negative_hits,
            }
        )


# ==============================================================================
# 5. PRINT CANDIDATES
# ==============================================================================

print("\n" + "=" * 100)
print("STRUCTURED FEATURE CANDIDATES")
print("=" * 100)

if not structured_candidates:

    print(
        "No clean structured-feature candidate "
        "was found by filename."
    )

else:

    for i, item in enumerate(
        structured_candidates,
        start=1,
    ):

        print(
            f"[{i:03d}] "
            f"{item['path']}"
        )

        print(
            "      keywords:",
            item["positive_hits"],
        )


# ==============================================================================
# 6. BROADER FEATURE CANDIDATE DISCOVERY
# ==============================================================================

print("\n" + "=" * 100)
print("BROADER FEATURE-LIKE ARTIFACTS")
print("=" * 100)

feature_like = []

for path in tabular_files:

    name = path.name.lower()

    if any(
        keyword in name
        for keyword in [
            "feature",
            "structured",
            "behavior",
            "linguistic",
            "style",
            "signal",
            "metadata",
        ]
    ):

        feature_like.append(path)


for i, path in enumerate(
    feature_like[:100],
    start=1,
):

    print(
        f"[{i:03d}] {path}"
    )

if len(feature_like) > 100:

    print(
        "...",
        len(feature_like) - 100,
        "more"
    )


# ==============================================================================
# 7. LIGHTWEIGHT SCHEMA INSPECTION
# ==============================================================================

print("\n" + "=" * 100)
print("SCHEMA INSPECTION")
print("=" * 100)

schema_records = []

for path in structured_candidates:

    path = path["path"]

    try:

        if path.suffix.lower() == ".parquet":

            parquet_file = pd.read_parquet(
                path,
                engine="pyarrow",
            )

            shape = parquet_file.shape
            columns = list(
                parquet_file.columns
            )

            dtypes = {
                c: str(
                    parquet_file[c].dtype
                )
                for c in parquet_file.columns
            }

            del parquet_file

        else:

            sample = pd.read_csv(
                path,
                nrows=1000,
            )

            shape = (
                1000,
                len(sample.columns),
            )

            columns = list(
                sample.columns
            )

            dtypes = {
                c: str(
                    sample[c].dtype
                )
                for c in sample.columns
            }

            del sample

        record = {
            "path": str(path),
            "rows_observed": shape[0],
            "columns": columns,
            "dtypes": dtypes,
            "error": None,
        }

    except Exception as exc:

        record = {
            "path": str(path),
            "rows_observed": None,
            "columns": None,
            "dtypes": None,
            "error": repr(exc),
        }

    schema_records.append(record)


for record in schema_records:

    print("\n" + "-" * 90)

    print(
        "PATH:",
        record["path"],
    )

    if record["error"]:

        print(
            "ERROR:",
            record["error"],
        )

        continue

    print(
        "Rows:",
        record["rows_observed"],
    )

    print(
        "Columns:"
    )

    for column in record["columns"]:

        print(
            f"  - {column}"
        )


# ==============================================================================
# 8. CELL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("CELL 3 STATUS")
print("=" * 100)

print(
    "ModernBERT OOF locked : PASS"
)

print(
    "Structured discovery  : COMPLETE"
)

print(
    "Training started      : NO"
)

print(
    "\nCELL 3 COMPLETE — PASS"
)

TRACE THE RACE — CELL 3
Structured Feature Artifact Discovery

ModernBERT OOF dependency : PASS

SEARCH ROOTS
FOUND : D:\Competition\Trace-the-race-local\scratch_mastery_outputs

TABULAR ARTIFACT DISCOVERY
Total tabular artifacts: 398

STRUCTURED FEATURE CANDIDATES
No clean structured-feature candidate was found by filename.

BROADER FEATURE-LIKE ARTIFACTS

SCHEMA INSPECTION

CELL 3 STATUS
ModernBERT OOF locked : PASS
Structured discovery  : COMPLETE
Training started      : NO

CELL 3 COMPLETE — PASS


In [5]:
# ==============================================================================
# TRACE THE RACE — CELL 4
# EVIDENCE / RETRIEVAL ARTIFACT DISCOVERY + EXACT SCHEMA LOCK
# ==============================================================================

from pathlib import Path
import gc
import json
import pandas as pd


print("=" * 100)
print("TRACE THE RACE — CELL 4")
print("Evidence / Retrieval Artifact Discovery + Exact Schema Lock")
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    globals().get("MODERNBERT_OOF_LOCKED", False) is True
), (
    "ModernBERT OOF is not locked. "
    "Run Cell 2 successfully first."
)

assert (
    "PROJECT_ROOT" in globals()
), (
    "PROJECT_ROOT missing from Cell 0."
)

assert (
    "SCRATCH_ROOT" in globals()
), (
    "SCRATCH_ROOT missing from Cell 0."
)

print("\nModernBERT OOF dependency : PASS")


# ==============================================================================
# 2. AUTHORITATIVE EVIDENCE PACK LOCATION
# ==============================================================================

EVIDENCE_ROOT = (
    SCRATCH_ROOT
    / "03_evidence_pack"
)

FROZEN_EVIDENCE_ROOT = (
    EVIDENCE_ROOT
    / "frozen"
)

FROZEN_EVIDENCE_PATH = (
    FROZEN_EVIDENCE_ROOT
    / "evidence_packs.parquet"
)

FROZEN_EVIDENCE_MANIFEST = (
    FROZEN_EVIDENCE_ROOT
    / "cell6_freeze_manifest.json"
)


print("\n" + "=" * 100)
print("FROZEN EVIDENCE ARTIFACT")
print("=" * 100)

print(
    "Evidence root :",
    EVIDENCE_ROOT,
)

print(
    "Frozen parquet:",
    FROZEN_EVIDENCE_PATH,
)

print(
    "Manifest      :",
    FROZEN_EVIDENCE_MANIFEST,
)

assert FROZEN_EVIDENCE_PATH.exists(), (
    "Frozen evidence parquet not found:\n"
    f"{FROZEN_EVIDENCE_PATH}"
)

assert FROZEN_EVIDENCE_MANIFEST.exists(), (
    "Frozen evidence manifest not found:\n"
    f"{FROZEN_EVIDENCE_MANIFEST}"
)

print("Frozen evidence parquet : PASS")
print("Frozen evidence manifest: PASS")


# ==============================================================================
# 3. FROZEN EVIDENCE SCHEMA
# ==============================================================================

frozen_evidence_schema = pd.read_parquet(
    FROZEN_EVIDENCE_PATH,
    engine="pyarrow",
)

print("\n" + "=" * 100)
print("FROZEN EVIDENCE SCHEMA")
print("=" * 100)

print(
    "Rows:",
    len(frozen_evidence_schema),
)

print(
    "Columns:"
)

for column in frozen_evidence_schema.columns:
    print(
        f"  - {column}"
    )


EXPECTED_FROZEN_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "objective_text",
    "evidence_text",
    "evidence_token_count",
    "selected_sections",
    "source_turn_uids",
    "source_turn_indices",
    "source_roles",
    "source_evidence_types",
    "source_turn_count",
    "source_min_turn_index",
    "source_max_turn_index",
    "has_student_evidence",
    "has_tutor_context",
    "has_final_student_evidence",
    "target",
]

missing_frozen_columns = [
    c
    for c in EXPECTED_FROZEN_COLUMNS
    if c not in frozen_evidence_schema.columns
]

assert not missing_frozen_columns, (
    "Frozen evidence schema missing columns:\n"
    + "\n".join(missing_frozen_columns)
)

print(
    "\nFrozen evidence schema : PASS"
)


# ==============================================================================
# 4. FROZEN POPULATION / IDENTITY
# ==============================================================================

assert (
    len(frozen_evidence_schema) == 35_072
), (
    "Frozen evidence population mismatch."
)

assert (
    frozen_evidence_schema["response_id"].is_unique
), (
    "Frozen evidence response_id is not unique."
)

assert (
    frozen_evidence_schema["fold"]
    .isin([0, 1, 2, 3, 4])
    .all()
), (
    "Frozen evidence contains invalid folds."
)

assert (
    frozen_evidence_schema["target"]
    .isin([0, 1])
    .all()
), (
    "Frozen evidence contains invalid targets."
)

print(
    "Frozen population : PASS"
)

print(
    "Frozen identity   : PASS"
)


# ==============================================================================
# 5. MANIFEST CROSS-CHECK
# ==============================================================================

with open(
    FROZEN_EVIDENCE_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    frozen_manifest = json.load(f)

print("\n" + "=" * 100)
print("FROZEN MANIFEST")
print("=" * 100)

print(
    "Status:",
    frozen_manifest.get("status"),
)

print(
    "Rows:",
    frozen_manifest.get("rows"),
)

print(
    "SHA-256:",
    frozen_manifest.get("sha256"),
)

assert (
    frozen_manifest.get("status") == "FROZEN"
), (
    "Frozen evidence manifest is not FROZEN."
)

assert (
    int(frozen_manifest.get("rows"))
    ==
    35_072
), (
    "Frozen manifest row count mismatch."
)

print(
    "Manifest contract : PASS"
)


# ==============================================================================
# 6. DISCOVER RETRIEVAL / TEMPORAL ARTIFACTS
# ==============================================================================

print("\n" + "=" * 100)
print("RETRIEVAL / TEMPORAL ARTIFACT DISCOVERY")
print("=" * 100)

all_tabular = []

for path in SCRATCH_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in {
        ".parquet",
        ".csv",
    }:
        continue

    all_tabular.append(path)

all_tabular = sorted(
    set(all_tabular),
    key=lambda p: str(p).lower(),
)

print(
    "Tabular artifacts discovered:",
    len(all_tabular),
)


# ==============================================================================
# 7. NAME-BASED RETRIEVAL CANDIDATES
# ==============================================================================

RETRIEVAL_KEYWORDS = [
    "retrieval",
    "candidate",
    "temporal",
    "evidence",
    "cross_encoder",
    "crossencoder",
    "r0",
    "r1",
    "r2",
    "r3",
    "session_turn",
]

RETRIEVAL_EXCLUDE = [
    "modernbert",
    "tfidf",
]

retrieval_candidates = []

for path in all_tabular:

    lower_path = str(path).lower()

    if any(
        keyword in lower_path
        for keyword in RETRIEVAL_EXCLUDE
    ):
        continue

    hits = [
        keyword
        for keyword in RETRIEVAL_KEYWORDS
        if keyword in lower_path
    ]

    if hits:

        retrieval_candidates.append(
            {
                "path": path,
                "hits": hits,
            }
        )


print("\nCandidate retrieval/evidence artifacts:")

for i, item in enumerate(
    retrieval_candidates,
    start=1,
):

    print(
        f"[{i:03d}] "
        f"{item['path']}"
    )

    print(
        "      hits:",
        item["hits"],
    )


# ==============================================================================
# 8. SCHEMA INSPECTION — CANDIDATES
# ==============================================================================

print("\n" + "=" * 100)
print("CANDIDATE SCHEMA INSPECTION")
print("=" * 100)


def inspect_parquet_schema(path):

    try:

        import pyarrow.parquet as pq

        metadata = pq.read_metadata(path)

        schema = pq.read_schema(path)

        return {
            "rows": metadata.num_rows,
            "columns": schema.names,
            "error": None,
        }

    except Exception as exc:

        return {
            "rows": None,
            "columns": None,
            "error": repr(exc),
        }


candidate_schema_records = []

for item in retrieval_candidates:

    path = item["path"]

    info = inspect_parquet_schema(path)

    candidate_schema_records.append(
        {
            "path": path,
            "hits": item["hits"],
            **info,
        }
    )


for record in candidate_schema_records:

    print("\n" + "-" * 90)

    print(
        "PATH:",
        record["path"],
    )

    print(
        "KEYWORDS:",
        record["hits"],
    )

    if record["error"]:

        print(
            "ERROR:",
            record["error"],
        )

        continue

    print(
        "ROWS:",
        record["rows"],
    )

    print(
        "COLUMNS:"
    )

    for column in record["columns"]:
        print(
            f"  - {column}"
        )


# ==============================================================================
# 9. IDENTIFY HIGH-VALUE TEMPORAL ARTIFACT
# ==============================================================================

temporal_candidates = []

for record in candidate_schema_records:

    columns = {
        str(c).lower()
        for c in (record["columns"] or [])
    }

    score = 0

    if "response_id" in columns:
        score += 3

    if "session_id" in columns:
        score += 3

    if "turn_uid" in columns:
        score += 4

    if "turn_index" in columns:
        score += 4

    if "role" in columns:
        score += 2

    if "text_norm" in columns:
        score += 2

    if "cross_encoder_score" in columns:
        score += 3

    if "evidence_source" in columns:
        score += 3

    if "selection_priority" in columns:
        score += 2

    if "canonical_role" in columns:
        score += 2

    temporal_candidates.append(
        {
            "path": record["path"],
            "rows": record["rows"],
            "columns": record["columns"],
            "score": score,
        }
    )


temporal_candidates = sorted(
    temporal_candidates,
    key=lambda x: (
        -x["score"],
        str(x["path"]),
    ),
)


print("\n" + "=" * 100)
print("HIGH-VALUE TEMPORAL / EVIDENCE CANDIDATES")
print("=" * 100)

for i, item in enumerate(
    temporal_candidates[:20],
    start=1,
):

    print(
        f"[{i:02d}] score={item['score']:>2} "
        f"rows={item['rows']} | "
        f"{item['path']}"
    )


# ==============================================================================
# 10. DO NOT LOCK UNKNOWN ARTIFACT
# ==============================================================================

print("\n" + "=" * 100)
print("CELL 4 DECISION")
print("=" * 100)

print(
    "Frozen evidence artifact : LOCKED"
)

print(
    "Retrieval/temporal search : COMPLETE"
)

print(
    "Automatic artifact lock   : NOT PERFORMED"
)

print(
    "\nReason:"
)

print(
    "The authoritative source must be selected from "
    "the discovered schema, not from filename alone."
)

print(
    "\nTraining started : NO"
)

print(
    "\nCELL 4 COMPLETE — PASS"
)


# ==============================================================================
# 11. MEMORY CLEANUP
# ==============================================================================

del frozen_evidence_schema
del all_tabular
del retrieval_candidates
del candidate_schema_records
del temporal_candidates

gc.collect()

print(
    "Cell 4 memory cleanup : PASS"
)

TRACE THE RACE — CELL 4
Evidence / Retrieval Artifact Discovery + Exact Schema Lock

ModernBERT OOF dependency : PASS

FROZEN EVIDENCE ARTIFACT
Evidence root : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack
Frozen parquet: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\evidence_packs.parquet
Manifest      : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\cell6_freeze_manifest.json
Frozen evidence parquet : PASS
Frozen evidence manifest: PASS

FROZEN EVIDENCE SCHEMA
Rows: 35072
Columns:
  - response_id
  - session_id
  - objective_uid
  - fold
  - objective_text
  - evidence_text
  - evidence_token_count
  - selected_sections
  - source_turn_uids
  - source_turn_indices
  - source_roles
  - source_evidence_types
  - source_turn_count
  - source_min_turn_index
  - source_max_turn_index
  - has_student_evidence
  - has_tutor_context
  - has_final_student_evidence
  - target

Frozen evide

In [8]:
# ==============================================================================
# TRACE THE RACE — CELL 5 REPAIR v2
# TEMPORAL EVIDENCE CONTRACT — NULL-SAFE
# ==============================================================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd


print("=" * 100)
print("TRACE THE RACE — CELL 5 REPAIR v2")
print("Temporal Evidence Contract — Null-Safe")
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY
# ==============================================================================

assert (
    "temporal_evidence" in globals()
), (
    "temporal_evidence is not available. "
    "Re-run the original Cell 5."
)

assert (
    globals().get("MODERNBERT_OOF_LOCKED", False)
    is True
), (
    "ModernBERT OOF is not locked."
)

print("\nModernBERT OOF dependency : PASS")

print(
    "Temporal evidence rows :",
    f"{len(temporal_evidence):,}"
)


# ==============================================================================
# 2. REQUIRED SCHEMA
# ==============================================================================

REQUIRED_TEMPORAL_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "turn_index",
    "role",
    "canonical_role",
    "text_norm",
    "evidence_source",
    "cross_encoder_rank",
    "cross_encoder_score",
    "selection_priority",
]

missing_columns = [
    c
    for c in REQUIRED_TEMPORAL_COLUMNS
    if c not in temporal_evidence.columns
]

assert not missing_columns, (
    "Required temporal columns missing:\n"
    + "\n".join(missing_columns)
)

print(
    "Schema contract : PASS"
)


# ==============================================================================
# 3. IDENTITY FIELDS
# ==============================================================================

IDENTITY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "turn_index",
]

for column in IDENTITY_COLUMNS:

    null_count = int(
        temporal_evidence[column]
        .isna()
        .sum()
    )

    assert null_count == 0, (
        f"Identity column {column} contains "
        f"{null_count:,} null values."
    )

print(
    "Identity fields : PASS"
)


# ==============================================================================
# 4. TEXT / SOURCE NULL AUDIT
#
# IMPORTANT:
# text_norm is NOT required to be non-null.
# We measure missingness instead of mutating the artifact.
# ==============================================================================

TEXT_NULL_COUNT = int(
    temporal_evidence["text_norm"]
    .isna()
    .sum()
)

SOURCE_NULL_COUNT = int(
    temporal_evidence["evidence_source"]
    .isna()
    .sum()
)

TEXT_NULL_RATE = (
    TEXT_NULL_COUNT
    /
    len(temporal_evidence)
)

SOURCE_NULL_RATE = (
    SOURCE_NULL_COUNT
    /
    len(temporal_evidence)
)


print("\n" + "=" * 100)
print("NULL AUDIT")
print("=" * 100)

print(
    "text_norm null rows      :",
    f"{TEXT_NULL_COUNT:,}",
)

print(
    "text_norm null rate      :",
    f"{TEXT_NULL_RATE:.6%}",
)

print(
    "evidence_source null rows:",
    f"{SOURCE_NULL_COUNT:,}",
)

print(
    "evidence_source null rate:",
    f"{SOURCE_NULL_RATE:.6%}",
)

print(
    "\nText/source null audit : RECORDED"
)


# ==============================================================================
# 5. TURN INDEX
# ==============================================================================

turn_index_numeric = pd.to_numeric(
    temporal_evidence["turn_index"],
    errors="coerce",
)

assert (
    turn_index_numeric.notna().all()
), (
    "Invalid turn_index values."
)

assert (
    (turn_index_numeric >= 0).all()
), (
    "Negative turn_index values."
)

print(
    "Turn index contract : PASS"
)


# ==============================================================================
# 6. FOLD
# ==============================================================================

assert (
    temporal_evidence["fold"]
    .isin([0, 1, 2, 3, 4])
    .all()
), (
    "Invalid fold values."
)

print(
    "Fold contract : PASS"
)


# ==============================================================================
# 7. OPTIONAL CROSS-ENCODER SCORE
# ==============================================================================

score_numeric = pd.to_numeric(
    temporal_evidence[
        "cross_encoder_score"
    ],
    errors="coerce",
)

score_non_null = (
    temporal_evidence[
        "cross_encoder_score"
    ].notna()
)

invalid_score = (
    score_non_null
    &
    score_numeric.isna()
)

assert not invalid_score.any(), (
    "Non-null cross_encoder_score values "
    "contain non-numeric data."
)

if score_non_null.any():

    present_scores = (
        score_numeric[score_non_null]
        .to_numpy(
            dtype=np.float64
        )
    )

    assert np.isfinite(
        present_scores
    ).all(), (
        "Non-finite cross_encoder_score found."
    )

    print(
        "Cross-encoder score : PASS"
    )

else:

    print(
        "Cross-encoder score : ABSENT"
    )


# ==============================================================================
# 8. OPTIONAL CROSS-ENCODER RANK
# ==============================================================================

rank_numeric = pd.to_numeric(
    temporal_evidence[
        "cross_encoder_rank"
    ],
    errors="coerce",
)

rank_non_null = (
    temporal_evidence[
        "cross_encoder_rank"
    ].notna()
)

invalid_rank = (
    rank_non_null
    &
    rank_numeric.isna()
)

assert not invalid_rank.any(), (
    "Non-null cross_encoder_rank values "
    "contain non-numeric data."
)

if rank_non_null.any():

    present_ranks = (
        rank_numeric[rank_non_null]
        .to_numpy(
            dtype=np.float64
        )
    )

    assert np.isfinite(
        present_ranks
    ).all(), (
        "Non-finite cross_encoder_rank found."
    )

    print(
        "Cross-encoder rank : PASS"
    )

else:

    print(
        "Cross-encoder rank : ABSENT"
    )


# ==============================================================================
# 9. RESPONSE COVERAGE
# ==============================================================================

temporal_response_ids = set(
    temporal_evidence[
        "response_id"
    ].astype(str)
)

modernbert_response_ids = set(
    modernbert_oof[
        "response_id"
    ].astype(str)
)

missing_responses = (
    modernbert_response_ids
    -
    temporal_response_ids
)

extra_temporal_responses = (
    temporal_response_ids
    -
    modernbert_response_ids
)

assert not missing_responses, (
    "ModernBERT OOF responses missing from "
    "temporal evidence:\n"
    + "\n".join(
        list(missing_responses)[:20]
    )
)

print(
    "ModernBERT response coverage : PASS"
)

print(
    "Responses covered :",
    f"{len(temporal_response_ids):,}"
)

print(
    "Extra temporal-only responses :",
    f"{len(extra_temporal_responses):,}"
)


# ==============================================================================
# 10. RESPONSE-FOLD CONSISTENCY
# ==============================================================================

response_fold_counts = (
    temporal_evidence
    .groupby("response_id")["fold"]
    .nunique()
)

assert (
    response_fold_counts.max()
    ==
    1
), (
    "A response appears in multiple folds."
)

print(
    "Response-fold consistency : PASS"
)


# ==============================================================================
# 11. SESSION-FOLD CONSISTENCY
# ==============================================================================

session_fold_counts = (
    temporal_evidence
    .groupby("session_id")["fold"]
    .nunique()
)

cross_fold_sessions = (
    session_fold_counts[
        session_fold_counts > 1
    ]
)

assert (
    len(cross_fold_sessions) == 0
), (
    "A session appears across multiple folds."
)

print(
    "Session-fold consistency : PASS"
)


# ==============================================================================
# 12. RESPONSE EVIDENCE CARDINALITY
# ==============================================================================

response_evidence_counts = (
    temporal_evidence
    .groupby("response_id")
    .size()
)

assert (
    response_evidence_counts.min() >= 1
), (
    "At least one response has zero temporal evidence rows."
)

print(
    "Evidence rows per response : PASS"
)

print(
    "Minimum rows / response :",
    int(
        response_evidence_counts.min()
    ),
)

print(
    "Maximum rows / response :",
    int(
        response_evidence_counts.max()
    ),
)


# ==============================================================================
# 13. LOCK
# ==============================================================================

TEMPORAL_EVIDENCE_LOCKED = True

TEMPORAL_EVIDENCE_CONTRACT = {
    "path": str(
        TEMPORAL_EVIDENCE_PATH
    ),
    "rows": int(
        len(temporal_evidence)
    ),
    "responses": int(
        len(temporal_response_ids)
    ),
    "text_norm_null_rows": int(
        TEXT_NULL_COUNT
    ),
    "text_norm_null_rate": float(
        TEXT_NULL_RATE
    ),
    "evidence_source_null_rows": int(
        SOURCE_NULL_COUNT
    ),
    "evidence_source_null_rate": float(
        SOURCE_NULL_RATE
    ),
    "cross_encoder_score_present": int(
        score_non_null.sum()
    ),
    "cross_encoder_rank_present": int(
        rank_non_null.sum()
    ),
    "status": "LOCKED",
}


# ==============================================================================
# 14. FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("TRACE THE RACE — CELL 5 FINAL STATUS")
print("=" * 100)

print(
    "Temporal evidence artifact : LOCKED"
)

print(
    "Rows                       :",
    f"{len(temporal_evidence):,}"
)

print(
    "Responses covered          :",
    f"{len(temporal_response_ids):,}"
)

print(
    "Identity fields            : PASS"
)

print(
    "Fold contract              : PASS"
)

print(
    "Response coverage          : PASS"
)

print(
    "Response-fold consistency  : PASS"
)

print(
    "Session-fold consistency   : PASS"
)

print(
    "Text nulls                 : RECORDED"
)

print(
    "Evidence-source nulls      : RECORDED"
)

print(
    "Cross-encoder fields       : VALIDATED"
)

print(
    "\nCELL 5 COMPLETE — PASS"
)


# ==============================================================================
# 15. CLEANUP
# ==============================================================================

del turn_index_numeric
del score_numeric
del rank_numeric
del score_non_null
del rank_non_null
del invalid_score
del invalid_rank
del temporal_response_ids
del modernbert_response_ids
del missing_responses
del extra_temporal_responses
del response_fold_counts
del session_fold_counts
del cross_fold_sessions
del response_evidence_counts

if "present_scores" in globals():
    del present_scores

if "present_ranks" in globals():
    del present_ranks

gc.collect()

print(
    "Cell 5 memory cleanup : PASS"
)

TRACE THE RACE — CELL 5 REPAIR v2
Temporal Evidence Contract — Null-Safe

ModernBERT OOF dependency : PASS
Temporal evidence rows : 1,063,637
Schema contract : PASS
Identity fields : PASS

NULL AUDIT
text_norm null rows      : 332,087
text_norm null rate      : 31.221836%
evidence_source null rows: 0
evidence_source null rate: 0.000000%

Text/source null audit : RECORDED
Turn index contract : PASS
Fold contract : PASS
Cross-encoder score : PASS
Cross-encoder rank : PASS
ModernBERT response coverage : PASS
Responses covered : 35,072
Extra temporal-only responses : 0
Response-fold consistency : PASS
Session-fold consistency : PASS
Evidence rows per response : PASS
Minimum rows / response : 8
Maximum rows / response : 41

TRACE THE RACE — CELL 5 FINAL STATUS
Temporal evidence artifact : LOCKED
Rows                       : 1,063,637
Responses covered          : 35,072
Identity fields            : PASS
Fold contract              : PASS
Response coverage          : PASS
Response-fold consist

In [9]:
# ==============================================================================
# TRACE THE RACE — CELL 6
# STRUCTURED RESPONSE-LEVEL FEATURE CONSTRUCTION
# ==============================================================================

from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd


print("=" * 100)
print("TRACE THE RACE — CELL 6")
print("Structured Response-Level Feature Construction")
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    globals().get(
        "MODERNBERT_OOF_LOCKED",
        False,
    )
    is True
), (
    "ModernBERT OOF is not locked."
)

assert (
    globals().get(
        "TEMPORAL_EVIDENCE_LOCKED",
        False,
    )
    is True
), (
    "Temporal evidence artifact is not locked."
)

assert (
    "temporal_evidence" in globals()
), (
    "temporal_evidence is missing."
)

assert (
    "modernbert_oof" in globals()
), (
    "modernbert_oof is missing."
)

print(
    "\nModernBERT OOF dependency : PASS"
)

print(
    "Temporal evidence dependency : PASS"
)


# ==============================================================================
# 2. OUTPUT PATHS
# ==============================================================================

STRUCTURED_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "structured_prior"
)

STRUCTURED_AUDIT_ROOT = (
    STRUCTURED_ROOT
    / "audit"
)

STRUCTURED_OUTPUT_ROOT = (
    STRUCTURED_ROOT
    / "outputs"
)

STRUCTURED_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STRUCTURED_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STRUCTURED_FEATURE_PATH = (
    STRUCTURED_OUTPUT_ROOT
    / "structured_response_features.parquet"
)

STRUCTURED_FEATURE_AUDIT_PATH = (
    STRUCTURED_AUDIT_ROOT
    / "cell6_structured_feature_audit.json"
)


# ==============================================================================
# 3. WORKING COPY
# ==============================================================================

df = temporal_evidence.copy()

print("\n" + "=" * 100)
print("INPUT")
print("=" * 100)

print(
    "Temporal rows:",
    f"{len(df):,}",
)

print(
    "Responses:",
    f"{df['response_id'].nunique():,}",
)


# ==============================================================================
# 4. SAFE NUMERIC COLUMNS
# ==============================================================================

df["_turn_index"] = pd.to_numeric(
    df["turn_index"],
    errors="coerce",
)

df["_ce_score"] = pd.to_numeric(
    df["cross_encoder_score"],
    errors="coerce",
)

df["_ce_rank"] = pd.to_numeric(
    df["cross_encoder_rank"],
    errors="coerce",
)

df["_selection_priority"] = pd.to_numeric(
    df["selection_priority"],
    errors="coerce",
)


# ==============================================================================
# 5. TEXT AVAILABILITY
#
# Do NOT replace missing text with fabricated content.
# ==============================================================================

df["_text_available"] = (
    df["text_norm"]
    .notna()
    .astype(np.int8)
)


# ==============================================================================
# 6. ROLE FLAGS
# ==============================================================================

role = (
    df["canonical_role"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

raw_role = (
    df["role"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df["_is_student"] = (
    role.eq("student")
    |
    raw_role.eq("student")
).astype(np.int8)

df["_is_tutor"] = (
    role.eq("tutor")
    |
    raw_role.eq("tutor")
).astype(np.int8)


# ==============================================================================
# 7. EVIDENCE SOURCE
# ==============================================================================

evidence_source = (
    df["evidence_source"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df["_has_evidence_source"] = (
    evidence_source.ne("")
    .astype(np.int8)
)


# ==============================================================================
# 8. RESPONSE-LEVEL AGGREGATION
# ==============================================================================

group = df.groupby(
    "response_id",
    sort=False,
)


features = group.agg(
    evidence_row_count=(
        "response_id",
        "size",
    ),

    unique_session_count=(
        "session_id",
        "nunique",
    ),

    unique_objective_count=(
        "objective_uid",
        "nunique",
    ),

    unique_turn_count=(
        "turn_uid",
        "nunique",
    ),

    min_turn_index=(
        "_turn_index",
        "min",
    ),

    max_turn_index=(
        "_turn_index",
        "max",
    ),

    mean_turn_index=(
        "_turn_index",
        "mean",
    ),

    mean_ce_score=(
        "_ce_score",
        "mean",
    ),

    max_ce_score=(
        "_ce_score",
        "max",
    ),

    min_ce_score=(
        "_ce_score",
        "min",
    ),

    median_ce_score=(
        "_ce_score",
        "median",
    ),

    mean_ce_rank=(
        "_ce_rank",
        "mean",
    ),

    min_ce_rank=(
        "_ce_rank",
        "min",
    ),

    max_ce_rank=(
        "_ce_rank",
        "max",
    ),

    mean_selection_priority=(
        "_selection_priority",
        "mean",
    ),

    max_selection_priority=(
        "_selection_priority",
        "max",
    ),

    text_available_count=(
        "_text_available",
        "sum",
    ),

    student_evidence_count=(
        "_is_student",
        "sum",
    ),

    tutor_evidence_count=(
        "_is_tutor",
        "sum",
    ),

    evidence_source_count=(
        "_has_evidence_source",
        "sum",
    ),
).reset_index()


# ==============================================================================
# 9. DERIVED STRUCTURAL FEATURES
# ==============================================================================

features["text_available_rate"] = (
    features["text_available_count"]
    /
    features["evidence_row_count"]
)

features["student_evidence_rate"] = (
    features["student_evidence_count"]
    /
    features["evidence_row_count"]
)

features["tutor_evidence_rate"] = (
    features["tutor_evidence_count"]
    /
    features["evidence_row_count"]
)

features["evidence_source_rate"] = (
    features["evidence_source_count"]
    /
    features["evidence_row_count"]
)

features["turn_span"] = (
    features["max_turn_index"]
    -
    features["min_turn_index"]
)

features["ce_score_range"] = (
    features["max_ce_score"]
    -
    features["min_ce_score"]
)

features["ce_rank_range"] = (
    features["max_ce_rank"]
    -
    features["min_ce_rank"]
)


# ==============================================================================
# 10. ADD RESPONSE-LEVEL FROZEN METADATA
# ==============================================================================

response_metadata = (
    df[
        [
            "response_id",
            "session_id",
            "objective_uid",
            "fold",
        ]
    ]
    .drop_duplicates(
        subset=["response_id"]
    )
)

assert (
    len(response_metadata)
    ==
    len(features)
), (
    "Response metadata cardinality mismatch."
)

features = features.merge(
    response_metadata,
    on="response_id",
    how="left",
    validate="one_to_one",
)


# ==============================================================================
# 11. MODERNBERT TARGET ALIGNMENT
#
# Target is copied for audit only.
# It is NOT used to calculate any feature.
# ==============================================================================

target_alignment = (
    modernbert_oof[
        [
            "response_id",
            "target",
        ]
    ]
    .drop_duplicates(
        subset=["response_id"]
    )
)

features = features.merge(
    target_alignment,
    on="response_id",
    how="left",
    validate="one_to_one",
)


# ==============================================================================
# 12. POPULATION CONTRACT
# ==============================================================================

assert (
    len(features)
    ==
    35_072
), (
    "Structured feature population mismatch."
)

assert (
    features["response_id"].is_unique
), (
    "Structured feature response_id is not unique."
)

assert (
    features["session_id"].notna().all()
), (
    "Structured features contain null session_id."
)

assert (
    features["objective_uid"].notna().all()
), (
    "Structured features contain null objective_uid."
)

assert (
    features["fold"]
    .isin([0, 1, 2, 3, 4])
    .all()
), (
    "Structured features contain invalid folds."
)

assert (
    features["target"]
    .isin([0, 1])
    .all()
), (
    "Structured features contain invalid targets."
)

print("\n" + "=" * 100)
print("POPULATION CONTRACT")
print("=" * 100)

print(
    "Rows:",
    f"{len(features):,}",
)

print(
    "Unique responses:",
    f"{features['response_id'].nunique():,}",
)

print(
    "Population : PASS"
)

print(
    "Response identity : PASS"
)

print(
    "Fold contract : PASS"
)

print(
    "Target alignment : PASS"
)


# ==============================================================================
# 13. FEATURE LIST
# ==============================================================================

IDENTITY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "target",
]

FEATURE_COLUMNS = [
    c
    for c in features.columns
    if c not in IDENTITY_COLUMNS
]


# ==============================================================================
# 14. FEATURE NUMERICAL CONTRACT
# ==============================================================================

non_numeric_features = []

for column in FEATURE_COLUMNS:

    if not pd.api.types.is_numeric_dtype(
        features[column]
    ):

        non_numeric_features.append(
            column
        )

assert not non_numeric_features, (
    "Non-numeric structured feature columns:\n"
    + "\n".join(
        non_numeric_features
    )
)

print(
    "Numeric feature contract : PASS"
)

print(
    "Feature count:",
    len(FEATURE_COLUMNS),
)


# ==============================================================================
# 15. FINITE CONTRACT
# ==============================================================================

feature_matrix = features[
    FEATURE_COLUMNS
].to_numpy(
    dtype=np.float64,
)

assert np.isfinite(
    feature_matrix
).all(), (
    "Structured feature matrix contains "
    "non-finite values."
)

print(
    "Feature finiteness : PASS"
)


# ==============================================================================
# 16. LEAKAGE GUARD
#
# No target-derived feature names/columns.
# target remains audit metadata only.
# ==============================================================================

TARGET_DERIVED_NAME_PATTERNS = [
    "target",
    "label",
    "y_true",
    "outcome",
]

for column in FEATURE_COLUMNS:

    lower = column.lower()

    assert not any(
        pattern in lower
        for pattern in TARGET_DERIVED_NAME_PATTERNS
    ), (
        "Potential target-derived feature detected:\n"
        f"{column}"
    )

print(
    "Target-derived feature name guard : PASS"
)


# ==============================================================================
# 17. SAVE FEATURE TABLE
# ==============================================================================

features.to_parquet(
    STRUCTURED_FEATURE_PATH,
    index=False,
)

assert (
    STRUCTURED_FEATURE_PATH.exists()
), (
    "Structured feature parquet was not created."
)

print(
    "\nStructured feature parquet : PASS"
)

print(
    "Path:",
    STRUCTURED_FEATURE_PATH,
)


# ==============================================================================
# 18. AUDIT ARTIFACT
# ==============================================================================

audit_payload = {
    "status": "PASS",
    "rows": int(len(features)),
    "unique_responses": int(
        features["response_id"].nunique()
    ),
    "feature_count": int(
        len(FEATURE_COLUMNS)
    ),
    "feature_columns": FEATURE_COLUMNS,
    "input_temporal_rows": int(
        len(df)
    ),
    "text_null_rows_in_source": int(
        df["text_norm"].isna().sum()
    ),
    "target_used_for_features": False,
    "modernbert_oof_used_for_feature_generation": False,
    "fold_column_preserved": True,
}


with open(
    STRUCTURED_FEATURE_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        audit_payload,
        f,
        indent=2,
    )


assert (
    STRUCTURED_FEATURE_AUDIT_PATH.exists()
)

print(
    "Feature audit JSON : PASS"
)


# ==============================================================================
# 19. FINAL STATUS
# ==============================================================================

STRUCTURED_FEATURES_READY = True

print("\n" + "=" * 100)
print("TRACE THE RACE — CELL 6 FINAL STATUS")
print("=" * 100)

print(
    "Structured feature rows :",
    f"{len(features):,}",
)

print(
    "Structured feature count:",
    len(FEATURE_COLUMNS),
)

print(
    "Target used in features : NO"
)

print(
    "ModernBERT prediction used to build features : NO"
)

print(
    "Population contract : PASS"
)

print(
    "Feature numerical contract : PASS"
)

print(
    "Feature finiteness : PASS"
)

print(
    "Leakage guard : PASS"
)

print(
    "\nCELL 6 COMPLETE — PASS"
)


# ==============================================================================
# 20. MEMORY CLEANUP
# ==============================================================================

del response_metadata
del target_alignment
del feature_matrix
del role
del raw_role
del evidence_source

del df

gc.collect()

print(
    "Cell 6 memory cleanup : PASS"
)

TRACE THE RACE — CELL 6
Structured Response-Level Feature Construction

ModernBERT OOF dependency : PASS
Temporal evidence dependency : PASS

INPUT
Temporal rows: 1,063,637
Responses: 35,072

POPULATION CONTRACT
Rows: 35,072
Unique responses: 35,072
Population : PASS
Response identity : PASS
Fold contract : PASS
Target alignment : PASS
Numeric feature contract : PASS
Feature count: 27
Feature finiteness : PASS
Target-derived feature name guard : PASS

Structured feature parquet : PASS
Path: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\structured_prior\outputs\structured_response_features.parquet
Feature audit JSON : PASS

TRACE THE RACE — CELL 6 FINAL STATUS
Structured feature rows : 35,072
Structured feature count: 27
Target used in features : NO
ModernBERT prediction used to build features : NO
Population contract : PASS
Feature numerical contract : PASS
Feature finiteness : PASS
Leakage guard : PASS

CELL 6 COMPLETE — PASS
Cell 6 memory cleanup : PASS


In [10]:
# ==============================================================================
# TRACE THE RACE — CELL 7
# STRUCTURED FEATURE INTEGRITY + LEAKAGE AUDIT
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd


print("=" * 100)
print("TRACE THE RACE — CELL 7")
print("Structured Feature Integrity + Leakage Audit")
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    globals().get(
        "STRUCTURED_FEATURES_READY",
        False,
    )
    is True
), (
    "Cell 6 dependency failed. "
    "Run Cell 6 successfully first."
)

assert (
    "STRUCTURED_FEATURE_PATH" in globals()
), (
    "STRUCTURED_FEATURE_PATH missing from Cell 6."
)

assert (
    Path(STRUCTURED_FEATURE_PATH).exists()
), (
    "Structured feature artifact does not exist:\n"
    f"{STRUCTURED_FEATURE_PATH}"
)

print(
    "\nCell 6 dependency : PASS"
)


# ==============================================================================
# 2. LOAD FEATURE TABLE
# ==============================================================================

structured_features = pd.read_parquet(
    STRUCTURED_FEATURE_PATH,
    engine="pyarrow",
)

print("\n" + "=" * 100)
print("STRUCTURED FEATURE ARTIFACT")
print("=" * 100)

print(
    "Path:",
    STRUCTURED_FEATURE_PATH,
)

print(
    "Rows:",
    f"{len(structured_features):,}",
)

print(
    "Columns:",
    len(structured_features.columns),
)


# ==============================================================================
# 3. REQUIRED IDENTITY CONTRACT
# ==============================================================================

IDENTITY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "target",
]

missing_identity = [
    c
    for c in IDENTITY_COLUMNS
    if c not in structured_features.columns
]

assert not missing_identity, (
    "Missing identity columns:\n"
    + "\n".join(missing_identity)
)

assert (
    len(structured_features) == 35_072
), (
    "Structured feature population mismatch."
)

assert (
    structured_features[
        "response_id"
    ].is_unique
), (
    "response_id is not unique."
)

assert (
    structured_features[
        "session_id"
    ].notna().all()
), (
    "session_id contains null values."
)

assert (
    structured_features[
        "objective_uid"
    ].notna().all()
), (
    "objective_uid contains null values."
)

assert (
    structured_features[
        "fold"
    ].isin([0, 1, 2, 3, 4]).all()
), (
    "Invalid fold values detected."
)

assert (
    structured_features[
        "target"
    ].isin([0, 1]).all()
), (
    "Invalid target values detected."
)

print(
    "\nIdentity contract : PASS"
)


# ==============================================================================
# 4. FEATURE COLUMN CONTRACT
# ==============================================================================

FEATURE_COLUMNS = [
    c
    for c in structured_features.columns
    if c not in IDENTITY_COLUMNS
]

assert (
    len(FEATURE_COLUMNS) == 27
), (
    "Expected exactly 27 structured features.\n"
    f"Observed: {len(FEATURE_COLUMNS)}"
)

print(
    "Feature count : 27"
)


# ==============================================================================
# 5. NUMERICAL CONTRACT
# ==============================================================================

non_numeric = [
    c
    for c in FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(
        structured_features[c]
    )
]

assert not non_numeric, (
    "Non-numeric structured features detected:\n"
    + "\n".join(non_numeric)
)

print(
    "Numeric feature contract : PASS"
)


# ==============================================================================
# 6. NULL / INFINITE AUDIT
# ==============================================================================

feature_null_counts = (
    structured_features[
        FEATURE_COLUMNS
    ].isna().sum()
)

feature_inf_counts = {}

for column in FEATURE_COLUMNS:

    values = pd.to_numeric(
        structured_features[column],
        errors="coerce",
    )

    feature_inf_counts[column] = int(
        np.isinf(
            values.to_numpy(
                dtype=np.float64
            )
        ).sum()
    )

null_feature_columns = [
    c
    for c, count in feature_null_counts.items()
    if int(count) > 0
]

inf_feature_columns = [
    c
    for c, count in feature_inf_counts.items()
    if count > 0
]

print("\n" + "=" * 100)
print("MISSING / INFINITE AUDIT")
print("=" * 100)

print(
    "Features with nulls:",
    len(null_feature_columns),
)

for column in null_feature_columns:

    print(
        f"  {column}: "
        f"{int(feature_null_counts[column]):,}"
    )

print(
    "Features with infinity:",
    len(inf_feature_columns),
)

for column in inf_feature_columns:

    print(
        f"  {column}: "
        f"{feature_inf_counts[column]:,}"
    )

# Cell 6 promised finite features.
assert not inf_feature_columns, (
    "Infinite values detected in structured features."
)

print(
    "Infinite-value contract : PASS"
)


# ==============================================================================
# 7. CONSTANT / NEAR-CONSTANT AUDIT
# ==============================================================================

constant_features = []
near_constant_features = []

near_constant_threshold = 0.995

for column in FEATURE_COLUMNS:

    series = structured_features[column]

    value_counts = (
        series
        .value_counts(
            dropna=False,
            normalize=True,
        )
    )

    if len(value_counts) <= 1:

        constant_features.append(column)

    elif (
        float(value_counts.iloc[0])
        >=
        near_constant_threshold
    ):

        near_constant_features.append(
            {
                "feature": column,
                "dominant_rate": float(
                    value_counts.iloc[0]
                ),
            }
        )

print("\n" + "=" * 100)
print("VARIANCE AUDIT")
print("=" * 100)

print(
    "Constant features:",
    len(constant_features),
)

if constant_features:

    for column in constant_features:
        print(
            "  CONSTANT:",
            column,
        )

print(
    "Near-constant features:",
    len(near_constant_features),
)

for item in near_constant_features:

    print(
        f"  {item['feature']}: "
        f"dominant_rate="
        f"{item['dominant_rate']:.6f}"
    )


# Do not automatically delete anything.
# Record the finding for the next stage.

print(
    "Variance audit : PASS"
)


# ==============================================================================
# 8. FEATURE RANGE / DISTRIBUTION AUDIT
# ==============================================================================

distribution_records = []

for column in FEATURE_COLUMNS:

    values = pd.to_numeric(
        structured_features[column],
        errors="coerce",
    )

    valid = values.dropna()

    if len(valid) == 0:

        record = {
            "feature": column,
            "count": 0,
            "null_count": int(
                values.isna().sum()
            ),
            "min": None,
            "max": None,
            "mean": None,
            "std": None,
            "p01": None,
            "p50": None,
            "p99": None,
        }

    else:

        record = {
            "feature": column,
            "count": int(len(valid)),
            "null_count": int(
                values.isna().sum()
            ),
            "min": float(
                valid.min()
            ),
            "max": float(
                valid.max()
            ),
            "mean": float(
                valid.mean()
            ),
            "std": float(
                valid.std()
            ),
            "p01": float(
                valid.quantile(0.01)
            ),
            "p50": float(
                valid.quantile(0.50)
            ),
            "p99": float(
                valid.quantile(0.99)
            ),
        }

    distribution_records.append(
        record
    )

feature_distribution = pd.DataFrame(
    distribution_records
)

print("\n" + "=" * 100)
print("FEATURE DISTRIBUTION")
print("=" * 100)

print(
    feature_distribution[
        [
            "feature",
            "count",
            "null_count",
            "min",
            "max",
            "mean",
            "std",
        ]
    ].to_string(
        index=False
    )
)


# ==============================================================================
# 9. FOLD DISTRIBUTION AUDIT
# ==============================================================================

print("\n" + "=" * 100)
print("FOLD DISTRIBUTION AUDIT")
print("=" * 100)

fold_counts = (
    structured_features[
        "fold"
    ]
    .value_counts()
    .sort_index()
)

for fold, count in fold_counts.items():

    print(
        f"Fold {int(fold)} : "
        f"{int(count):,}"
    )

assert (
    fold_counts.to_dict()
    ==
    {
        0: 6958,
        1: 7050,
        2: 7023,
        3: 7081,
        4: 6960,
    }
), (
    "Structured feature fold counts differ "
    "from frozen evidence contract."
)

print(
    "Fold distribution : PASS"
)


# ==============================================================================
# 10. SESSION LEAKAGE AUDIT
# ==============================================================================

session_fold_counts = (
    structured_features
    .groupby("session_id")[
        "fold"
    ]
    .nunique()
)

cross_fold_sessions = (
    session_fold_counts[
        session_fold_counts > 1
    ]
)

print("\n" + "=" * 100)
print("SESSION LEAKAGE AUDIT")
print("=" * 100)

print(
    "Unique sessions:",
    f"{structured_features['session_id'].nunique():,}",
)

print(
    "Cross-fold sessions:",
    len(cross_fold_sessions),
)

assert (
    len(cross_fold_sessions) == 0
), (
    "Session crosses fold boundary."
)

print(
    "Session-fold isolation : PASS"
)


# ==============================================================================
# 11. OBJECTIVE / FOLD AUDIT
# ==============================================================================

objective_fold_counts = (
    structured_features
    .groupby("objective_uid")[
        "fold"
    ]
    .nunique()
)

multi_fold_objectives = (
    objective_fold_counts[
        objective_fold_counts > 1
    ]
)

print("\n" + "=" * 100)
print("OBJECTIVE AUDIT")
print("=" * 100)

print(
    "Unique objectives:",
    f"{structured_features['objective_uid'].nunique():,}",
)

print(
    "Objectives spanning folds:",
    len(multi_fold_objectives),
)

# This is informational, not a failure.
# The frozen dataset already establishes that objectives
# can legitimately span folds.

print(
    "Objective identity audit : PASS"
)


# ==============================================================================
# 12. TARGET SEPARATION — AUDIT ONLY
#
# This does NOT modify features and is NOT used for training.
# It is only to detect suspiciously deterministic features.
# ==============================================================================

target_separation_records = []

for column in FEATURE_COLUMNS:

    series = structured_features[column]

    positive = series[
        structured_features["target"] == 1
    ]

    negative = series[
        structured_features["target"] == 0
    ]

    positive_mean = (
        float(positive.mean())
        if positive.notna().any()
        else None
    )

    negative_mean = (
        float(negative.mean())
        if negative.notna().any()
        else None
    )

    target_separation_records.append(
        {
            "feature": column,
            "positive_mean": positive_mean,
            "negative_mean": negative_mean,
            "mean_difference": (
                None
                if (
                    positive_mean is None
                    or negative_mean is None
                )
                else
                positive_mean
                -
                negative_mean
            ),
        }
    )

target_separation = pd.DataFrame(
    target_separation_records
)

print("\n" + "=" * 100)
print("TARGET SEPARATION AUDIT")
print("=" * 100)

print(
    target_separation.to_string(
        index=False
    )
)

print(
    "\nTarget separation is AUDIT ONLY."
)

print(
    "No target-derived feature construction : PASS"
)


# ==============================================================================
# 13. EXPLICIT TARGET-DEPENDENCY CHECK
#
# Detect exact equality between any feature and target.
# This is a cheap but useful leakage guard.
# ==============================================================================

exact_target_features = []

target_values = (
    structured_features[
        "target"
    ].to_numpy()
)

for column in FEATURE_COLUMNS:

    values = (
        structured_features[column]
        .to_numpy()
    )

    if len(values) != len(target_values):
        continue

    finite_mask = (
        pd.notna(values)
    )

    if not finite_mask.all():
        continue

    try:

        if np.array_equal(
            values,
            target_values,
        ):
            exact_target_features.append(
                column
            )

    except Exception:
        pass


assert not exact_target_features, (
    "Feature exactly equals target:\n"
    + "\n".join(
        exact_target_features
    )
)

print(
    "Exact target-copy leakage : PASS"
)


# ==============================================================================
# 14. RESPONSE ORDER / DUPLICATE AUDIT
# ==============================================================================

assert (
    structured_features[
        "response_id"
    ].is_unique
), (
    "Duplicate response rows detected."
)

print(
    "Response duplicate audit : PASS"
)


# ==============================================================================
# 15. FEATURE NAME AUDIT
# ==============================================================================

FORBIDDEN_FEATURE_NAME_PARTS = [
    "target",
    "label",
    "prediction",
    "oof",
    "probability",
]

suspicious_feature_names = []

for column in FEATURE_COLUMNS:

    lower = column.lower()

    if any(
        part in lower
        for part in FORBIDDEN_FEATURE_NAME_PARTS
    ):

        suspicious_feature_names.append(
            column
        )

print("\n" + "=" * 100)
print("FEATURE NAME LEAKAGE AUDIT")
print("=" * 100)

print(
    "Suspicious names:",
    suspicious_feature_names,
)

# No current feature should contain these names.
assert not suspicious_feature_names, (
    "Suspicious target/prediction-derived "
    "feature name detected:\n"
    + "\n".join(
        suspicious_feature_names
    )
)

print(
    "Feature naming guard : PASS"
)


# ==============================================================================
# 16. AUDIT JSON
# ==============================================================================

STRUCTURED_CELL7_AUDIT_PATH = (
    STRUCTURED_AUDIT_ROOT
    / "cell7_feature_integrity_audit.json"
)

cell7_audit = {
    "status": "PASS",
    "rows": int(
        len(structured_features)
    ),
    "feature_count": int(
        len(FEATURE_COLUMNS)
    ),
    "feature_columns": FEATURE_COLUMNS,
    "constant_features": constant_features,
    "near_constant_features": (
        near_constant_features
    ),
    "null_feature_columns": (
        null_feature_columns
    ),
    "infinite_feature_columns": (
        inf_feature_columns
    ),
    "cross_fold_sessions": int(
        len(cross_fold_sessions)
    ),
    "multi_fold_objectives": int(
        len(multi_fold_objectives)
    ),
    "exact_target_copy_features": (
        exact_target_features
    ),
    "suspicious_feature_names": (
        suspicious_feature_names
    ),
    "target_used_to_construct_features": False,
    "modernbert_prediction_used_to_construct_features": False,
}


with open(
    STRUCTURED_CELL7_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell7_audit,
        f,
        indent=2,
    )


assert (
    STRUCTURED_CELL7_AUDIT_PATH.exists()
)

print(
    "\nAudit JSON : PASS"
)


# ==============================================================================
# 17. FINAL STATUS
# ==============================================================================

STRUCTURED_FEATURE_AUDIT_READY = True

print("\n" + "=" * 100)
print("TRACE THE RACE — CELL 7 FINAL STATUS")
print("=" * 100)

print(
    "Structured feature population : PASS"
)

print(
    "27-feature contract            : PASS"
)

print(
    "Numeric contract               : PASS"
)

print(
    "Infinite-value contract        : PASS"
)

print(
    "Fold distribution              : PASS"
)

print(
    "Session-fold isolation         : PASS"
)

print(
    "Objective audit                : PASS"
)

print(
    "Target-copy leakage            : PASS"
)

print(
    "Feature naming guard           : PASS"
)

print(
    "Target used for construction   : NO"
)

print(
    "ModernBERT prediction used     : NO"
)

print(
    "\nCELL 7 COMPLETE — PASS"
)


# ==============================================================================
# 18. MEMORY CLEANUP
# ==============================================================================

del feature_null_counts
del feature_inf_counts
del distribution_records
del target_separation_records
del target_values
del fold_counts
del session_fold_counts
del cross_fold_sessions
del objective_fold_counts
del multi_fold_objectives

if "values" in globals():
    del values

if "series" in globals():
    del series

gc.collect()

print(
    "Cell 7 memory cleanup : PASS"
)

TRACE THE RACE — CELL 7
Structured Feature Integrity + Leakage Audit

Cell 6 dependency : PASS

STRUCTURED FEATURE ARTIFACT
Path: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\structured_prior\outputs\structured_response_features.parquet
Rows: 35,072
Columns: 32

Identity contract : PASS
Feature count : 27
Numeric feature contract : PASS

MISSING / INFINITE AUDIT
Features with nulls: 0
Features with infinity: 0
Infinite-value contract : PASS

VARIANCE AUDIT
Constant features: 3
  CONSTANT: unique_session_count
  CONSTANT: unique_objective_count
  CONSTANT: evidence_source_rate
Near-constant features: 1
  max_selection_priority: dominant_rate=0.998033
Variance audit : PASS

FEATURE DISTRIBUTION
                feature  count  null_count        min        max       mean       std
     evidence_row_count  35072           0   8.000000  41.000000  30.327241  5.843421
   unique_session_count  35072           0   1.000000   1.000000   1.000000  0.000000
 unique_objective_cou

In [13]:
# ==============================================================================
# TRACE THE RACE — CELL 8
# FOLD-SAFE OBJECTIVE PRIOR OOF — CORRECTED
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score


print("=" * 100)
print("TRACE THE RACE — CELL 8")
print("Fold-Safe Objective Prior OOF — Corrected")
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY
# ==============================================================================

assert (
    globals().get(
        "STRUCTURED_FEATURE_AUDIT_READY",
        False,
    )
    is True
), (
    "Cell 7 dependency failed. "
    "Run Cell 7 successfully first."
)

assert (
    "structured_features" in globals()
), (
    "structured_features is missing."
)

print(
    "\nCell 7 dependency : PASS"
)


# ==============================================================================
# 2. CONFIGURATION
# ==============================================================================

N_FOLDS = 5

PRIOR_ALPHA = 20.0

FOLD_VALUES = [
    0,
    1,
    2,
    3,
    4,
]

EXPECTED_FOLD_COUNTS = {
    0: 6958,
    1: 7050,
    2: 7023,
    3: 7081,
    4: 6960,
}


# ==============================================================================
# 3. OUTPUT PATHS
# ==============================================================================

PRIOR_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "structured_prior"
)

PRIOR_OUTPUT_ROOT = (
    PRIOR_ROOT
    / "outputs"
)

PRIOR_AUDIT_ROOT = (
    PRIOR_ROOT
    / "audit"
)

PRIOR_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PRIOR_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PRIOR_OOF_PATH = (
    PRIOR_OUTPUT_ROOT
    / "objective_prior_oof.parquet"
)

PRIOR_METRICS_PATH = (
    PRIOR_AUDIT_ROOT
    / "cell8_prior_oof_metrics.json"
)

PRIOR_FOLD_METRICS_PATH = (
    PRIOR_AUDIT_ROOT
    / "cell8_prior_fold_metrics.parquet"
)


# ==============================================================================
# 4. INPUT CONTRACT
# ==============================================================================

REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "target",
]

missing_columns = [
    c
    for c in REQUIRED_COLUMNS
    if c not in structured_features.columns
]

assert not missing_columns, (
    "Required columns missing:\n"
    + "\n".join(missing_columns)
)

assert (
    len(structured_features)
    ==
    35_072
), (
    "Expected 35,072 structured rows."
)

assert (
    structured_features[
        "response_id"
    ].is_unique
), (
    "response_id is not unique."
)

assert (
    structured_features[
        "fold"
    ].isin(FOLD_VALUES).all()
), (
    "Invalid fold value."
)

assert (
    structured_features[
        "target"
    ].isin([0, 1]).all()
), (
    "Invalid target value."
)

print(
    "Input contract : PASS"
)


# ==============================================================================
# 5. FOLD CONTRACT
# ==============================================================================

observed_fold_counts = (
    structured_features[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

assert (
    observed_fold_counts
    ==
    EXPECTED_FOLD_COUNTS
), (
    "Fold counts differ from frozen contract.\n"
    f"Observed: {observed_fold_counts}\n"
    f"Expected: {EXPECTED_FOLD_COUNTS}"
)

print(
    "Five-fold contract : PASS"
)


# ==============================================================================
# 6. INITIALIZE OOF TABLE
# ==============================================================================

prior_oof = (
    structured_features[
        [
            "response_id",
            "session_id",
            "objective_uid",
            "fold",
            "target",
        ]
    ]
    .copy()
)

prior_oof[
    "objective_prior"
] = np.nan

prior_oof[
    "objective_prior_logit"
] = np.nan

prior_oof[
    "objective_train_count"
] = np.nan

prior_oof[
    "objective_train_positive"
] = np.nan

prior_oof[
    "global_train_prior"
] = np.nan

prior_oof[
    "prior_seen_in_train"
] = False


# ==============================================================================
# 7. FOLD-SAFE PRIOR GENERATION
# ==============================================================================

fold_metrics = []

print("\n" + "=" * 100)
print("FOLD-SAFE OBJECTIVE PRIOR GENERATION")
print("=" * 100)


for validation_fold in FOLD_VALUES:

    print(
        f"\nProcessing validation fold "
        f"{validation_fold}..."
    )

    train_mask = (
        structured_features["fold"]
        != validation_fold
    )

    valid_mask = (
        structured_features["fold"]
        == validation_fold
    )

    train_df = structured_features.loc[
        train_mask,
        [
            "objective_uid",
            "target",
        ],
    ]

    valid_df = structured_features.loc[
        valid_mask,
        [
            "response_id",
            "objective_uid",
            "target",
        ],
    ].copy()


    # ==========================================================================
    # 7A. GLOBAL TRAINING PRIOR
    # ==========================================================================

    global_train_prior = float(
        train_df["target"].mean()
    )

    assert np.isfinite(
        global_train_prior
    )

    assert (
        0.0
        <
        global_train_prior
        <
        1.0
    )


    # ==========================================================================
    # 7B. OBJECTIVE STATISTICS — TRAINING FOLDS ONLY
    # ==========================================================================

    objective_stats = (
        train_df
        .groupby(
            "objective_uid",
            sort=False,
        )["target"]
        .agg(
            objective_train_count="size",
            objective_train_positive="sum",
        )
    )

    objective_stats[
        "objective_prior"
    ] = (
        objective_stats[
            "objective_train_positive"
        ]
        +
        PRIOR_ALPHA
        *
        global_train_prior
    ) / (
        objective_stats[
            "objective_train_count"
        ]
        +
        PRIOR_ALPHA
    )


    # ==========================================================================
    # 7C. VALIDATION OBJECTIVES
    # ==========================================================================

    valid_with_prior = (
        valid_df
        .merge(
            objective_stats.reset_index(),
            on="objective_uid",
            how="left",
            validate="many_to_one",
        )
    )


    # ==========================================================================
    # 7D. UNSEEN OBJECTIVE FALLBACK
    # ==========================================================================

    unseen_mask = (
        valid_with_prior[
            "objective_train_count"
        ].isna()
    )

    unseen_count = int(
        unseen_mask.sum()
    )

    valid_with_prior.loc[
        unseen_mask,
        "objective_train_count",
    ] = 0.0

    valid_with_prior.loc[
        unseen_mask,
        "objective_train_positive",
    ] = 0.0

    valid_with_prior.loc[
        unseen_mask,
        "objective_prior",
    ] = global_train_prior


    # ==========================================================================
    # 7E. NUMERICAL SAFETY
    # ==========================================================================

    valid_with_prior[
        "objective_prior"
    ] = (
        valid_with_prior[
            "objective_prior"
        ]
        .astype(float)
        .clip(
            1e-6,
            1.0 - 1e-6,
        )
    )

    valid_with_prior[
        "objective_prior_logit"
    ] = np.log(
        valid_with_prior[
            "objective_prior"
        ]
        /
        (
            1.0
            -
            valid_with_prior[
                "objective_prior"
            ]
        )
    )

    valid_with_prior[
        "global_train_prior"
    ] = global_train_prior

    valid_with_prior[
        "prior_seen_in_train"
    ] = ~unseen_mask


    # ==========================================================================
    # 7F. CRITICAL FIX:
    # DIRECT RESPONSE-ID INDEXED ASSIGNMENT
    #
    # No Index.map(callable) bug.
    # No positional assignment.
    # No dependency on row ordering.
    # ==========================================================================

    valid_lookup = (
        valid_with_prior
        .set_index(
            "response_id"
        )
    )

    validation_response_ids = (
        prior_oof.loc[
            valid_mask,
            "response_id"
        ]
    )

    assert (
        validation_response_ids.is_unique
    )

    assert (
        set(
            validation_response_ids
        )
        ==
        set(
            valid_lookup.index
        )
    ), (
        f"Response alignment mismatch "
        f"for fold {validation_fold}."
    )

    assignment_index = (
        prior_oof.index[
            valid_mask
        ]
    )

    aligned_prior = (
        valid_lookup.loc[
            validation_response_ids,
            [
                "objective_prior",
                "objective_prior_logit",
                "objective_train_count",
                "objective_train_positive",
                "global_train_prior",
                "prior_seen_in_train",
            ],
        ]
    )

    assert (
        len(aligned_prior)
        ==
        len(assignment_index)
    )

    # Positionally safe AFTER exact response-ID alignment.
    prior_oof.loc[
        assignment_index,
        "objective_prior"
    ] = (
        aligned_prior[
            "objective_prior"
        ].to_numpy()
    )

    prior_oof.loc[
        assignment_index,
        "objective_prior_logit"
    ] = (
        aligned_prior[
            "objective_prior_logit"
        ].to_numpy()
    )

    prior_oof.loc[
        assignment_index,
        "objective_train_count"
    ] = (
        aligned_prior[
            "objective_train_count"
        ].to_numpy()
    )

    prior_oof.loc[
        assignment_index,
        "objective_train_positive"
    ] = (
        aligned_prior[
            "objective_train_positive"
        ].to_numpy()
    )

    prior_oof.loc[
        assignment_index,
        "global_train_prior"
    ] = (
        aligned_prior[
            "global_train_prior"
        ].to_numpy()
    )

    prior_oof.loc[
        assignment_index,
        "prior_seen_in_train"
    ] = (
        aligned_prior[
            "prior_seen_in_train"
        ].to_numpy()
    )


    # ==========================================================================
    # 7G. FOLD VALIDATION
    # ==========================================================================

    fold_predictions = prior_oof.loc[
        valid_mask
    ]

    assert (
        len(fold_predictions)
        ==
        EXPECTED_FOLD_COUNTS[
            validation_fold
        ]
    )

    assert (
        fold_predictions[
            "objective_prior"
        ].notna().all()
    )

    assert (
        fold_predictions[
            "objective_prior_logit"
        ].notna().all()
    )

    assert (
        fold_predictions[
            "objective_train_count"
        ].notna().all()
    )


    # ==========================================================================
    # 7H. FOLD METRICS
    # ==========================================================================

    fold_y = (
        fold_predictions[
            "target"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    fold_p = (
        fold_predictions[
            "objective_prior"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    fold_ll = log_loss(
        fold_y,
        fold_p,
        labels=[0, 1],
    )

    fold_auc = roc_auc_score(
        fold_y,
        fold_p,
    )

    fold_metrics.append(
        {
            "fold": int(
                validation_fold
            ),
            "rows": int(
                len(fold_predictions)
            ),
            "train_rows": int(
                len(train_df)
            ),
            "global_train_prior": float(
                global_train_prior
            ),
            "unique_train_objectives": int(
                objective_stats.shape[0]
            ),
            "validation_objectives": int(
                valid_df[
                    "objective_uid"
                ].nunique()
            ),
            "unseen_objective_rows": (
                unseen_count
            ),
            "unseen_objective_rate": (
                unseen_count
                /
                len(fold_predictions)
            ),
            "log_loss": float(
                fold_ll
            ),
            "roc_auc": float(
                fold_auc
            ),
        }
    )

    print(
        f"Fold {validation_fold}: "
        f"train={len(train_df):,} | "
        f"valid={len(valid_df):,} | "
        f"global_prior={global_train_prior:.6f} | "
        f"unseen={unseen_count:,} | "
        f"LL={fold_ll:.6f} | "
        f"AUC={fold_auc:.6f}"
    )

    del train_df
    del valid_df
    del objective_stats
    del valid_with_prior
    del valid_lookup
    del aligned_prior
    del validation_response_ids
    del assignment_index
    del fold_predictions


# ==============================================================================
# 8. GLOBAL OOF CONTRACT
# ==============================================================================

assert (
    len(prior_oof)
    ==
    35_072
)

assert (
    prior_oof[
        "response_id"
    ].is_unique
)

assert (
    prior_oof[
        "objective_prior"
    ].notna().all()
)

assert (
    prior_oof[
        "objective_prior_logit"
    ].notna().all()
)

assert (
    prior_oof[
        "objective_train_count"
    ].notna().all()
)

assert (
    prior_oof[
        "objective_train_positive"
    ].notna().all()
)

assert (
    prior_oof[
        "global_train_prior"
    ].notna().all()
)

print("\n" + "=" * 100)
print("OOF COMPLETENESS")
print("=" * 100)

print(
    "OOF rows:",
    f"{len(prior_oof):,}",
)

print(
    "Unique responses:",
    f"{prior_oof['response_id'].nunique():,}",
)

print(
    "Prior nulls:",
    int(
        prior_oof[
            "objective_prior"
        ].isna().sum()
    ),
)

print(
    "Prior OOF completeness : PASS"
)


# ==============================================================================
# 9. NUMERICAL CONTRACT
# ==============================================================================

prior_values = (
    prior_oof[
        "objective_prior"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

prior_logits = (
    prior_oof[
        "objective_prior_logit"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

assert np.isfinite(
    prior_values
).all()

assert np.isfinite(
    prior_logits
).all()

assert (
    (prior_values > 0.0)
    &
    (prior_values < 1.0)
).all()

print(
    "Prior numerical contract : PASS"
)


# ==============================================================================
# 10. EXPLICIT FOLD-SAFE LEAKAGE AUDIT
# ==============================================================================

print("\n" + "=" * 100)
print("FOLD-SAFE LEAKAGE AUDIT")
print("=" * 100)


for validation_fold in FOLD_VALUES:

    validation_mask = (
        prior_oof[
            "fold"
        ]
        ==
        validation_fold
    )

    training_mask = (
        structured_features[
            "fold"
        ]
        !=
        validation_fold
    )

    expected_training_counts = (
        structured_features.loc[
            training_mask
        ]
        .groupby(
            "objective_uid"
        )
        .size()
    )

    check_table = (
        prior_oof.loc[
            validation_mask,
            [
                "objective_uid",
                "objective_train_count",
            ],
        ]
        .drop_duplicates()
    )

    for row in check_table.itertuples(
        index=False
    ):

        objective_uid = (
            row.objective_uid
        )

        expected_count = int(
            expected_training_counts.get(
                objective_uid,
                0,
            )
        )

        observed_count = int(
            row.objective_train_count
        )

        assert (
            observed_count
            ==
            expected_count
        ), (
            "Prior leakage detected: "
            f"fold={validation_fold}, "
            f"objective={objective_uid}, "
            f"observed={observed_count}, "
            f"expected={expected_count}"
        )

    print(
        f"Fold {validation_fold}: "
        "training-only counts : PASS"
    )

    del validation_mask
    del training_mask
    del expected_training_counts
    del check_table


# ==============================================================================
# 11. OOF METRICS
# ==============================================================================

oof_y = (
    prior_oof[
        "target"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

oof_p = (
    prior_oof[
        "objective_prior"
    ]
    .to_numpy(
        dtype=np.float64
    )

)

oof_log_loss = log_loss(
    oof_y,
    oof_p,
    labels=[0, 1],
)

oof_roc_auc = roc_auc_score(
    oof_y,
    oof_p,
)

positive_mask = (
    oof_y == 1
)

negative_mask = (
    oof_y == 0
)

positive_class_log_loss = float(
    -np.mean(
        np.log(
            np.clip(
                oof_p[
                    positive_mask
                ],
                1e-15,
                1.0,
            )
        )
    )
)

negative_class_log_loss = float(
    -np.mean(
        np.log(
            np.clip(
                1.0
                -
                oof_p[
                    negative_mask
                ],
                1e-15,
                1.0,
            )
        )
    )
)


print("\n" + "=" * 100)
print("OBJECTIVE PRIOR OOF METRICS")
print("=" * 100)

print(
    "OOF rows           :",
    f"{len(prior_oof):,}",
)

print(
    "OOF Log Loss       :",
    f"{oof_log_loss:.12f}",
)

print(
    "OOF ROC-AUC        :",
    f"{oof_roc_auc:.12f}",
)

print(
    "Positive-class LL  :",
    f"{positive_class_log_loss:.12f}",
)

print(
    "Negative-class LL  :",
    f"{negative_class_log_loss:.12f}",
)


# ==============================================================================
# 12. SAVE OOF ARTIFACT
# ==============================================================================

prior_oof.to_parquet(
    PRIOR_OOF_PATH,
    index=False,
)

assert (
    PRIOR_OOF_PATH.exists()
)

print(
    "\nPrior OOF parquet : PASS"
)

print(
    "Path:",
    PRIOR_OOF_PATH,
)


# ==============================================================================
# 13. SAVE FOLD METRICS
# ==============================================================================

prior_fold_metrics = pd.DataFrame(
    fold_metrics
)

prior_fold_metrics.to_parquet(
    PRIOR_FOLD_METRICS_PATH,
    index=False,
)

assert (
    PRIOR_FOLD_METRICS_PATH.exists()
)

print(
    "Prior fold metrics : PASS"
)


# ==============================================================================
# 14. SAVE METRICS JSON
# ==============================================================================

prior_metrics_payload = {
    "status": "PASS",
    "rows": int(
        len(prior_oof)
    ),
    "unique_responses": int(
        prior_oof[
            "response_id"
        ].nunique()
    ),
    "n_folds": 5,
    "prior_alpha": float(
        PRIOR_ALPHA
    ),
    "oof_log_loss": float(
        oof_log_loss
    ),
    "oof_roc_auc": float(
        oof_roc_auc
    ),
    "positive_class_log_loss": (
        positive_class_log_loss
    ),
    "negative_class_log_loss": (
        negative_class_log_loss
    ),
    "fold_metrics": fold_metrics,
    "validation_target_used_in_prior": False,
    "session_prior_used": False,
    "objective_prior_formula": (
        "(objective_train_positive + "
        "alpha * global_train_prior) / "
        "(objective_train_count + alpha)"
    ),
    "status_contract": "FOLD_SAFE_OOF",
}


with open(
    PRIOR_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        prior_metrics_payload,
        f,
        indent=2,
    )

assert (
    PRIOR_METRICS_PATH.exists()
)

print(
    "Prior metrics JSON : PASS"
)


# ==============================================================================
# 15. FINAL STATUS
# ==============================================================================

FOLD_SAFE_PRIOR_OOF_READY = True

print("\n" + "=" * 100)
print("TRACE THE RACE — CELL 8 FINAL STATUS")
print("=" * 100)

print(
    "Fold-safe objective statistics : PASS"
)

print(
    "Validation labels used in prior : NO"
)

print(
    "Session prior used : NO"
)

print(
    "Unseen-objective fallback : "
    "TRAIN-FOLD GLOBAL PRIOR"
)

print(
    "OOF population : PASS"
)

print(
    "OOF response uniqueness : PASS"
)

print(
    "Prior numerical contract : PASS"
)

print(
    "Leakage audit : PASS"
)

print(
    "Prior OOF artifact : PASS"
)

print(
    "\nCELL 8 COMPLETE — PASS"
)


# ==============================================================================
# 16. MEMORY CLEANUP
# ==============================================================================

del prior_values
del prior_logits
del oof_y
del oof_p
del positive_mask
del negative_mask

gc.collect()

print(
    "Cell 8 memory cleanup : PASS"
)

TRACE THE RACE — CELL 8
Fold-Safe Objective Prior OOF — Corrected

Cell 7 dependency : PASS
Input contract : PASS
Five-fold contract : PASS

FOLD-SAFE OBJECTIVE PRIOR GENERATION

Processing validation fold 0...
Fold 0: train=28,114 | valid=6,958 | global_prior=0.702782 | unseen=22 | LL=0.553234 | AUC=0.706316

Processing validation fold 1...
Fold 1: train=28,022 | valid=7,050 | global_prior=0.702270 | unseen=23 | LL=0.548357 | AUC=0.713376

Processing validation fold 2...
Fold 2: train=28,049 | valid=7,023 | global_prior=0.703127 | unseen=15 | LL=0.557679 | AUC=0.701138

Processing validation fold 3...
Fold 3: train=27,991 | valid=7,081 | global_prior=0.702333 | unseen=20 | LL=0.549733 | AUC=0.709830

Processing validation fold 4...
Fold 4: train=28,112 | valid=6,960 | global_prior=0.701836 | unseen=10 | LL=0.553611 | AUC=0.700041

OOF COMPLETENESS
OOF rows: 35,072
Unique responses: 35,072
Prior nulls: 0
Prior OOF completeness : PASS
Prior numerical contract : PASS

FOLD-SAFE LEAKAGE A

In [14]:
# ==============================================================================
# TRACE THE RACE — CELL 9
# STRUCTURED + FOLD-SAFE OBJECTIVE PRIOR OOF
# NESTED CROSS-FITTED META-TRAINING
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
)


print("=" * 100)
print("TRACE THE RACE — CELL 9")
print("Structured + Fold-Safe Objective Prior OOF")
print("Nested Cross-Fitted Meta-Training")
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    globals().get(
        "STRUCTURED_FEATURE_AUDIT_READY",
        False,
    )
    is True
), (
    "Cell 7 dependency failed."
)

assert (
    globals().get(
        "FOLD_SAFE_PRIOR_OOF_READY",
        False,
    )
    is True
), (
    "Cell 8 dependency failed."
)

assert (
    "structured_features" in globals()
), (
    "structured_features is missing."
)

assert (
    "prior_oof" in globals()
), (
    "prior_oof is missing."
)

print(
    "\nCell 7 dependency : PASS"
)

print(
    "Cell 8 dependency : PASS"
)


# ==============================================================================
# 2. CONFIGURATION
# ==============================================================================

FOLD_VALUES = [
    0,
    1,
    2,
    3,
    4,
]

EXPECTED_ROWS = 35_072

INNER_FOLDS = 4

RANDOM_STATE = 42

LOGISTIC_C = 1.0

MAX_ITER = 2000


# ==============================================================================
# 3. OUTPUT PATHS
# ==============================================================================

STRUCTURED_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "structured_prior"
)

STRUCTURED_OUTPUT_ROOT = (
    STRUCTURED_ROOT
    / "outputs"
)

STRUCTURED_AUDIT_ROOT = (
    STRUCTURED_ROOT
    / "audit"
)

STRUCTURED_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STRUCTURED_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STRUCTURED_PRIOR_OOF_PATH = (
    STRUCTURED_OUTPUT_ROOT
    / "structured_prior_oof.parquet"
)

STRUCTURED_PRIOR_METRICS_PATH = (
    STRUCTURED_AUDIT_ROOT
    / "cell9_structured_prior_oof_metrics.json"
)

STRUCTURED_PRIOR_FOLD_METRICS_PATH = (
    STRUCTURED_AUDIT_ROOT
    / "cell9_structured_prior_fold_metrics.parquet"
)


# ==============================================================================
# 4. BUILD MASTER INPUT TABLE
# ==============================================================================

ID_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "target",
]

STRUCTURED_FEATURE_COLUMNS = [
    c
    for c in structured_features.columns
    if c not in ID_COLUMNS
]

assert (
    len(STRUCTURED_FEATURE_COLUMNS)
    ==
    27
), (
    "Expected 27 structured features."
)

# Only use the structured features plus the prior.
META_FEATURE_COLUMNS = (
    STRUCTURED_FEATURE_COLUMNS
    +
    [
        "objective_prior",
        "objective_prior_logit",
    ]
)

meta_base = (
    structured_features[
        ID_COLUMNS
        +
        STRUCTURED_FEATURE_COLUMNS
    ]
    .merge(
        prior_oof[
            [
                "response_id",
                "objective_prior",
                "objective_prior_logit",
            ]
        ],
        on="response_id",
        how="left",
        validate="one_to_one",
    )
)

assert (
    len(meta_base)
    ==
    EXPECTED_ROWS
)

assert (
    meta_base[
        "response_id"
    ].is_unique
)

assert (
    meta_base[
        META_FEATURE_COLUMNS
    ]
    .notna()
    .all()
    .all()
)

print(
    "\nMeta input table : PASS"
)

print(
    "Rows:",
    f"{len(meta_base):,}",
)

print(
    "Structured features:",
    len(STRUCTURED_FEATURE_COLUMNS),
)

print(
    "Prior features: 2"
)

print(
    "Total meta features:",
    len(META_FEATURE_COLUMNS),
)


# ==============================================================================
# 5. OUTER OOF CONTAINER
# ==============================================================================

structured_prior_oof = (
    meta_base[
        ID_COLUMNS
    ]
    .copy()
)

structured_prior_oof[
    "prediction"
] = np.nan

structured_prior_oof[
    "outer_train_rows"
] = np.nan

structured_prior_oof[
    "inner_prior_rows"
] = np.nan


# ==============================================================================
# 6. HELPER:
#    BUILD OUTER-TRAIN META PRIOR VIA INNER CROSS-FITTING
# ==============================================================================

def build_inner_cross_fitted_prior(
    outer_train_df,
    inner_fold_count=4,
    alpha=20.0,
    seed=42,
):
    """
    For rows belonging to outer training data, construct objective priors
    without using each row's own inner-fold labels.

    The resulting prior is safe to use as a training feature for the
    outer-fold fusion model.
    """

    working = outer_train_df[
        [
            "response_id",
            "objective_uid",
            "target",
        ]
    ].copy()

    # Deterministic inner folds by response order.
    #
    # IMPORTANT:
    # We do NOT use target to create these fold assignments.
    ordered_ids = (
        working[
            "response_id"
        ]
        .astype(str)
        .sort_values(
            kind="stable"
        )
        .to_numpy()
    )

    inner_assignment = {
        response_id: (
            i % inner_fold_count
        )
        for i, response_id
        in enumerate(ordered_ids)
    }

    working[
        "_inner_fold"
    ] = (
        working[
            "response_id"
        ]
        .astype(str)
        .map(
            inner_assignment
        )
    )

    working[
        "inner_objective_prior"
    ] = np.nan

    for inner_valid_fold in range(
        inner_fold_count
    ):

        inner_train = working.loc[
            working[
                "_inner_fold"
            ]
            !=
            inner_valid_fold
        ]

        inner_valid_mask = (
            working[
                "_inner_fold"
            ]
            ==
            inner_valid_fold
        )

        global_prior = float(
            inner_train[
                "target"
            ].mean()
        )

        objective_stats = (
            inner_train
            .groupby(
                "objective_uid",
                sort=False,
            )[
                "target"
            ]
            .agg(
                count="size",
                positive="sum",
            )
        )

        objective_stats[
            "prior"
        ] = (
            objective_stats[
                "positive"
            ]
            +
            alpha
            *
            global_prior
        ) / (
            objective_stats[
                "count"
            ]
            +
            alpha
        )

        valid_objectives = (
            working.loc[
                inner_valid_mask,
                "objective_uid",
            ]
        )

        valid_priors = (
            valid_objectives
            .map(
                objective_stats[
                    "prior"
                ]
            )
        )

        valid_priors = (
            valid_priors
            .fillna(
                global_prior
            )
            .clip(
                1e-6,
                1.0 - 1e-6,
            )
        )

        working.loc[
            inner_valid_mask,
            "inner_objective_prior",
        ] = (
            valid_priors
            .to_numpy()
        )

        del inner_train
        del objective_stats
        del valid_objectives
        del valid_priors

    assert (
        working[
            "inner_objective_prior"
        ]
        .notna()
        .all()
    )

    assert np.isfinite(
        working[
            "inner_objective_prior"
        ]
        .to_numpy(
            dtype=np.float64
        )
    ).all()

    return working[
        [
            "response_id",
            "inner_objective_prior",
        ]
    ]


# ==============================================================================
# 7. OUTER FIVE-FOLD OOF
# ==============================================================================

fold_metrics = []

print("\n" + "=" * 100)
print("OUTER 5-FOLD STRUCTURED + PRIOR OOF")
print("=" * 100)


for validation_fold in FOLD_VALUES:

    print(
        f"\nProcessing outer fold "
        f"{validation_fold}..."
    )

    outer_train_mask = (
        meta_base[
            "fold"
        ]
        !=
        validation_fold
    )

    outer_valid_mask = (
        meta_base[
            "fold"
        ]
        ==
        validation_fold
    )

    outer_train = meta_base.loc[
        outer_train_mask
    ].copy()

    outer_valid = meta_base.loc[
        outer_valid_mask
    ].copy()


    # ==========================================================================
    # 7A. OUTER TRAINING PRIOR — INNER CROSS-FITTED
    # ==========================================================================

    inner_prior = (
        build_inner_cross_fitted_prior(
            outer_train[
                [
                    "response_id",
                    "objective_uid",
                    "target",
                ]
            ],
            inner_fold_count=INNER_FOLDS,
            alpha=20.0,
            seed=RANDOM_STATE,
        )
    )

    outer_train = (
        outer_train
        .merge(
            inner_prior,
            on="response_id",
            how="left",
            validate="one_to_one",
        )
    )

    assert (
        outer_train[
            "inner_objective_prior"
        ]
        .notna()
        .all()
    )


    # ==========================================================================
    # 7B. VALIDATION PRIOR
    #
    # Cell 8 already generated this using OUTER TRAIN ONLY.
    # ==========================================================================

    validation_prior = (
        prior_oof.loc[
            prior_oof[
                "fold"
            ]
            ==
            validation_fold,
            [
                "response_id",
                "objective_prior",
            ],
        ]
    )

    assert (
        len(validation_prior)
        ==
        len(outer_valid)
    )

    outer_valid = (
        outer_valid
        .drop(
            columns=[
                "objective_prior"
            ]
        )
        .merge(
            validation_prior,
            on="response_id",
            how="left",
            validate="one_to_one",
        )
    )

    assert (
        outer_valid[
            "objective_prior"
        ]
        .notna()
        .all()
    )


    # ==========================================================================
    # 7C. BUILD TRAINING MATRIX
    #
    # IMPORTANT:
    # For outer-train rows, use INNER-CROSSFITTED prior.
    # Do not use Cell 8's outer OOF prior here.
    # ==========================================================================

    train_feature_columns = (
        STRUCTURED_FEATURE_COLUMNS
        +
        [
            "inner_objective_prior",
        ]
    )

    valid_feature_columns = (
        STRUCTURED_FEATURE_COLUMNS
        +
        [
            "objective_prior",
        ]
    )

    X_train = (
        outer_train[
            train_feature_columns
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    y_train = (
        outer_train[
            "target"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    X_valid = (
        outer_valid[
            valid_feature_columns
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    y_valid = (
        outer_valid[
            "target"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    # ==========================================================================
    # 7D. NUMERICAL CONTRACT
    # ==========================================================================

    assert np.isfinite(
        X_train
    ).all()

    assert np.isfinite(
        X_valid
    ).all()

    assert np.isfinite(
        y_train
    ).all()

    assert np.isfinite(
        y_valid
    ).all()


    # ==========================================================================
    # 7E. MODEL
    #
    # Standardization is fit ONLY on outer training data.
    # Logistic regression is fit ONLY on outer training data.
    # ==========================================================================

    fusion_model = Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(
                    with_mean=True,
                    with_std=True,
                ),
            ),
            (
                "logistic",
                LogisticRegression(
                    C=LOGISTIC_C,
                    max_iter=MAX_ITER,
                    solver="lbfgs",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    fusion_model.fit(
        X_train,
        y_train,
    )


    # ==========================================================================
    # 7F. VALIDATION PREDICTION
    # ==========================================================================

    valid_prediction = (
        fusion_model
        .predict_proba(
            X_valid
        )[:, 1]
    )

    assert np.isfinite(
        valid_prediction
    ).all()

    assert (
        (
            valid_prediction
            >
            0.0
        )
        &
        (
            valid_prediction
            <
            1.0
        )
    ).all()


    # ==========================================================================
    # 7G. EXACT RESPONSE ALIGNMENT
    # ==========================================================================

    validation_response_ids = (
        outer_valid[
            "response_id"
        ]
        .to_numpy()
    )

    assert (
        len(
            validation_response_ids
        )
        ==
        len(
            valid_prediction
        )
    )

    prediction_lookup = pd.Series(
        valid_prediction,
        index=validation_response_ids,
    )

    output_response_ids = (
        structured_prior_oof.loc[
            structured_prior_oof[
                "fold"
            ]
            ==
            validation_fold,
            "response_id",
        ]
    )

    assert (
        set(output_response_ids)
        ==
        set(validation_response_ids)
    )

    structured_prior_oof.loc[
        structured_prior_oof[
            "fold"
        ]
        ==
        validation_fold,
        "prediction",
    ] = (
        output_response_ids
        .map(
            prediction_lookup
        )
        .to_numpy()
    )

    structured_prior_oof.loc[
        structured_prior_oof[
            "fold"
        ]
        ==
        validation_fold,
        "outer_train_rows",
    ] = len(
        outer_train
    )

    structured_prior_oof.loc[
        structured_prior_oof[
            "fold"
        ]
        ==
        validation_fold,
        "inner_prior_rows",
    ] = len(
        inner_prior
    )


    # ==========================================================================
    # 7H. FOLD METRICS
    # ==========================================================================

    fold_ll = log_loss(
        y_valid,
        valid_prediction,
        labels=[
            0,
            1,
        ],
    )

    fold_auc = roc_auc_score(
        y_valid,
        valid_prediction,
    )

    fold_metrics.append(
        {
            "fold": int(
                validation_fold
            ),
            "train_rows": int(
                len(outer_train)
            ),
            "valid_rows": int(
                len(outer_valid)
            ),
            "log_loss": float(
                fold_ll
            ),
            "roc_auc": float(
                fold_auc
            ),
            "feature_count": int(
                X_train.shape[1]
            ),
            "inner_prior_cross_fitted": True,
        }
    )

    print(
        f"Fold {validation_fold}: "
        f"train={len(outer_train):,} | "
        f"valid={len(outer_valid):,} | "
        f"LL={fold_ll:.6f} | "
        f"AUC={fold_auc:.6f}"
    )

    del outer_train
    del outer_valid
    del inner_prior
    del validation_prior
    del X_train
    del X_valid
    del y_train
    del y_valid
    del valid_prediction
    del validation_response_ids
    del output_response_ids
    del prediction_lookup
    del fusion_model


# ==============================================================================
# 8. GLOBAL OOF CONTRACT
# ==============================================================================

assert (
    structured_prior_oof[
        "prediction"
    ]
    .notna()
    .all()
)

assert (
    structured_prior_oof[
        "response_id"
    ]
    .is_unique
)

assert (
    len(structured_prior_oof)
    ==
    EXPECTED_ROWS
)

prediction_values = (
    structured_prior_oof[
        "prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

assert np.isfinite(
    prediction_values
).all()

assert (
    (
        prediction_values
        >
        0.0
    )
    &
    (
        prediction_values
        <
        1.0
    )
).all()

print("\n" + "=" * 100)
print("STRUCTURED + PRIOR OOF CONTRACT")
print("=" * 100)

print(
    "OOF rows:",
    f"{len(structured_prior_oof):,}",
)

print(
    "Unique responses:",
    f"{structured_prior_oof['response_id'].nunique():,}",
)

print(
    "Prediction nulls:",
    int(
        structured_prior_oof[
            "prediction"
        ].isna().sum()
    ),
)

print(
    "Prediction range : PASS"
)

print(
    "OOF population : PASS"
)


# ==============================================================================
# 9. GLOBAL OOF METRICS
# ==============================================================================

oof_y = (
    structured_prior_oof[
        "target"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

oof_prediction = (
    structured_prior_oof[
        "prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

structured_prior_log_loss = log_loss(
    oof_y,
    oof_prediction,
    labels=[
        0,
        1,
    ],
)

structured_prior_auc = roc_auc_score(
    oof_y,
    oof_prediction,
)

positive_mask = (
    oof_y == 1
)

negative_mask = (
    oof_y == 0
)

positive_class_log_loss = float(
    -np.mean(
        np.log(
            np.clip(
                oof_prediction[
                    positive_mask
                ],
                1e-15,
                1.0,
            )
        )
    )
)

negative_class_log_loss = float(
    -np.mean(
        np.log(
            np.clip(
                1.0
                -
                oof_prediction[
                    negative_mask
                ],
                1e-15,
                1.0,
            )
        )
    )
)


# ==============================================================================
# 10. BASELINE COMPARISON
# ==============================================================================

MODERNBERT_OOF_LOG_LOSS = 0.5475514519008905

prior_only_log_loss = log_loss(
    oof_y,
    prior_oof[
        "objective_prior"
    ].to_numpy(
        dtype=np.float64
    ),
    labels=[
        0,
        1,
    ],
)

delta_vs_modernbert = (
    structured_prior_log_loss
    -
    MODERNBERT_OOF_LOG_LOSS
)

delta_vs_prior = (
    structured_prior_log_loss
    -
    prior_only_log_loss
)


print("\n" + "=" * 100)
print("STRUCTURED + PRIOR OOF METRICS")
print("=" * 100)

print(
    "OOF rows           :",
    f"{len(structured_prior_oof):,}",
)

print(
    "OOF Log Loss       :",
    f"{structured_prior_log_loss:.12f}",
)

print(
    "OOF ROC-AUC        :",
    f"{structured_prior_auc:.12f}",
)

print(
    "Positive-class LL  :",
    f"{positive_class_log_loss:.12f}",
)

print(
    "Negative-class LL  :",
    f"{negative_class_log_loss:.12f}",
)

print(
    "\nModernBERT OOF LL  :",
    f"{MODERNBERT_OOF_LOG_LOSS:.12f}",
)

print(
    "Delta vs ModernBERT:",
    f"{delta_vs_modernbert:+.12f}",
)

print(
    "Prior-only OOF LL  :",
    f"{prior_only_log_loss:.12f}",
)

print(
    "Delta vs Prior     :",
    f"{delta_vs_prior:+.12f}",
)


# ==============================================================================
# 11. FOLD METRICS
# ==============================================================================

structured_prior_fold_metrics = (
    pd.DataFrame(
        fold_metrics
    )
)

structured_prior_fold_metrics.to_parquet(
    STRUCTURED_PRIOR_FOLD_METRICS_PATH,
    index=False,
)

assert (
    STRUCTURED_PRIOR_FOLD_METRICS_PATH.exists()
)

print(
    "\nFold metrics parquet : PASS"
)


# ==============================================================================
# 12. SAVE OOF
# ==============================================================================

structured_prior_oof.to_parquet(
    STRUCTURED_PRIOR_OOF_PATH,
    index=False,
)

assert (
    STRUCTURED_PRIOR_OOF_PATH.exists()
)

print(
    "Structured + prior OOF : PASS"
)

print(
    "Path:",
    STRUCTURED_PRIOR_OOF_PATH,
)


# ==============================================================================
# 13. METRICS JSON
# ==============================================================================

cell9_metrics = {
    "status": "PASS",
    "rows": int(
        len(structured_prior_oof)
    ),
    "unique_responses": int(
        structured_prior_oof[
            "response_id"
        ].nunique()
    ),
    "feature_count": int(
        len(META_FEATURE_COLUMNS)
    ),
    "structured_feature_count": int(
        len(STRUCTURED_FEATURE_COLUMNS)
    ),
    "prior_feature_count": 2,
    "oof_log_loss": float(
        structured_prior_log_loss
    ),
    "oof_roc_auc": float(
        structured_prior_auc
    ),
    "positive_class_log_loss": (
        positive_class_log_loss
    ),
    "negative_class_log_loss": (
        negative_class_log_loss
    ),
    "modernbert_oof_log_loss": float(
        MODERNBERT_OOF_LOG_LOSS
    ),
    "delta_vs_modernbert": float(
        delta_vs_modernbert
    ),
    "prior_only_log_loss": float(
        prior_only_log_loss
    ),
    "delta_vs_prior": float(
        delta_vs_prior
    ),
    "inner_folds": INNER_FOLDS,
    "logistic_C": LOGISTIC_C,
    "random_state": RANDOM_STATE,
    "outer_five_fold_oof": True,
    "training_prior_inner_cross_fitted": True,
    "validation_prior_outer_train_only": True,
    "validation_targets_used_in_features": False,
}


with open(
    STRUCTURED_PRIOR_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell9_metrics,
        f,
        indent=2,
    )

assert (
    STRUCTURED_PRIOR_METRICS_PATH.exists()
)

print(
    "Metrics JSON : PASS"
)


# ==============================================================================
# 14. FINAL STATUS
# ==============================================================================

STRUCTURED_PRIOR_OOF_READY = True

print("\n" + "=" * 100)
print("TRACE THE RACE — CELL 9 FINAL STATUS")
print("=" * 100)

print(
    "Outer 5-fold OOF              : PASS"
)

print(
    "Inner cross-fitted train prior: PASS"
)

print(
    "Outer-train-only valid prior  : PASS"
)

print(
    "Structured feature contract   : PASS"
)

print(
    "Prediction population         : PASS"
)

print(
    "Prediction numerical range   : PASS"
)

print(
    "Session-grouped folds         : PASS"
)

print(
    "Target leakage guard          : PASS"
)

print(
    "Structured + prior OOF        : PASS"
)

print(
    "\nCELL 9 COMPLETE — PASS"
)


# ==============================================================================
# 15. MEMORY CLEANUP
# ==============================================================================

del prediction_values
del oof_y
del oof_prediction
del positive_mask
del negative_mask
del fold_metrics

gc.collect()

print(
    "Cell 9 memory cleanup : PASS"
)

TRACE THE RACE — CELL 9
Structured + Fold-Safe Objective Prior OOF
Nested Cross-Fitted Meta-Training

Cell 7 dependency : PASS
Cell 8 dependency : PASS

Meta input table : PASS
Rows: 35,072
Structured features: 27
Prior features: 2
Total meta features: 29

OUTER 5-FOLD STRUCTURED + PRIOR OOF

Processing outer fold 0...
Fold 0: train=28,114 | valid=6,958 | LL=0.550276 | AUC=0.714214

Processing outer fold 1...
Fold 1: train=28,022 | valid=7,050 | LL=0.546517 | AUC=0.717142

Processing outer fold 2...
Fold 2: train=28,049 | valid=7,023 | LL=0.554667 | AUC=0.706089

Processing outer fold 3...
Fold 3: train=27,991 | valid=7,081 | LL=0.545632 | AUC=0.718442

Processing outer fold 4...
Fold 4: train=28,112 | valid=6,960 | LL=0.549278 | AUC=0.709681

STRUCTURED + PRIOR OOF CONTRACT
OOF rows: 35,072
Unique responses: 35,072
Prediction nulls: 0
Prediction range : PASS
OOF population : PASS

STRUCTURED + PRIOR OOF METRICS
OOF rows           : 35,072
OOF Log Loss       : 0.549264028594
OOF ROC-AU

In [16]:
# ==============================================================================
# TRACE THE RACE — CELL 10
# TF-IDF + LOGISTIC REGRESSION — STRICT 5-FOLD OOF
#
# INPUT:
#   Frozen response-level evidence_packs.parquet
#
# IMPORTANT:
#   We DO NOT use responses.parquet for evidence_text.
#   responses.parquet does not contain evidence_text.
#
#   TF-IDF is fit independently inside each outer fold.
#   Validation-fold text is NEVER used to fit the vectorizer.
# ==============================================================================

from pathlib import Path
import gc
import json
import hashlib

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
)


print("=" * 100)
print(
    "TRACE THE RACE — CELL 10"
)
print(
    "TF-IDF + Logistic Regression — Strict 5-Fold OOF"
)
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATES
# ==============================================================================

assert (
    globals().get(
        "STRUCTURED_PRIOR_OOF_READY",
        False,
    )
    is True
), (
    "Cell 9 dependency failed. "
    "Run Cell 9 successfully first."
)

assert (
    "structured_features" in globals()
), (
    "structured_features is missing."
)

print(
    "\nCell 9 dependency : PASS"
)


# ==============================================================================
# 2. CONFIGURATION
# ==============================================================================

FOLD_VALUES = [
    0,
    1,
    2,
    3,
    4,
]

EXPECTED_ROWS = 35_072

RANDOM_STATE = 42

TFIDF_MAX_FEATURES = 150_000

TFIDF_NGRAM_RANGE = (
    1,
    2,
)

TFIDF_MIN_DF = 2

TFIDF_MAX_DF = 0.995

TFIDF_SUBLINEAR_TF = True

TFIDF_NORM = "l2"

LOGISTIC_C = 2.0

MAX_ITER = 1000


# ==============================================================================
# 3. FROZEN EVIDENCE PATH
# ==============================================================================

if (
    "FROZEN_EVIDENCE_PATH"
    in globals()
):

    FROZEN_EVIDENCE_LOCAL = (
        Path(
            FROZEN_EVIDENCE_PATH
        )
    )

else:

    FROZEN_EVIDENCE_LOCAL = (
        PROJECT_ROOT
        / "scratch_mastery_outputs"
        / "03_evidence_pack"
        / "frozen"
        / "evidence_packs.parquet"
    )


assert (
    FROZEN_EVIDENCE_LOCAL.exists()
), (
    "Frozen evidence parquet not found:\n"
    f"{FROZEN_EVIDENCE_LOCAL}"
)

print(
    "\nFrozen evidence : PASS"
)

print(
    "Path:",
    FROZEN_EVIDENCE_LOCAL,
)


# ==============================================================================
# 4. LOAD ONLY REQUIRED COLUMNS
# ==============================================================================

EVIDENCE_REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "objective_text",
    "evidence_text",
    "target",
]

evidence_df = pd.read_parquet(
    FROZEN_EVIDENCE_LOCAL,
    columns=EVIDENCE_REQUIRED_COLUMNS,
)

assert (
    len(evidence_df)
    ==
    EXPECTED_ROWS
), (
    "Frozen evidence population mismatch."
)

assert (
    evidence_df[
        "response_id"
    ].is_unique
), (
    "response_id is not unique."
)

print(
    "\nFrozen evidence rows:",
    f"{len(evidence_df):,}",
)

print(
    "Evidence schema : PASS"
)


# ==============================================================================
# 5. EXACT FOLD / TARGET CONTRACT
# ==============================================================================

assert (
    evidence_df[
        "fold"
    ]
    .isin(FOLD_VALUES)
    .all()
)

assert (
    evidence_df[
        "target"
    ]
    .isin([0, 1])
    .all()
)

expected_fold_counts = {
    0: 6958,
    1: 7050,
    2: 7023,
    3: 7081,
    4: 6960,
}

observed_fold_counts = (
    evidence_df[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

assert (
    observed_fold_counts
    ==
    expected_fold_counts
), (
    "Frozen evidence fold counts mismatch.\n"
    f"Observed: {observed_fold_counts}\n"
    f"Expected: {expected_fold_counts}"
)

print(
    "Five-fold contract : PASS"
)


# ==============================================================================
# 6. TARGET ALIGNMENT AGAINST STRUCTURED MASTER
# ==============================================================================

target_alignment = (
    evidence_df[
        [
            "response_id",
            "target",
            "fold",
        ]
    ]
    .merge(
        structured_features[
            [
                "response_id",
                "target",
                "fold",
            ]
        ],
        on="response_id",
        how="inner",
        suffixes=(
            "_evidence",
            "_structured",
        ),
        validate="one_to_one",
    )
)

assert (
    len(target_alignment)
    ==
    EXPECTED_ROWS
)

assert (
    (
        target_alignment[
            "target_evidence"
        ]
        ==
        target_alignment[
            "target_structured"
        ]
    )
    .all()
)

assert (
    (
        target_alignment[
            "fold_evidence"
        ]
        ==
        target_alignment[
            "fold_structured"
        ]
    )
    .all()
)

print(
    "Target alignment : PASS"
)

del target_alignment


# ==============================================================================
# 7. TEXT CONTRACT
#
# TF-IDF input:
#
#     objective_text
#           +
#     evidence_text
#
# This is response-level objective-specific evidence,
# not the entire global transcript.
# ==============================================================================

assert (
    evidence_df[
        "objective_text"
    ].notna()
    .all()
)

assert (
    evidence_df[
        "evidence_text"
    ].notna()
    .all()
)

evidence_df[
    "objective_text"
] = (
    evidence_df[
        "objective_text"
    ]
    .astype(str)
    .str.strip()
)

evidence_df[
    "evidence_text"
] = (
    evidence_df[
        "evidence_text"
    ]
    .astype(str)
    .str.strip()
)

assert (
    (
        evidence_df[
            "objective_text"
        ].str.len()
        >
        0
    )
    .all()
)

assert (
    (
        evidence_df[
            "evidence_text"
        ].str.len()
        >
        0
    )
    .all()
)


# Objective + evidence lexical representation.
#
# Delimiter makes the two sources separable while still producing
# one document for the TF-IDF model.

evidence_df[
    "_tfidf_text"
] = (
    evidence_df[
        "objective_text"
    ]
    +
    "\nOBJECTIVE_EVIDENCE\n"
    +
    evidence_df[
        "evidence_text"
    ]
)

assert (
    evidence_df[
        "_tfidf_text"
    ].notna()
    .all()
)

assert (
    (
        evidence_df[
            "_tfidf_text"
        ].str.len()
        >
        0
    )
    .all()
)

print(
    "TF-IDF text construction : PASS"
)

print(
    "Text source : objective_text + evidence_text"
)


# ==============================================================================
# 8. OOF CONTAINER
# ==============================================================================

tfidf_oof = (
    evidence_df[
        [
            "response_id",
            "session_id",
            "fold",
            "target",
        ]
    ]
    .copy()
)

tfidf_oof[
    "prediction"
] = np.nan


# ==============================================================================
# 9. OUTER 5-FOLD TF-IDF OOF
# ==============================================================================

fold_metrics = []

print("\n" + "=" * 100)
print(
    "STRICT OUTER 5-FOLD TF-IDF OOF"
)
print("=" * 100)


for validation_fold in FOLD_VALUES:

    print(
        f"\nProcessing validation fold "
        f"{validation_fold}..."
    )


    # --------------------------------------------------------------------------
    # OUTER SPLIT
    # --------------------------------------------------------------------------

    train_mask = (
        evidence_df[
            "fold"
        ]
        !=
        validation_fold
    )

    valid_mask = (
        evidence_df[
            "fold"
        ]
        ==
        validation_fold
    )

    train_df = evidence_df.loc[
        train_mask
    ]

    valid_df = evidence_df.loc[
        valid_mask
    ]

    assert (
        len(train_df)
        +
        len(valid_df)
        ==
        EXPECTED_ROWS
    )

    assert (
        len(valid_df)
        ==
        expected_fold_counts[
            validation_fold
        ]
    )


    # --------------------------------------------------------------------------
    # TF-IDF VECTORISER
    #
    # CRITICAL:
    # fit() sees TRAINING TEXT ONLY.
    # Validation text is transform() only.
    # --------------------------------------------------------------------------

    vectorizer = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF,
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        norm=TFIDF_NORM,
        dtype=np.float32,
    )

    X_train = vectorizer.fit_transform(
        train_df[
            "_tfidf_text"
        ]
    )

    X_valid = vectorizer.transform(
        valid_df[
            "_tfidf_text"
        ]
    )

    vocabulary_size = int(
        len(
            vectorizer.vocabulary_
        )
    )

    assert (
        vocabulary_size
        >
        0
    )

    assert (
        X_train.shape[1]
        ==
        X_valid.shape[1]
    )

    print(
        "Vocabulary:",
        f"{vocabulary_size:,}",
    )

    print(
        "Train matrix:",
        X_train.shape,
    )

    print(
        "Valid matrix:",
        X_valid.shape,
    )


    # --------------------------------------------------------------------------
    # TARGET
    # --------------------------------------------------------------------------

    y_train = (
        train_df[
            "target"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    y_valid = (
        valid_df[
            "target"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    assert (
        np.unique(
            y_train
        ).tolist()
        ==
        [0, 1]
    )


    # --------------------------------------------------------------------------
    # LOGISTIC REGRESSION
    # --------------------------------------------------------------------------

    classifier = LogisticRegression(
        C=LOGISTIC_C,
        max_iter=MAX_ITER,
        solver="liblinear",
        random_state=RANDOM_STATE,
    )

    classifier.fit(
        X_train,
        y_train,
    )


    # --------------------------------------------------------------------------
    # VALIDATION PREDICTION
    # --------------------------------------------------------------------------

    valid_prediction = (
        classifier
        .predict_proba(
            X_valid
        )[:, 1]
    )

    assert np.isfinite(
        valid_prediction
    ).all()

    assert (
        (
            valid_prediction
            >
            0.0
        )
        &
        (
            valid_prediction
            <
            1.0
        )
    ).all()


    # --------------------------------------------------------------------------
    # EXACT RESPONSE ALIGNMENT
    # --------------------------------------------------------------------------

    valid_response_ids = (
        valid_df[
            "response_id"
        ]
        .to_numpy()
    )

    prediction_lookup = pd.Series(
        valid_prediction,
        index=valid_response_ids,
    )

    output_mask = (
        tfidf_oof[
            "fold"
        ]
        ==
        validation_fold
    )

    output_response_ids = (
        tfidf_oof.loc[
            output_mask,
            "response_id",
        ]
    )

    assert (
        set(
            output_response_ids
        )
        ==
        set(
            valid_response_ids
        )
    )

    aligned_predictions = (
        output_response_ids
        .map(
            prediction_lookup
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    assert (
        np.isfinite(
            aligned_predictions
        ).all()
    )

    tfidf_oof.loc[
        output_mask,
        "prediction",
    ] = aligned_predictions


    # --------------------------------------------------------------------------
    # FOLD METRICS
    # --------------------------------------------------------------------------

    fold_ll = log_loss(
        y_valid,
        valid_prediction,
        labels=[
            0,
            1,
        ],
    )

    fold_auc = roc_auc_score(
        y_valid,
        valid_prediction,
    )

    fold_metrics.append(
        {
            "fold": int(
                validation_fold
            ),
            "train_rows": int(
                len(train_df)
            ),
            "valid_rows": int(
                len(valid_df)
            ),
            "vocabulary_size": vocabulary_size,
            "log_loss": float(
                fold_ll
            ),
            "roc_auc": float(
                fold_auc
            ),
        }
    )

    print(
        f"Fold {validation_fold}: "
        f"train={len(train_df):,} | "
        f"valid={len(valid_df):,} | "
        f"LL={fold_ll:.6f} | "
        f"AUC={fold_auc:.6f}"
    )


    # --------------------------------------------------------------------------
    # MEMORY
    # --------------------------------------------------------------------------

    del train_df
    del valid_df
    del vectorizer
    del classifier
    del X_train
    del X_valid
    del y_train
    del y_valid
    del valid_prediction
    del valid_response_ids
    del prediction_lookup
    del output_response_ids
    del aligned_predictions

    gc.collect()


# ==============================================================================
# 10. GLOBAL OOF CONTRACT
# ==============================================================================

assert (
    tfidf_oof[
        "prediction"
    ]
    .notna()
    .all()
)

assert (
    tfidf_oof[
        "response_id"
    ].is_unique
)

assert (
    len(tfidf_oof)
    ==
    EXPECTED_ROWS
)

tfidf_prediction = (
    tfidf_oof[
        "prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

assert np.isfinite(
    tfidf_prediction
).all()

assert (
    (
        tfidf_prediction
        >
        0.0
    )
    &
    (
        tfidf_prediction
        <
        1.0
    )
).all()


print("\n" + "=" * 100)
print(
    "TF-IDF OOF CONTRACT"
)
print("=" * 100)

print(
    "OOF rows:",
    f"{len(tfidf_oof):,}",
)

print(
    "Unique responses:",
    f"{tfidf_oof['response_id'].nunique():,}",
)

print(
    "Prediction nulls:",
    int(
        tfidf_oof[
            "prediction"
        ].isna().sum()
    ),
)

print(
    "Prediction range : PASS"
)

print(
    "OOF population : PASS"
)


# ==============================================================================
# 11. GLOBAL OOF METRICS
# ==============================================================================

tfidf_y = (
    tfidf_oof[
        "target"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

tfidf_prediction = (
    tfidf_oof[
        "prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

tfidf_log_loss = log_loss(
    tfidf_y,
    tfidf_prediction,
    labels=[
        0,
        1,
    ],
)

tfidf_auc = roc_auc_score(
    tfidf_y,
    tfidf_prediction,
)


# ==============================================================================
# 12. CLASS-WISE LOG LOSS
# ==============================================================================

tfidf_positive_mask = (
    tfidf_y == 1
)

tfidf_negative_mask = (
    tfidf_y == 0
)

tfidf_positive_ll = float(
    -np.mean(
        np.log(
            np.clip(
                tfidf_prediction[
                    tfidf_positive_mask
                ],
                1e-15,
                1.0,
            )
        )
    )
)

tfidf_negative_ll = float(
    -np.mean(
        np.log(
            np.clip(
                1.0
                -
                tfidf_prediction[
                    tfidf_negative_mask
                ],
                1e-15,
                1.0,
            )
        )
    )
)


# ==============================================================================
# 13. BASELINE COMPARISON
# ==============================================================================

MODERNBERT_OOF_LOG_LOSS = (
    0.5475514519008905
)

STRUCTURED_PRIOR_OOF_LOG_LOSS = (
    0.549264028594
)

tfidf_delta_vs_modernbert = (
    tfidf_log_loss
    -
    MODERNBERT_OOF_LOG_LOSS
)

tfidf_delta_vs_structured = (
    tfidf_log_loss
    -
    STRUCTURED_PRIOR_OOF_LOG_LOSS
)


print("\n" + "=" * 100)
print(
    "TF-IDF OOF METRICS"
)
print("=" * 100)

print(
    "OOF rows           :",
    f"{len(tfidf_oof):,}",
)

print(
    "OOF Log Loss       :",
    f"{tfidf_log_loss:.12f}",
)

print(
    "OOF ROC-AUC        :",
    f"{tfidf_auc:.12f}",
)

print(
    "Positive-class LL  :",
    f"{tfidf_positive_ll:.12f}",
)

print(
    "Negative-class LL  :",
    f"{tfidf_negative_ll:.12f}",
)

print(
    "\nModernBERT OOF LL  :",
    f"{MODERNBERT_OOF_LOG_LOSS:.12f}",
)

print(
    "Delta vs ModernBERT:",
    f"{tfidf_delta_vs_modernbert:+.12f}",
)

print(
    "\nStructured+Prior LL:",
    f"{STRUCTURED_PRIOR_OOF_LOG_LOSS:.12f}",
)

print(
    "Delta vs Structured:",
    f"{tfidf_delta_vs_structured:+.12f}",
)


# ==============================================================================
# 14. OUTPUT DIRECTORIES
# ==============================================================================

TFIDF_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "tfidf"
)

TFIDF_OUTPUT_ROOT = (
    TFIDF_ROOT
    / "outputs"
)

TFIDF_AUDIT_ROOT = (
    TFIDF_ROOT
    / "audit"
)

TFIDF_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TFIDF_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==============================================================================
# 15. OUTPUT PATHS
# ==============================================================================

TFIDF_OOF_PATH = (
    TFIDF_OUTPUT_ROOT
    / "tfidf_5fold_oof.parquet"
)

TFIDF_METRICS_PATH = (
    TFIDF_AUDIT_ROOT
    / "cell10_tfidf_oof_metrics.json"
)

TFIDF_FOLD_METRICS_PATH = (
    TFIDF_AUDIT_ROOT
    / "cell10_tfidf_fold_metrics.parquet"
)


# ==============================================================================
# 16. SAVE OOF
# ==============================================================================

tfidf_oof.to_parquet(
    TFIDF_OOF_PATH,
    index=False,
)

assert (
    TFIDF_OOF_PATH.exists()
)

print(
    "\nTF-IDF OOF parquet : PASS"
)

print(
    "Path:",
    TFIDF_OOF_PATH,
)


# ==============================================================================
# 17. SAVE FOLD METRICS
# ==============================================================================

tfidf_fold_metrics = (
    pd.DataFrame(
        fold_metrics
    )
)

tfidf_fold_metrics.to_parquet(
    TFIDF_FOLD_METRICS_PATH,
    index=False,
)

assert (
    TFIDF_FOLD_METRICS_PATH.exists()
)

print(
    "TF-IDF fold metrics : PASS"
)


# ==============================================================================
# 18. SAVE METRICS JSON
# ==============================================================================

cell10_metrics = {
    "status": "PASS",

    "rows": int(
        len(tfidf_oof)
    ),

    "unique_responses": int(
        tfidf_oof[
            "response_id"
        ].nunique()
    ),

    "text_source": (
        "objective_text + evidence_text"
    ),

    "frozen_evidence": str(
        FROZEN_EVIDENCE_LOCAL
    ),

    "oof_log_loss": float(
        tfidf_log_loss
    ),

    "oof_roc_auc": float(
        tfidf_auc
    ),

    "positive_class_log_loss": (
        tfidf_positive_ll
    ),

    "negative_class_log_loss": (
        tfidf_negative_ll
    ),

    "modernbert_oof_log_loss": (
        MODERNBERT_OOF_LOG_LOSS
    ),

    "structured_prior_oof_log_loss": (
        STRUCTURED_PRIOR_OOF_LOG_LOSS
    ),

    "delta_vs_modernbert": (
        float(
            tfidf_delta_vs_modernbert
        )
    ),

    "delta_vs_structured_prior": (
        float(
            tfidf_delta_vs_structured
        )
    ),

    "tfidf_max_features": (
        TFIDF_MAX_FEATURES
    ),

    "tfidf_ngram_range": list(
        TFIDF_NGRAM_RANGE
    ),

    "tfidf_min_df": (
        TFIDF_MIN_DF
    ),

    "tfidf_max_df": (
        TFIDF_MAX_DF
    ),

    "tfidf_sublinear_tf": (
        TFIDF_SUBLINEAR_TF
    ),

    "tfidf_norm": (
        TFIDF_NORM
    ),

    "logistic_C": (
        LOGISTIC_C
    ),

    "outer_five_fold_oof": True,

    "vectorizer_fit_on_validation": False,

    "classifier_fit_on_validation": False,

    "target_leakage": False,
}


with open(
    TFIDF_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell10_metrics,
        f,
        indent=2,
    )

assert (
    TFIDF_METRICS_PATH.exists()
)

print(
    "Metrics JSON : PASS"
)


# ==============================================================================
# 19. FINAL STATUS
# ==============================================================================

TFIDF_OOF_READY = True

print("\n" + "=" * 100)
print(
    "TRACE THE RACE — CELL 10 FINAL STATUS"
)
print("=" * 100)

print(
    "Frozen evidence input      : PASS"
)

print(
    "Objective + evidence text  : PASS"
)

print(
    "Strict outer 5-fold OOF    : PASS"
)

print(
    "Training-only TF-IDF fit   : PASS"
)

print(
    "Training-only classifier   : PASS"
)

print(
    "Response alignment         : PASS"
)

print(
    "Target leakage guard       : PASS"
)

print(
    "OOF population             : PASS"
)

print(
    "TF-IDF OOF Log Loss        :",
    f"{tfidf_log_loss:.12f}",
)

print(
    "TF-IDF OOF ROC-AUC         :",
    f"{tfidf_auc:.12f}",
)

print(
    "\nCELL 10 COMPLETE — PASS"
)


# ==============================================================================
# 20. MEMORY CLEANUP
# ==============================================================================

del tfidf_prediction
del tfidf_y
del tfidf_positive_mask
del tfidf_negative_mask
del fold_metrics
del tfidf_fold_metrics
del evidence_df

gc.collect()

print(
    "Cell 10 memory cleanup : PASS"
)

TRACE THE RACE — CELL 10
TF-IDF + Logistic Regression — Strict 5-Fold OOF

Cell 9 dependency : PASS

Frozen evidence : PASS
Path: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\evidence_packs.parquet

Frozen evidence rows: 35,072
Evidence schema : PASS
Five-fold contract : PASS
Target alignment : PASS
TF-IDF text construction : PASS
Text source : objective_text + evidence_text

STRICT OUTER 5-FOLD TF-IDF OOF

Processing validation fold 0...
Vocabulary: 150,000
Train matrix: (28114, 150000)
Valid matrix: (6958, 150000)
Fold 0: train=28,114 | valid=6,958 | LL=0.562919 | AUC=0.693895

Processing validation fold 1...
Vocabulary: 150,000
Train matrix: (28022, 150000)
Valid matrix: (7050, 150000)
Fold 1: train=28,022 | valid=7,050 | LL=0.555107 | AUC=0.702412

Processing validation fold 2...
Vocabulary: 150,000
Train matrix: (28049, 150000)
Valid matrix: (7023, 150000)
Fold 2: train=28,049 | valid=7,023 | LL=0.561233 | AUC=0.698941

Processing validation 

In [18]:
# ==============================================================================
# TRACE THE RACE — CELL 11 REPAIR
# Fix duplicate session_id before Parquet serialization
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd


print("=" * 100)
print(
    "TRACE THE RACE — CELL 11 REPAIR"
)
print(
    "OOF Blend Artifact Serialization"
)
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY
# ==============================================================================

assert (
    "blend_table" in globals()
), "blend_table is missing."

assert (
    "best_prediction" in globals()
), "best_prediction is missing."

assert (
    "best_weights" in globals()
), "best_weights is missing."

assert (
    len(blend_table)
    ==
    35_072
), "Blend table population mismatch."


print(
    "\nBlend optimization state : PASS"
)


# ==============================================================================
# 2. CHECK CURRENT DUPLICATE COLUMNS
# ==============================================================================

print(
    "\nCurrent columns:"
)

print(
    list(blend_table.columns)
)

duplicate_columns = (
    blend_table.columns[
        blend_table.columns.duplicated()
    ]
    .tolist()
)

print(
    "Duplicate columns:",
    duplicate_columns
)


# ==============================================================================
# 3. REBUILD CLEAN RESPONSE-LEVEL OOF TABLE
#
# Do NOT reuse the duplicated session_id structure.
# Use the canonical values from the already aligned table.
# ==============================================================================

clean_blend_oof = pd.DataFrame(
    {
        "response_id":
            blend_table[
                "response_id"
            ].to_numpy(),

        "session_id":
            blend_table[
                "session_id"
            ].iloc[:, 0]
            .to_numpy()
            if isinstance(
                blend_table["session_id"],
                pd.DataFrame,
            )
            else
            blend_table[
                "session_id"
            ].to_numpy(),

        "fold":
            blend_table[
                "fold"
            ].to_numpy(),

        "target":
            blend_table[
                "target"
            ].to_numpy(),

        "modernbert_prediction":
            np.asarray(
                m,
                dtype=np.float64,
            ),

        "structured_prediction":
            np.asarray(
                s,
                dtype=np.float64,
            ),

        "tfidf_prediction":
            np.asarray(
                t,
                dtype=np.float64,
            ),

        "blend_prediction":
            np.asarray(
                best_prediction,
                dtype=np.float64,
            ),
    }
)


# ==============================================================================
# 4. EXACT SCHEMA CONTRACT
# ==============================================================================

EXPECTED_BLEND_COLUMNS = [
    "response_id",
    "session_id",
    "fold",
    "target",
    "modernbert_prediction",
    "structured_prediction",
    "tfidf_prediction",
    "blend_prediction",
]

assert (
    clean_blend_oof.columns.tolist()
    ==
    EXPECTED_BLEND_COLUMNS
), (
    "Clean blend schema mismatch."
)

assert (
    clean_blend_oof.columns.is_unique
), (
    "Duplicate columns still exist."
)

assert (
    len(clean_blend_oof)
    ==
    35_072
)

assert (
    clean_blend_oof[
        "response_id"
    ].is_unique
)

assert (
    clean_blend_oof[
        "target"
    ]
    .isin([0, 1])
    .all()
)

assert (
    clean_blend_oof[
        "fold"
    ]
    .isin([0, 1, 2, 3, 4])
    .all()
)


print(
    "\nClean schema : PASS"
)

print(
    "Unique columns:",
    clean_blend_oof.columns.is_unique,
)


# ==============================================================================
# 5. PREDICTION CONTRACT
# ==============================================================================

prediction_columns = [
    "modernbert_prediction",
    "structured_prediction",
    "tfidf_prediction",
    "blend_prediction",
]

for column in prediction_columns:

    values = (
        clean_blend_oof[
            column
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    assert np.isfinite(
        values
    ).all(), (
        f"{column} contains non-finite values."
    )

    assert (
        (
            values
            >
            0.0
        )
        &
        (
            values
            <
            1.0
        )
    ).all(), (
        f"{column} contains invalid probabilities."
    )


print(
    "Prediction contract : PASS"
)


# ==============================================================================
# 6. EXACT FOLD COUNTS
# ==============================================================================

expected_fold_counts = {
    0: 6958,
    1: 7050,
    2: 7023,
    3: 7081,
    4: 6960,
}

observed_fold_counts = (
    clean_blend_oof[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

assert (
    observed_fold_counts
    ==
    expected_fold_counts
), (
    "Fold counts mismatch."
)

print(
    "Fold contract : PASS"
)


# ==============================================================================
# 7. OUTPUT PATHS
# ==============================================================================

BLEND_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "blend"
)

BLEND_OUTPUT_ROOT = (
    BLEND_ROOT
    / "outputs"
)

BLEND_AUDIT_ROOT = (
    BLEND_ROOT
    / "audit"
)

BLEND_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BLEND_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BLEND_OOF_PATH = (
    BLEND_OUTPUT_ROOT
    / "blend_5fold_oof.parquet"
)

BLEND_WEIGHTS_PATH = (
    BLEND_OUTPUT_ROOT
    / "blend_weights.json"
)

BLEND_METRICS_PATH = (
    BLEND_AUDIT_ROOT
    / "cell11_blend_metrics.json"
)


# ==============================================================================
# 8. WRITE CLEAN OOF
# ==============================================================================

clean_blend_oof.to_parquet(
    BLEND_OOF_PATH,
    index=False,
)

assert (
    BLEND_OOF_PATH.exists()
)

print(
    "\nBlend OOF parquet : PASS"
)

print(
    "Path:",
    BLEND_OOF_PATH,
)


# ==============================================================================
# 9. RELOAD TEST
# ==============================================================================

reload_test = pd.read_parquet(
    BLEND_OOF_PATH
)

assert (
    reload_test.columns.tolist()
    ==
    EXPECTED_BLEND_COLUMNS
)

assert (
    len(reload_test)
    ==
    35_072
)

assert (
    reload_test[
        "response_id"
    ].is_unique
)

assert (
    reload_test[
        "blend_prediction"
    ].notna()
    .all()
)

print(
    "Parquet reload : PASS"
)


# ==============================================================================
# 10. WEIGHTS
# ==============================================================================

best_m, best_s, best_t = (
    best_weights
)

assert (
    abs(
        best_m
        +
        best_s
        +
        best_t
        -
        1.0
    )
    <
    1e-9
)

blend_weights = {
    "modernbert": float(
        best_m
    ),
    "structured_prior": float(
        best_s
    ),
    "tfidf": float(
        best_t
    ),
    "weight_sum": float(
        best_m
        +
        best_s
        +
        best_t
    ),
}


with open(
    BLEND_WEIGHTS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        blend_weights,
        f,
        indent=2,
    )

assert (
    BLEND_WEIGHTS_PATH.exists()
)

print(
    "Blend weights JSON : PASS"
)


# ==============================================================================
# 11. METRICS
# ==============================================================================

y_clean = (
    clean_blend_oof[
        "target"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

blend_clean_prediction = (
    clean_blend_oof[
        "blend_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

from sklearn.metrics import (
    log_loss,
    roc_auc_score,
)

clean_blend_ll = log_loss(
    y_clean,
    blend_clean_prediction,
    labels=[
        0,
        1,
    ],
)

clean_blend_auc = roc_auc_score(
    y_clean,
    blend_clean_prediction,
)

modernbert_clean_ll = log_loss(
    y_clean,
    clean_blend_oof[
        "modernbert_prediction"
    ],
    labels=[
        0,
        1,
    ],
)

improvement = (
    modernbert_clean_ll
    -
    clean_blend_ll
)


cell11_metrics = {
    "status": "PASS",

    "rows": int(
        len(clean_blend_oof)
    ),

    "unique_responses": int(
        clean_blend_oof[
            "response_id"
        ].nunique()
    ),

    "modernbert_log_loss": float(
        modernbert_clean_ll
    ),

    "blend_log_loss": float(
        clean_blend_ll
    ),

    "blend_roc_auc": float(
        clean_blend_auc
    ),

    "improvement_vs_modernbert": float(
        improvement
    ),

    "weights": blend_weights,

    "strict_oof": True,

    "training_predictions_used": False,
}


with open(
    BLEND_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell11_metrics,
        f,
        indent=2,
    )

assert (
    BLEND_METRICS_PATH.exists()
)

print(
    "Blend metrics JSON : PASS"
)


# ==============================================================================
# 12. FINAL STATUS
# ==============================================================================

OOF_BLEND_READY = True

print("\n" + "=" * 100)
print(
    "TRACE THE RACE — CELL 11 REPAIR FINAL STATUS"
)
print("=" * 100)

print(
    "Duplicate-column repair : PASS"
)

print(
    "Exact OOF population    : PASS"
)

print(
    "Response identity       : PASS"
)

print(
    "Fold contract           : PASS"
)

print(
    "Prediction contract     : PASS"
)

print(
    "Parquet write           : PASS"
)

print(
    "Parquet reload          : PASS"
)

print(
    "\nBlend weights:"
)

print(
    "ModernBERT     :",
    f"{best_m:.4f}"
)

print(
    "Structured     :",
    f"{best_s:.4f}"
)

print(
    "TF-IDF         :",
    f"{best_t:.4f}"
)

print(
    "\nBlend OOF Log Loss:",
    f"{clean_blend_ll:.12f}"
)

print(
    "Blend OOF ROC-AUC:",
    f"{clean_blend_auc:.12f}"
)

print(
    "LL improvement vs ModernBERT:",
    f"{improvement:+.12f}"
)

print(
    "\nCELL 11 COMPLETE — PASS"
)

print(
    "Cell 11 repair memory cleanup : PASS"
)


# ==============================================================================
# 13. CLEANUP
# ==============================================================================

del reload_test
del y_clean
del blend_clean_prediction
del clean_blend_oof

gc.collect()

TRACE THE RACE — CELL 11 REPAIR
OOF Blend Artifact Serialization

Blend optimization state : PASS

Current columns:
['response_id', 'session_id', 'fold', 'target', 'modernbert_prediction', 'structured_prediction', 'session_id', 'tfidf_prediction']
Duplicate columns: ['session_id']

Clean schema : PASS
Unique columns: True
Prediction contract : PASS
Fold contract : PASS

Blend OOF parquet : PASS
Path: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\blend\outputs\blend_5fold_oof.parquet
Parquet reload : PASS
Blend weights JSON : PASS
Blend metrics JSON : PASS

TRACE THE RACE — CELL 11 REPAIR FINAL STATUS
Duplicate-column repair : PASS
Exact OOF population    : PASS
Response identity       : PASS
Fold contract           : PASS
Prediction contract     : PASS
Parquet write           : PASS
Parquet reload          : PASS

Blend weights:
ModernBERT     : 0.4190
Structured     : 0.3510
TF-IDF         : 0.2300

Blend OOF Log Loss: 0.543293053387
Blend OOF ROC-AUC: 0.721930083655

464

In [19]:
# ==============================================================================
# TRACE THE RACE — CELL 12
# OOF BLEND CALIBRATION EVALUATION
#
# Input:
#   Cell 11 optimized blend OOF
#
# Methods:
#   1. Raw optimized blend
#   2. Platt / logistic calibration
#   3. Isotonic calibration
#
# IMPORTANT:
#   This cell does NOT train ModernBERT.
#   This cell does NOT generate new model predictions.
#   It only calibrates the already-generated OOF blend.
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
)


print("=" * 100)
print(
    "TRACE THE RACE — CELL 12"
)
print(
    "OOF Blend Calibration Evaluation"
)
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    globals().get(
        "OOF_BLEND_READY",
        False,
    )
    is True
), (
    "Cell 11 dependency failed. "
    "Run the repaired Cell 11 successfully first."
)

print(
    "\nCell 11 dependency : PASS"
)


# ==============================================================================
# 2. BLEND ARTIFACT PATH
# ==============================================================================

BLEND_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "blend"
)

BLEND_OUTPUT_ROOT = (
    BLEND_ROOT
    / "outputs"
)

BLEND_AUDIT_ROOT = (
    BLEND_ROOT
    / "audit"
)

BLEND_OOF_PATH = (
    BLEND_OUTPUT_ROOT
    / "blend_5fold_oof.parquet"
)

BLEND_WEIGHTS_PATH = (
    BLEND_OUTPUT_ROOT
    / "blend_weights.json"
)

BLEND_METRICS_PATH = (
    BLEND_AUDIT_ROOT
    / "cell11_blend_metrics.json"
)


assert BLEND_OOF_PATH.exists(), (
    f"Blend OOF not found:\n"
    f"{BLEND_OOF_PATH}"
)

assert BLEND_WEIGHTS_PATH.exists(), (
    f"Blend weights not found:\n"
    f"{BLEND_WEIGHTS_PATH}"
)

assert BLEND_METRICS_PATH.exists(), (
    f"Blend metrics not found:\n"
    f"{BLEND_METRICS_PATH}"
)

print(
    "Blend artifacts : PASS"
)


# ==============================================================================
# 3. LOAD BLEND OOF
# ==============================================================================

blend_oof = pd.read_parquet(
    BLEND_OOF_PATH
)


EXPECTED_COLUMNS = [
    "response_id",
    "session_id",
    "fold",
    "target",
    "modernbert_prediction",
    "structured_prediction",
    "tfidf_prediction",
    "blend_prediction",
]


assert (
    blend_oof.columns.tolist()
    ==
    EXPECTED_COLUMNS
), (
    "Blend OOF schema mismatch."
)

assert (
    len(blend_oof)
    ==
    35_072
)

assert (
    blend_oof[
        "response_id"
    ].is_unique
)

assert (
    blend_oof[
        "target"
    ]
    .isin([0, 1])
    .all()
)

assert (
    blend_oof[
        "fold"
    ]
    .isin([0, 1, 2, 3, 4])
    .all()
)

print(
    "Blend OOF schema : PASS"
)

print(
    "Rows:",
    f"{len(blend_oof):,}",
)


# ==============================================================================
# 4. PREDICTION CONTRACT
# ==============================================================================

raw_prediction = (
    blend_oof[
        "blend_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

y = (
    blend_oof[
        "target"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

assert np.isfinite(
    raw_prediction
).all()

assert (
    (
        raw_prediction
        >
        0.0
    )
    &
    (
        raw_prediction
        <
        1.0
    )
).all()

print(
    "Raw blend prediction contract : PASS"
)


# ==============================================================================
# 5. RAW BLEND BASELINE
# ==============================================================================

raw_ll = log_loss(
    y,
    raw_prediction,
    labels=[
        0,
        1,
    ],
)

raw_auc = roc_auc_score(
    y,
    raw_prediction,
)


print("\n" + "=" * 100)
print(
    "RAW BLEND BASELINE"
)
print("=" * 100)

print(
    "Log Loss :",
    f"{raw_ll:.12f}",
)

print(
    "ROC-AUC  :",
    f"{raw_auc:.12f}",
)


# ==============================================================================
# 6. PREPARE LOGIT TRANSFORM
#
# Platt scaling is logistic regression on the log-odds of the
# original prediction.
# ==============================================================================

EPS = 1e-7

clipped_prediction = np.clip(
    raw_prediction,
    EPS,
    1.0 - EPS,
)

blend_logit = np.log(
    clipped_prediction
    /
    (
        1.0
        -
        clipped_prediction
    )
)

assert np.isfinite(
    blend_logit
).all()

print(
    "\nLogit transform : PASS"
)


# ==============================================================================
# 7. PLATT CALIBRATION
# ==============================================================================

platt_model = LogisticRegression(
    C=1e6,
    solver="lbfgs",
    max_iter=1000,
)

platt_model.fit(
    blend_logit.reshape(
        -1,
        1,
    ),
    y,
)

platt_prediction = (
    platt_model
    .predict_proba(
        blend_logit.reshape(
            -1,
            1,
        )
    )[
        :,
        1,
    ]
)

platt_prediction = np.clip(
    platt_prediction,
    EPS,
    1.0 - EPS,
)

assert np.isfinite(
    platt_prediction
).all()

assert (
    (
        platt_prediction
        >
        0.0
    )
    &
    (
        platt_prediction
        <
        1.0
    )
).all()


platt_ll = log_loss(
    y,
    platt_prediction,
    labels=[
        0,
        1,
    ],
)

platt_auc = roc_auc_score(
    y,
    platt_prediction,
)


print("\n" + "=" * 100)
print(
    "PLATT / LOGISTIC CALIBRATION"
)
print("=" * 100)

print(
    "Log Loss :",
    f"{platt_ll:.12f}",
)

print(
    "ROC-AUC  :",
    f"{platt_auc:.12f}",
)


# ==============================================================================
# 8. ISOTONIC CALIBRATION
# ==============================================================================

isotonic_model = IsotonicRegression(
    y_min=EPS,
    y_max=1.0 - EPS,
    out_of_bounds="clip",
)

isotonic_model.fit(
    raw_prediction,
    y,
)

isotonic_prediction = (
    isotonic_model.predict(
        raw_prediction
    )
)

isotonic_prediction = np.clip(
    isotonic_prediction,
    EPS,
    1.0 - EPS,
)

assert np.isfinite(
    isotonic_prediction
).all()

isotonic_ll = log_loss(
    y,
    isotonic_prediction,
    labels=[
        0,
        1,
    ],
)

isotonic_auc = roc_auc_score(
    y,
    isotonic_prediction,
)


print("\n" + "=" * 100)
print(
    "ISOTONIC CALIBRATION"
)
print("=" * 100)

print(
    "Log Loss :",
    f"{isotonic_ll:.12f}",
)

print(
    "ROC-AUC  :",
    f"{isotonic_auc:.12f}",
)


# ==============================================================================
# 9. CALIBRATION COMPARISON
# ==============================================================================

calibration_results = pd.DataFrame(
    [
        {
            "method":
                "raw_blend",
            "log_loss":
                float(
                    raw_ll
                ),
            "roc_auc":
                float(
                    raw_auc
                ),
        },
        {
            "method":
                "platt",
            "log_loss":
                float(
                    platt_ll
                ),
            "roc_auc":
                float(
                    platt_auc
                ),
        },
        {
            "method":
                "isotonic",
            "log_loss":
                float(
                    isotonic_ll
                ),
            "roc_auc":
                float(
                    isotonic_auc
                ),
        },
    ]
)

calibration_results = (
    calibration_results
    .sort_values(
        "log_loss"
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 100)
print(
    "CALIBRATION COMPARISON"
)
print("=" * 100)

print(
    calibration_results.to_string(
        index=False
    )
)


# ==============================================================================
# 10. SELECT BEST METHOD
# ==============================================================================

best_calibration_row = (
    calibration_results.iloc[
        0
    ]
)

best_method = (
    best_calibration_row[
        "method"
    ]
)

best_calibration_ll = float(
    best_calibration_row[
        "log_loss"
    ]
)

best_calibration_auc = float(
    best_calibration_row[
        "roc_auc"
    ]
)


# ==============================================================================
# 11. IMPORTANT DECISION
#
# Calibration is accepted only if it actually improves log loss
# relative to the raw blend.
# ==============================================================================

if (
    best_calibration_ll
    <
    raw_ll
):

    calibration_improves = True

else:

    calibration_improves = False


print("\n" + "=" * 100)
print(
    "CALIBRATION DECISION"
)
print("=" * 100)

print(
    "Raw blend LL       :",
    f"{raw_ll:.12f}",
)

print(
    "Best calibrated LL :",
    f"{best_calibration_ll:.12f}",
)

print(
    "Best method         :",
    best_method,
)

print(
    "LL delta             :",
    f"{raw_ll - best_calibration_ll:+.12f}",
)

print(
    "Calibration improves:",
    calibration_improves,
)


# ==============================================================================
# 12. SELECT PRODUCTION CANDIDATE
#
# If calibration does NOT improve OOF LL, keep raw blend.
# ==============================================================================

if (
    calibration_improves
    and
    best_method
    ==
    "platt"
):

    production_prediction = (
        platt_prediction
    )

    production_method = (
        "platt"
    )

elif (
    calibration_improves
    and
    best_method
    ==
    "isotonic"
):

    production_prediction = (
        isotonic_prediction
    )

    production_method = (
        "isotonic"
    )

else:

    production_prediction = (
        raw_prediction
    )

    production_method = (
        "raw_blend"
    )


production_ll = log_loss(
    y,
    production_prediction,
    labels=[
        0,
        1,
    ],
)

production_auc = roc_auc_score(
    y,
    production_prediction,
)


assert np.isfinite(
    production_prediction
).all()

assert (
    (
        production_prediction
        >
        0.0
    )
    &
    (
        production_prediction
        <
        1.0
    )
).all()


print("\n" + "=" * 100)
print(
    "PRODUCTION CANDIDATE"
)
print("=" * 100)

print(
    "Selected method :",
    production_method,
)

print(
    "Log Loss        :",
    f"{production_ll:.12f}",
)

print(
    "ROC-AUC         :",
    f"{production_auc:.12f}",
)


# ==============================================================================
# 13. OUTPUT PATHS
# ==============================================================================

CALIBRATION_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "calibration"
)

CALIBRATION_OUTPUT_ROOT = (
    CALIBRATION_ROOT
    / "outputs"
)

CALIBRATION_AUDIT_ROOT = (
    CALIBRATION_ROOT
    / "audit"
)

CALIBRATION_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CALIBRATION_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


CALIBRATION_OOF_PATH = (
    CALIBRATION_OUTPUT_ROOT
    / "calibrated_blend_5fold_oof.parquet"
)

CALIBRATION_RESULTS_PATH = (
    CALIBRATION_AUDIT_ROOT
    / "cell12_calibration_results.parquet"
)

CALIBRATION_METRICS_PATH = (
    CALIBRATION_AUDIT_ROOT
    / "cell12_calibration_metrics.json"
)


# ==============================================================================
# 14. SAVE CALIBRATION OOF
# ==============================================================================

calibrated_oof = (
    blend_oof[
        [
            "response_id",
            "session_id",
            "fold",
            "target",
        ]
    ]
    .copy()
)

calibrated_oof[
    "raw_blend_prediction"
] = raw_prediction

calibrated_oof[
    "platt_prediction"
] = platt_prediction

calibrated_oof[
    "isotonic_prediction"
] = isotonic_prediction

calibrated_oof[
    "production_prediction"
] = production_prediction

calibrated_oof.to_parquet(
    CALIBRATION_OOF_PATH,
    index=False,
)

assert (
    CALIBRATION_OOF_PATH.exists()
)

print(
    "\nCalibration OOF parquet : PASS"
)


# ==============================================================================
# 15. RELOAD CONTRACT
# ==============================================================================

calibration_reload = (
    pd.read_parquet(
        CALIBRATION_OOF_PATH
    )
)

assert (
    len(calibration_reload)
    ==
    35_072
)

assert (
    calibration_reload[
        "response_id"
    ].is_unique
)

assert (
    calibration_reload.columns.tolist()
    ==
    [
        "response_id",
        "session_id",
        "fold",
        "target",
        "raw_blend_prediction",
        "platt_prediction",
        "isotonic_prediction",
        "production_prediction",
    ]
)

print(
    "Calibration reload : PASS"
)


# ==============================================================================
# 16. SAVE COMPARISON TABLE
# ==============================================================================

calibration_results.to_parquet(
    CALIBRATION_RESULTS_PATH,
    index=False,
)

assert (
    CALIBRATION_RESULTS_PATH.exists()
)

print(
    "Calibration comparison : PASS"
)


# ==============================================================================
# 17. SAVE METRICS
# ==============================================================================

cell12_metrics = {
    "status":
        "PASS",

    "rows":
        int(
            len(
                calibrated_oof
            )
        ),

    "unique_responses":
        int(
            calibrated_oof[
                "response_id"
            ].nunique()
        ),

    "raw_blend": {
        "log_loss":
            float(
                raw_ll
            ),
        "roc_auc":
            float(
                raw_auc
            ),
    },

    "platt": {
        "log_loss":
            float(
                platt_ll
            ),
        "roc_auc":
            float(
                platt_auc
            ),
    },

    "isotonic": {
        "log_loss":
            float(
                isotonic_ll
            ),
        "roc_auc":
            float(
                isotonic_auc
            ),
    },

    "selected_method":
        production_method,

    "production_log_loss":
        float(
            production_ll
        ),

    "production_roc_auc":
        float(
            production_auc
        ),

    "calibration_improves":
        bool(
            calibration_improves
        ),

    "strict_oof":
        True,

    "modernbert_retraining":
        False,

    "blend_retraining":
        False,
}


with open(
    CALIBRATION_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell12_metrics,
        f,
        indent=2,
    )

assert (
    CALIBRATION_METRICS_PATH.exists()
)


# ==============================================================================
# 18. FINAL STATUS
# ==============================================================================

CALIBRATION_READY = True

print("\n" + "=" * 100)
print(
    "TRACE THE RACE — CELL 12 FINAL STATUS"
)
print("=" * 100)

print(
    "Cell 11 blend dependency : PASS"
)

print(
    "35,072 OOF rows          : PASS"
)

print(
    "Raw blend baseline       : PASS"
)

print(
    "Platt calibration        : PASS"
)

print(
    "Isotonic calibration     : PASS"
)

print(
    "Calibration comparison   : PASS"
)

print(
    "Production selection     : PASS"
)

print(
    "Calibration artifact     : PASS"
)

print(
    "\nSelected production method:",
    production_method,
)

print(
    "Production OOF Log Loss:",
    f"{production_ll:.12f}",
)

print(
    "Production OOF ROC-AUC:",
    f"{production_auc:.12f}",
)

print(
    "\nCELL 12 COMPLETE — PASS"
)


# ==============================================================================
# 19. MEMORY CLEANUP
# ==============================================================================

del calibration_reload
del calibration_results
del calibrated_oof
del blend_oof

del raw_prediction
del clipped_prediction
del blend_logit

del platt_prediction
del isotonic_prediction
del production_prediction

del y

del platt_model
del isotonic_model

gc.collect()

print(
    "Cell 12 memory cleanup : PASS"
)

TRACE THE RACE — CELL 12
OOF Blend Calibration Evaluation

Cell 11 dependency : PASS
Blend artifacts : PASS
Blend OOF schema : PASS
Rows: 35,072
Raw blend prediction contract : PASS

RAW BLEND BASELINE
Log Loss : 0.543293053387
ROC-AUC  : 0.721930083655

Logit transform : PASS

PLATT / LOGISTIC CALIBRATION
Log Loss : 0.543206746350
ROC-AUC  : 0.721930083655

ISOTONIC CALIBRATION
Log Loss : 0.541844408999
ROC-AUC  : 0.723006201459

CALIBRATION COMPARISON
   method  log_loss  roc_auc
 isotonic  0.541844 0.723006
    platt  0.543207 0.721930
raw_blend  0.543293 0.721930

CALIBRATION DECISION
Raw blend LL       : 0.543293053387
Best calibrated LL : 0.541844408999
Best method         : isotonic
LL delta             : +0.001448644388
Calibration improves: True

PRODUCTION CANDIDATE
Selected method : isotonic
Log Loss        : 0.541844408999
ROC-AUC         : 0.723006201459

Calibration OOF parquet : PASS
Calibration reload : PASS
Calibration comparison : PASS

TRACE THE RACE — CELL 12 FINAL 

In [20]:
# ==============================================================================
# TRACE THE RACE — CELL 13
# CROSS-FITTED OOF CALIBRATION
#
# Purpose:
#   Obtain an honest out-of-fold estimate of calibration improvement.
#
# Inputs:
#   Cell 11 optimized blend OOF
#
# Methods:
#   1. Raw blend
#   2. 5-fold cross-fitted Platt calibration
#   3. 5-fold cross-fitted isotonic calibration
#
# IMPORTANT:
#   Calibration model is NEVER fitted on the row being predicted.
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
)


print("=" * 100)
print(
    "TRACE THE RACE — CELL 13"
)
print(
    "5-FOLD CROSS-FITTED CALIBRATION"
)
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY
# ==============================================================================

assert (
    globals().get(
        "OOF_BLEND_READY",
        False,
    )
    is True
), (
    "Cell 11 dependency failed."
)

print(
    "\nCell 11 dependency : PASS"
)


# ==============================================================================
# 2. LOAD BLEND OOF
# ==============================================================================

BLEND_OOF_PATH = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "blend"
    / "outputs"
    / "blend_5fold_oof.parquet"
)

assert BLEND_OOF_PATH.exists(), (
    f"Blend OOF not found:\n{BLEND_OOF_PATH}"
)

blend_oof = pd.read_parquet(
    BLEND_OOF_PATH
)


EXPECTED_COLUMNS = [
    "response_id",
    "session_id",
    "fold",
    "target",
    "modernbert_prediction",
    "structured_prediction",
    "tfidf_prediction",
    "blend_prediction",
]

assert (
    blend_oof.columns.tolist()
    ==
    EXPECTED_COLUMNS
)

assert (
    len(blend_oof)
    ==
    35_072
)

assert (
    blend_oof["response_id"].is_unique
)

assert (
    blend_oof["fold"].isin(
        [0, 1, 2, 3, 4]
    ).all()
)

assert (
    blend_oof["target"].isin(
        [0, 1]
    ).all()
)

print(
    "Blend OOF schema : PASS"
)

print(
    "Rows:",
    f"{len(blend_oof):,}",
)


# ==============================================================================
# 3. ARRAYS
# ==============================================================================

y = (
    blend_oof[
        "target"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

raw_prediction = (
    blend_oof[
        "blend_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

folds = (
    blend_oof[
        "fold"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


assert np.isfinite(
    raw_prediction
).all()

assert (
    (
        raw_prediction > 0.0
    )
    &
    (
        raw_prediction < 1.0
    )
).all()


# ==============================================================================
# 4. RAW BASELINE
# ==============================================================================

raw_ll = log_loss(
    y,
    raw_prediction,
    labels=[0, 1],
)

raw_auc = roc_auc_score(
    y,
    raw_prediction,
)

print("\n" + "=" * 100)
print(
    "RAW BLEND BASELINE"
)
print("=" * 100)

print(
    "Log Loss :",
    f"{raw_ll:.12f}",
)

print(
    "ROC-AUC  :",
    f"{raw_auc:.12f}",
)


# ==============================================================================
# 5. CROSS-FITTED PREDICTION ARRAYS
# ==============================================================================

EPS = 1e-7

platt_oof = np.full(
    len(blend_oof),
    np.nan,
    dtype=np.float64,
)

isotonic_oof = np.full(
    len(blend_oof),
    np.nan,
    dtype=np.float64,
)


# ==============================================================================
# 6. 5-FOLD CROSS-FITTED CALIBRATION
# ==============================================================================

print("\n" + "=" * 100)
print(
    "5-FOLD CROSS-FITTED CALIBRATION"
)
print("=" * 100)


for validation_fold in range(5):

    train_mask = (
        folds
        !=
        validation_fold
    )

    valid_mask = (
        folds
        ==
        validation_fold
    )

    train_count = int(
        train_mask.sum()
    )

    valid_count = int(
        valid_mask.sum()
    )

    assert (
        train_count
        +
        valid_count
        ==
        35_072
    )

    print(
        f"\nCalibration fold "
        f"{validation_fold}..."
    )

    print(
        f"train={train_count:,} | "
        f"valid={valid_count:,}"
    )


    # --------------------------------------------------------------------------
    # Training-side predictions
    # --------------------------------------------------------------------------

    train_prediction = np.clip(
        raw_prediction[
            train_mask
        ],
        EPS,
        1.0 - EPS,
    )

    valid_prediction = np.clip(
        raw_prediction[
            valid_mask
        ],
        EPS,
        1.0 - EPS,
    )

    train_target = y[
        train_mask
    ]


    # --------------------------------------------------------------------------
    # PLATT
    # --------------------------------------------------------------------------

    train_logit = np.log(
        train_prediction
        /
        (
            1.0
            -
            train_prediction
        )
    )

    valid_logit = np.log(
        valid_prediction
        /
        (
            1.0
            -
            valid_prediction
        )
    )

    platt_model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=1000,
    )

    platt_model.fit(
        train_logit.reshape(
            -1,
            1,
        ),
        train_target,
    )

    fold_platt = (
        platt_model
        .predict_proba(
            valid_logit.reshape(
                -1,
                1,
            )
        )[
            :,
            1,
        ]
    )

    fold_platt = np.clip(
        fold_platt,
        EPS,
        1.0 - EPS,
    )

    platt_oof[
        valid_mask
    ] = fold_platt


    # --------------------------------------------------------------------------
    # ISOTONIC
    # --------------------------------------------------------------------------

    isotonic_model = (
        IsotonicRegression(
            y_min=EPS,
            y_max=1.0 - EPS,
            out_of_bounds="clip",
        )
    )

    isotonic_model.fit(
        train_prediction,
        train_target,
    )

    fold_isotonic = (
        isotonic_model.predict(
            valid_prediction
        )
    )

    fold_isotonic = np.clip(
        fold_isotonic,
        EPS,
        1.0 - EPS,
    )

    isotonic_oof[
        valid_mask
    ] = fold_isotonic


    # --------------------------------------------------------------------------
    # Fold contract
    # --------------------------------------------------------------------------

    assert np.isfinite(
        platt_oof[
            valid_mask
        ]
    ).all()

    assert np.isfinite(
        isotonic_oof[
            valid_mask
        ]
    ).all()

    print(
        f"Fold {validation_fold}: "
        f"cross-fitted calibration PASS"
    )


# ==============================================================================
# 7. GLOBAL CROSS-FITTED CONTRACT
# ==============================================================================

assert np.isfinite(
    platt_oof
).all()

assert np.isfinite(
    isotonic_oof
).all()

assert (
    (
        platt_oof > 0.0
    )
    &
    (
        platt_oof < 1.0
    )
).all()

assert (
    (
        isotonic_oof > 0.0
    )
    &
    (
        isotonic_oof < 1.0
    )
).all()


print(
    "\nCross-fitted prediction contract : PASS"
)


# ==============================================================================
# 8. CROSS-FITTED METRICS
# ==============================================================================

platt_ll = log_loss(
    y,
    platt_oof,
    labels=[0, 1],
)

platt_auc = roc_auc_score(
    y,
    platt_oof,
)

isotonic_ll = log_loss(
    y,
    isotonic_oof,
    labels=[0, 1],
)

isotonic_auc = roc_auc_score(
    y,
    isotonic_oof,
)


print("\n" + "=" * 100)
print(
    "HONEST CROSS-FITTED CALIBRATION METRICS"
)
print("=" * 100)

print(
    f"Raw blend       LL={raw_ll:.12f} "
    f"AUC={raw_auc:.12f}"
)

print(
    f"Cross-fit Platt LL={platt_ll:.12f} "
    f"AUC={platt_auc:.12f}"
)

print(
    f"Cross-fit Iso   LL={isotonic_ll:.12f} "
    f"AUC={isotonic_auc:.12f}"
)


# ==============================================================================
# 9. COMPARISON
# ==============================================================================

results = pd.DataFrame(
    [
        {
            "method":
                "raw_blend",
            "log_loss":
                float(raw_ll),
            "roc_auc":
                float(raw_auc),
        },
        {
            "method":
                "cross_fitted_platt",
            "log_loss":
                float(platt_ll),
            "roc_auc":
                float(platt_auc),
        },
        {
            "method":
                "cross_fitted_isotonic",
            "log_loss":
                float(isotonic_ll),
            "roc_auc":
                float(isotonic_auc),
        },
    ]
)

results = (
    results
    .sort_values(
        "log_loss"
    )
    .reset_index(
        drop=True
    )
)

print("\n" + "=" * 100)
print(
    "CROSS-FITTED CALIBRATION COMPARISON"
)
print("=" * 100)

print(
    results.to_string(
        index=False
    )
)


# ==============================================================================
# 10. SELECT ONLY IF HONEST OOF IMPROVES
# ==============================================================================

best_row = results.iloc[0]

best_method = str(
    best_row["method"]
)

best_ll = float(
    best_row["log_loss"]
)

best_auc = float(
    best_row["roc_auc"]
)

calibration_gain = (
    raw_ll
    -
    best_ll
)

if (
    best_ll
    <
    raw_ll
):
    calibration_improves = True
else:
    calibration_improves = False


print("\n" + "=" * 100)
print(
    "HONEST CALIBRATION DECISION"
)
print("=" * 100)

print(
    "Raw blend LL :",
    f"{raw_ll:.12f}",
)

print(
    "Best CF LL   :",
    f"{best_ll:.12f}",
)

print(
    "Best method  :",
    best_method,
)

print(
    "LL gain      :",
    f"{calibration_gain:+.12f}",
)

print(
    "Improves     :",
    calibration_improves,
)


# ==============================================================================
# 11. SAVE CROSS-FITTED OOF
# ==============================================================================

CALIBRATION_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "calibration"
)

CALIBRATION_OUTPUT_ROOT = (
    CALIBRATION_ROOT
    / "outputs"
)

CALIBRATION_AUDIT_ROOT = (
    CALIBRATION_ROOT
    / "audit"
)

CALIBRATION_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CALIBRATION_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


CF_OOF_PATH = (
    CALIBRATION_OUTPUT_ROOT
    / "cross_fitted_calibration_oof.parquet"
)

CF_METRICS_PATH = (
    CALIBRATION_AUDIT_ROOT
    / "cell13_cross_fitted_calibration.json"
)

CF_RESULTS_PATH = (
    CALIBRATION_AUDIT_ROOT
    / "cell13_cross_fitted_comparison.parquet"
)


cross_fitted_oof = (
    blend_oof[
        [
            "response_id",
            "session_id",
            "fold",
            "target",
        ]
    ]
    .copy()
)

cross_fitted_oof[
    "raw_blend_prediction"
] = raw_prediction

cross_fitted_oof[
    "platt_prediction"
] = platt_oof

cross_fitted_oof[
    "isotonic_prediction"
] = isotonic_oof


cross_fitted_oof.to_parquet(
    CF_OOF_PATH,
    index=False,
)

assert CF_OOF_PATH.exists()

print(
    "\nCross-fitted OOF write : PASS"
)


# ==============================================================================
# 12. RELOAD TEST
# ==============================================================================

reload_test = pd.read_parquet(
    CF_OOF_PATH
)

assert (
    len(reload_test)
    ==
    35_072
)

assert (
    reload_test[
        "response_id"
    ].is_unique
)

assert (
    reload_test.columns.tolist()
    ==
    [
        "response_id",
        "session_id",
        "fold",
        "target",
        "raw_blend_prediction",
        "platt_prediction",
        "isotonic_prediction",
    ]
)

print(
    "Cross-fitted OOF reload : PASS"
)


# ==============================================================================
# 13. SAVE COMPARISON
# ==============================================================================

results.to_parquet(
    CF_RESULTS_PATH,
    index=False,
)

assert CF_RESULTS_PATH.exists()

print(
    "Comparison artifact : PASS"
)


# ==============================================================================
# 14. SAVE METRICS
# ==============================================================================

cell13_metrics = {
    "status":
        "PASS",

    "rows":
        int(
            len(
                cross_fitted_oof
            )
        ),

    "unique_responses":
        int(
            cross_fitted_oof[
                "response_id"
            ].nunique()
        ),

    "raw_blend_log_loss":
        float(raw_ll),

    "raw_blend_roc_auc":
        float(raw_auc),

    "cross_fitted_platt_log_loss":
        float(platt_ll),

    "cross_fitted_platt_roc_auc":
        float(platt_auc),

    "cross_fitted_isotonic_log_loss":
        float(isotonic_ll),

    "cross_fitted_isotonic_roc_auc":
        float(isotonic_auc),

    "best_method":
        best_method,

    "best_log_loss":
        float(best_ll),

    "best_roc_auc":
        float(best_auc),

    "calibration_gain":
        float(calibration_gain),

    "calibration_improves":
        bool(calibration_improves),

    "strict_cross_fitted":
        True,

    "calibration_training_rows_excluded_from_validation":
        True,
}


with open(
    CF_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell13_metrics,
        f,
        indent=2,
    )

assert CF_METRICS_PATH.exists()

print(
    "Metrics JSON : PASS"
)


# ==============================================================================
# 15. FINAL STATUS
# ==============================================================================

CROSS_FITTED_CALIBRATION_READY = True

print("\n" + "=" * 100)
print(
    "TRACE THE RACE — CELL 13 FINAL STATUS"
)
print("=" * 100)

print(
    "Cell 11 blend dependency       : PASS"
)

print(
    "35,072-row OOF population      : PASS"
)

print(
    "5-fold cross-fitted Platt      : PASS"
)

print(
    "5-fold cross-fitted isotonic   : PASS"
)

print(
    "No same-row calibration fit    : PASS"
)

print(
    "Cross-fitted OOF artifact      : PASS"
)

print(
    "Calibration comparison        : PASS"
)

print(
    "\nBEST HONEST CALIBRATION METHOD:",
    best_method,
)

print(
    "BEST CROSS-FITTED OOF LL:",
    f"{best_ll:.12f}",
)

print(
    "BEST CROSS-FITTED OOF AUC:",
    f"{best_auc:.12f}",
)

print(
    "GAIN VS RAW BLEND:",
    f"{calibration_gain:+.12f}",
)

print(
    "\nCELL 13 COMPLETE — PASS"
)


# ==============================================================================
# 16. MEMORY CLEANUP
# ==============================================================================

del reload_test
del results
del cross_fitted_oof

del raw_prediction
del platt_oof
del isotonic_oof

del blend_oof
del y
del folds

gc.collect()

print(
    "Cell 13 memory cleanup : PASS"
)

TRACE THE RACE — CELL 13
5-FOLD CROSS-FITTED CALIBRATION

Cell 11 dependency : PASS
Blend OOF schema : PASS
Rows: 35,072

RAW BLEND BASELINE
Log Loss : 0.543293053387
ROC-AUC  : 0.721930083655

5-FOLD CROSS-FITTED CALIBRATION

Calibration fold 0...
train=28,114 | valid=6,958
Fold 0: cross-fitted calibration PASS

Calibration fold 1...
train=28,022 | valid=7,050
Fold 1: cross-fitted calibration PASS

Calibration fold 2...
train=28,049 | valid=7,023
Fold 2: cross-fitted calibration PASS

Calibration fold 3...
train=27,991 | valid=7,081
Fold 3: cross-fitted calibration PASS

Calibration fold 4...
train=28,112 | valid=6,960
Fold 4: cross-fitted calibration PASS

Cross-fitted prediction contract : PASS

HONEST CROSS-FITTED CALIBRATION METRICS
Raw blend       LL=0.543293053387 AUC=0.721930083655
Cross-fit Platt LL=0.543286106288 AUC=0.721860181274
Cross-fit Iso   LL=0.544229056994 AUC=0.720044607840

CROSS-FITTED CALIBRATION COMPARISON
               method  log_loss  roc_auc
   cross_fitted


### Current honest result

| Candidate             |       OOF Log Loss |        OOF ROC-AUC |
| --------------------- | -----------------: | -----------------: |
| **Raw blend**         | **0.543293053387** | **0.721930083655** |
| Cross-fitted Platt    | **0.543286106288** |     0.721860181274 |
| Cross-fitted Isotonic |     0.544229056994 |     0.720044607840 |

Platt improves Log Loss by only:

```text
0.000006947099
```

That's **6.95e-6**. Practically, this is negligible.

So I would **not claim that calibration meaningfully improves the model**. The statistically clean conclusion is:

> **Raw blend and cross-fitted Platt are essentially tied; isotonic is worse.**

Also notice that Platt's ROC-AUC is slightly lower. That's expected because calibration is a monotonic transformation and should generally preserve ranking; the tiny AUC difference here comes from the particular cross-fitted transformations.

## What I would lock

For the final architecture, I'd choose:

```text
ModernBERT OOF
      +
Structured + Prior OOF
      +
TF-IDF OOF
      ↓
Optimized convex blend
      ↓
RAW BLEND
      ↓
FINAL
```

**Not isotonic.**

For Platt, the gain is so tiny that adding another fitted transformation adds complexity without meaningful evidence of improvement. Unless the competition metric is specifically sensitive enough that `6.95e-6` matters, **raw blend is the cleaner choice**.

### Current full pipeline

```text
                    TRACE THE RACE
                         │
                         ▼
              ModernBERT 5-fold OOF
              LL 0.547551451901
                         │
          ┌──────────────┴──────────────┐
          ▼                             ▼
 Structured + Prior                 TF-IDF
 LL 0.549264028594               LL 0.558696844276
          │                             │
          └──────────────┬──────────────┘
                         ▼
                  OOF Blend
                LL 0.543293053387
                AUC 0.721930083655
                         │
                         ▼
              Cross-fitted calibration
                         │
              ┌──────────┴──────────┐
              ▼                     ▼
            Platt                 Isotonic
          0.543286              0.544229
              │
              ▼
        Difference negligible
              │
              ▼
          RAW BLEND LOCK
```




In [22]:
# ==============================================================================
# TRACE THE RACE — CELL 14 REPAIR
# FINAL MODEL / BLEND LOCK
#
# IMPORTANT:
#   Does NOT assume any particular key names inside Cell 11 metrics JSON.
#   All final metrics are recomputed directly from the locked OOF artifacts.
#
# FINAL PRODUCTION METHOD:
#   raw_blend
#
# No new training.
# No GPU required.
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score


print("=" * 100)
print(
    "TRACE THE RACE — CELL 14 REPAIR"
)
print(
    "FINAL MODEL / BLEND LOCK"
)
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATES
# ==============================================================================

assert (
    globals().get(
        "OOF_BLEND_READY",
        False,
    )
    is True
), (
    "Cell 11 dependency failed. "
    "Run repaired Cell 11 first."
)

assert (
    globals().get(
        "CROSS_FITTED_CALIBRATION_READY",
        False,
    )
    is True
), (
    "Cell 13 dependency failed. "
    "Run Cell 13 first."
)

print(
    "\nCell 11 dependency : PASS"
)

print(
    "Cell 13 dependency : PASS"
)


# ==============================================================================
# 2. PATH CONFIGURATION
# ==============================================================================

BLEND_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "blend"
)

BLEND_OUTPUT_ROOT = (
    BLEND_ROOT
    / "outputs"
)

BLEND_AUDIT_ROOT = (
    BLEND_ROOT
    / "audit"
)

CALIBRATION_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "calibration"
)

CALIBRATION_OUTPUT_ROOT = (
    CALIBRATION_ROOT
    / "outputs"
)

CALIBRATION_AUDIT_ROOT = (
    CALIBRATION_ROOT
    / "audit"
)


BLEND_OOF_PATH = (
    BLEND_OUTPUT_ROOT
    / "blend_5fold_oof.parquet"
)

BLEND_WEIGHTS_PATH = (
    BLEND_OUTPUT_ROOT
    / "blend_weights.json"
)

CF_OOF_PATH = (
    CALIBRATION_OUTPUT_ROOT
    / "cross_fitted_calibration_oof.parquet"
)

CF_METRICS_PATH = (
    CALIBRATION_AUDIT_ROOT
    / "cell13_cross_fitted_calibration.json"
)


assert BLEND_OOF_PATH.exists(), (
    f"Missing blend OOF:\n{BLEND_OOF_PATH}"
)

assert BLEND_WEIGHTS_PATH.exists(), (
    f"Missing blend weights:\n{BLEND_WEIGHTS_PATH}"
)

assert CF_OOF_PATH.exists(), (
    f"Missing cross-fitted calibration OOF:\n{CF_OOF_PATH}"
)

assert CF_METRICS_PATH.exists(), (
    f"Missing Cell 13 metrics:\n{CF_METRICS_PATH}"
)

print(
    "\nRequired artifacts : PASS"
)


# ==============================================================================
# 3. LOAD ARTIFACTS
# ==============================================================================

blend_oof = pd.read_parquet(
    BLEND_OOF_PATH
)

cross_fitted_oof = pd.read_parquet(
    CF_OOF_PATH
)

with open(
    BLEND_WEIGHTS_PATH,
    "r",
    encoding="utf-8",
) as f:
    blend_weights = json.load(f)

with open(
    CF_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    cf_metrics = json.load(f)


# ==============================================================================
# 4. BLEND SCHEMA
# ==============================================================================

EXPECTED_BLEND_COLUMNS = [
    "response_id",
    "session_id",
    "fold",
    "target",
    "modernbert_prediction",
    "structured_prediction",
    "tfidf_prediction",
    "blend_prediction",
]

assert (
    blend_oof.columns.tolist()
    ==
    EXPECTED_BLEND_COLUMNS
), (
    "Blend OOF schema mismatch."
)

assert (
    blend_oof.columns.is_unique
)

assert (
    len(blend_oof)
    ==
    35_072
)

assert (
    blend_oof[
        "response_id"
    ].is_unique
)

assert (
    blend_oof[
        "fold"
    ]
    .isin([0, 1, 2, 3, 4])
    .all()
)

assert (
    blend_oof[
        "target"
    ]
    .isin([0, 1])
    .all()
)

print(
    "\nBlend OOF schema : PASS"
)

print(
    "Rows:",
    f"{len(blend_oof):,}"
)


# ==============================================================================
# 5. CROSS-FITTED SCHEMA
# ==============================================================================

EXPECTED_CF_COLUMNS = [
    "response_id",
    "session_id",
    "fold",
    "target",
    "raw_blend_prediction",
    "platt_prediction",
    "isotonic_prediction",
]

assert (
    cross_fitted_oof.columns.tolist()
    ==
    EXPECTED_CF_COLUMNS
)

assert (
    len(cross_fitted_oof)
    ==
    35_072
)

assert (
    cross_fitted_oof[
        "response_id"
    ].is_unique
)

print(
    "Cross-fitted OOF schema : PASS"
)


# ==============================================================================
# 6. EXACT ALIGNMENT
# ==============================================================================

assert np.array_equal(
    blend_oof[
        "response_id"
    ].to_numpy(),
    cross_fitted_oof[
        "response_id"
    ].to_numpy(),
)

assert np.array_equal(
    blend_oof[
        "session_id"
    ].to_numpy(),
    cross_fitted_oof[
        "session_id"
    ].to_numpy(),
)

assert np.array_equal(
    blend_oof[
        "fold"
    ].to_numpy(),
    cross_fitted_oof[
        "fold"
    ].to_numpy(),
)

assert np.array_equal(
    blend_oof[
        "target"
    ].to_numpy(),
    cross_fitted_oof[
        "target"
    ].to_numpy(),
)

print(
    "\nExact response alignment : PASS"
)

print(
    "Exact session alignment  : PASS"
)

print(
    "Exact fold alignment     : PASS"
)

print(
    "Exact target alignment   : PASS"
)


# ==============================================================================
# 7. LOAD COMPONENT PREDICTIONS
# ==============================================================================

y = (
    blend_oof[
        "target"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

modernbert_prediction = (
    blend_oof[
        "modernbert_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

structured_prediction = (
    blend_oof[
        "structured_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

tfidf_prediction = (
    blend_oof[
        "tfidf_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

saved_blend = (
    blend_oof[
        "blend_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


# ==============================================================================
# 8. NUMERICAL CONTRACT
# ==============================================================================

for name, values in (
    (
        "ModernBERT",
        modernbert_prediction,
    ),
    (
        "Structured+Prior",
        structured_prediction,
    ),
    (
        "TF-IDF",
        tfidf_prediction,
    ),
    (
        "Blend",
        saved_blend,
    ),
):

    assert np.isfinite(
        values
    ).all(), (
        f"{name} contains non-finite predictions."
    )

    assert (
        (
            values > 0.0
        )
        &
        (
            values < 1.0
        )
    ).all(), (
        f"{name} prediction outside (0,1)."
    )


print(
    "\nPrediction numerical contract : PASS"
)


# ==============================================================================
# 9. WEIGHT DISCOVERY
#
# The repair deliberately supports either:
#
#   modernbert
#   structured_prior
#   tfidf
#
# OR common alternate naming variants.
# ==============================================================================

def resolve_weight(
    source_dict,
    candidates,
    source_name,
):
    for key in candidates:
        if key in source_dict:
            return float(
                source_dict[key]
            )

    raise KeyError(
        f"Could not find {source_name} "
        f"weight in blend_weights.json.\n"
        f"Available keys: "
        f"{list(source_dict.keys())}"
    )


w_modernbert = resolve_weight(
    blend_weights,
    [
        "modernbert",
        "modernbert_weight",
        "w_modernbert",
    ],
    "ModernBERT",
)

w_structured = resolve_weight(
    blend_weights,
    [
        "structured_prior",
        "structured",
        "structured_weight",
        "w_structured",
    ],
    "Structured+Prior",
)

w_tfidf = resolve_weight(
    blend_weights,
    [
        "tfidf",
        "tfidf_weight",
        "w_tfidf",
    ],
    "TF-IDF",
)


assert (
    w_modernbert >= 0.0
)

assert (
    w_structured >= 0.0
)

assert (
    w_tfidf >= 0.0
)

weight_sum = (
    w_modernbert
    +
    w_structured
    +
    w_tfidf
)

assert abs(
    weight_sum - 1.0
) < 1e-8, (
    f"Blend weights do not sum to 1: "
    f"{weight_sum}"
)

print(
    "\nBlend weights : PASS"
)

print(
    "ModernBERT     :",
    f"{w_modernbert:.8f}",
)

print(
    "Structured     :",
    f"{w_structured:.8f}",
)

print(
    "TF-IDF         :",
    f"{w_tfidf:.8f}",
)

print(
    "Weight sum     :",
    f"{weight_sum:.12f}",
)


# ==============================================================================
# 10. RECOMPUTE BLEND
# ==============================================================================

recomputed_blend = (
    w_modernbert
    *
    modernbert_prediction
    +
    w_structured
    *
    structured_prediction
    +
    w_tfidf
    *
    tfidf_prediction
)

recomputed_blend = np.clip(
    recomputed_blend,
    1e-7,
    1.0 - 1e-7,
)

max_blend_difference = float(
    np.max(
        np.abs(
            saved_blend
            -
            recomputed_blend
        )
    )
)

assert (
    max_blend_difference
    <
    1e-10
), (
    "Saved blend cannot be reproduced "
    "from component predictions and weights."
)

print(
    "\nBlend recomputation : PASS"
)

print(
    "Maximum difference:",
    f"{max_blend_difference:.3e}"
)


# ==============================================================================
# 11. RECOMPUTE ALL COMPONENT METRICS DIRECTLY
#
# This avoids depending on Cell 11 JSON key names.
# ==============================================================================

modernbert_ll = log_loss(
    y,
    modernbert_prediction,
    labels=[0, 1],
)

modernbert_auc = roc_auc_score(
    y,
    modernbert_prediction,
)

structured_ll = log_loss(
    y,
    structured_prediction,
    labels=[0, 1],
)

structured_auc = roc_auc_score(
    y,
    structured_prediction,
)

tfidf_ll = log_loss(
    y,
    tfidf_prediction,
    labels=[0, 1],
)

tfidf_auc = roc_auc_score(
    y,
    tfidf_prediction,
)

raw_blend_ll = log_loss(
    y,
    saved_blend,
    labels=[0, 1],
)

raw_blend_auc = roc_auc_score(
    y,
    saved_blend,
)


print("\n" + "=" * 100)
print(
    "RECOMPUTED OOF METRICS"
)
print("=" * 100)

print(
    "ModernBERT       LL:",
    f"{modernbert_ll:.12f}",
    "| AUC:",
    f"{modernbert_auc:.12f}",
)

print(
    "Structured+Prior LL:",
    f"{structured_ll:.12f}",
    "| AUC:",
    f"{structured_auc:.12f}",
)

print(
    "TF-IDF           LL:",
    f"{tfidf_ll:.12f}",
    "| AUC:",
    f"{tfidf_auc:.12f}",
)

print(
    "Raw Blend        LL:",
    f"{raw_blend_ll:.12f}",
    "| AUC:",
    f"{raw_blend_auc:.12f}",
)


# ==============================================================================
# 12. CROSS-FITTED CALIBRATION METRICS
# ==============================================================================

cf_raw = (
    cross_fitted_oof[
        "raw_blend_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

cf_platt = (
    cross_fitted_oof[
        "platt_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

cf_isotonic = (
    cross_fitted_oof[
        "isotonic_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


cf_raw_ll = log_loss(
    y,
    cf_raw,
    labels=[0, 1],
)

cf_raw_auc = roc_auc_score(
    y,
    cf_raw,
)

cf_platt_ll = log_loss(
    y,
    cf_platt,
    labels=[0, 1],
)

cf_platt_auc = roc_auc_score(
    y,
    cf_platt,
)

cf_isotonic_ll = log_loss(
    y,
    cf_isotonic,
    labels=[0, 1],
)

cf_isotonic_auc = roc_auc_score(
    y,
    cf_isotonic,
)


print("\n" + "=" * 100)
print(
    "CROSS-FITTED CALIBRATION"
)
print("=" * 100)

print(
    "Raw blend       LL:",
    f"{cf_raw_ll:.12f}",
    "| AUC:",
    f"{cf_raw_auc:.12f}",
)

print(
    "Platt           LL:",
    f"{cf_platt_ll:.12f}",
    "| AUC:",
    f"{cf_platt_auc:.12f}",
)

print(
    "Isotonic        LL:",
    f"{cf_isotonic_ll:.12f}",
    "| AUC:",
    f"{cf_isotonic_auc:.12f}",
)


# ==============================================================================
# 13. FINAL PRODUCTION DECISION
#
# We deliberately lock RAW BLEND.
#
# Reason:
#   Cross-fitted Platt improvement is only ~6.95e-6.
#   Isotonic is worse.
#   Raw blend is simpler and has slightly better AUC.
# ==============================================================================

FINAL_PRODUCTION_METHOD = (
    "raw_blend"
)

FINAL_PRODUCTION_PREDICTION = (
    saved_blend.copy()
)

FINAL_PRODUCTION_LL = (
    raw_blend_ll
)

FINAL_PRODUCTION_AUC = (
    raw_blend_auc
)


assert (
    FINAL_PRODUCTION_METHOD
    ==
    "raw_blend"
)

print(
    "\nProduction method : PASS"
)

print(
    "Selected:",
    FINAL_PRODUCTION_METHOD
)


# ==============================================================================
# 14. FINAL OUTPUT DIRECTORIES
# ==============================================================================

FINAL_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "final"
)

FINAL_OUTPUT_ROOT = (
    FINAL_ROOT
    / "outputs"
)

FINAL_AUDIT_ROOT = (
    FINAL_ROOT
    / "audit"
)

FINAL_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


FINAL_OOF_PATH = (
    FINAL_OUTPUT_ROOT
    / "final_production_oof.parquet"
)

FINAL_METRICS_PATH = (
    FINAL_AUDIT_ROOT
    / "cell14_final_metrics.json"
)

FINAL_MANIFEST_PATH = (
    FINAL_AUDIT_ROOT
    / "cell14_final_manifest.json"
)


# ==============================================================================
# 15. FINAL OOF
# ==============================================================================

final_oof = (
    blend_oof[
        [
            "response_id",
            "session_id",
            "fold",
            "target",
            "modernbert_prediction",
            "structured_prediction",
            "tfidf_prediction",
        ]
    ]
    .copy()
)

final_oof[
    "final_prediction"
] = FINAL_PRODUCTION_PREDICTION


EXPECTED_FINAL_COLUMNS = [
    "response_id",
    "session_id",
    "fold",
    "target",
    "modernbert_prediction",
    "structured_prediction",
    "tfidf_prediction",
    "final_prediction",
]

assert (
    final_oof.columns.tolist()
    ==
    EXPECTED_FINAL_COLUMNS
)

assert (
    final_oof.columns.is_unique
)

assert (
    len(final_oof)
    ==
    35_072
)

assert (
    final_oof[
        "response_id"
    ].is_unique
)


final_oof.to_parquet(
    FINAL_OOF_PATH,
    index=False,
)

assert FINAL_OOF_PATH.exists()

print(
    "\nFinal OOF write : PASS"
)


# ==============================================================================
# 16. RELOAD FINAL OOF
# ==============================================================================

final_reload = pd.read_parquet(
    FINAL_OOF_PATH
)

assert (
    final_reload.columns.tolist()
    ==
    EXPECTED_FINAL_COLUMNS
)

assert (
    len(final_reload)
    ==
    35_072
)

assert (
    final_reload[
        "response_id"
    ].is_unique
)

assert np.allclose(
    final_reload[
        "final_prediction"
    ].to_numpy(
        dtype=np.float64
    ),
    FINAL_PRODUCTION_PREDICTION,
    rtol=0.0,
    atol=1e-12,
)

print(
    "Final OOF reload : PASS"
)


# ==============================================================================
# 17. FINAL METRICS JSON
# ==============================================================================

final_metrics = {
    "status": "PASS",

    "population": {
        "rows": 35072,
        "unique_responses": 35072,
        "folds": 5,
    },

    "component_metrics": {
        "modernbert": {
            "log_loss":
                float(modernbert_ll),
            "roc_auc":
                float(modernbert_auc),
        },

        "structured_prior": {
            "log_loss":
                float(structured_ll),
            "roc_auc":
                float(structured_auc),
        },

        "tfidf": {
            "log_loss":
                float(tfidf_ll),
            "roc_auc":
                float(tfidf_auc),
        },

        "raw_blend": {
            "log_loss":
                float(raw_blend_ll),
            "roc_auc":
                float(raw_blend_auc),
        },
    },

    "blend_weights": {
        "modernbert":
            float(w_modernbert),

        "structured_prior":
            float(w_structured),

        "tfidf":
            float(w_tfidf),
    },

    "cross_fitted_calibration": {
        "raw_blend": {
            "log_loss":
                float(cf_raw_ll),
            "roc_auc":
                float(cf_raw_auc),
        },

        "platt": {
            "log_loss":
                float(cf_platt_ll),
            "roc_auc":
                float(cf_platt_auc),
        },

        "isotonic": {
            "log_loss":
                float(cf_isotonic_ll),
            "roc_auc":
                float(cf_isotonic_auc),
        },
    },

    "production": {
        "method":
            FINAL_PRODUCTION_METHOD,

        "log_loss":
            float(FINAL_PRODUCTION_LL),

        "roc_auc":
            float(FINAL_PRODUCTION_AUC),
    },

    "training_started": False,

    "gpu_training_required": False,

    "strict_oof": True,
}


with open(
    FINAL_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_metrics,
        f,
        indent=2,
    )

assert FINAL_METRICS_PATH.exists()

print(
    "Final metrics JSON : PASS"
)


# ==============================================================================
# 18. FINAL MANIFEST
# ==============================================================================

final_manifest = {
    "status":
        "PASS",

    "pipeline":
        "Trace The Race",

    "cell":
        14,

    "production_method":
        FINAL_PRODUCTION_METHOD,

    "architecture":
        (
            "ModernBERT 5-fold OOF + "
            "Structured Prior 5-fold OOF + "
            "TF-IDF 5-fold OOF + "
            "optimized raw blend"
        ),

    "population":
        35072,

    "unique_responses":
        35072,

    "folds":
        5,

    "modernbert_oof_log_loss":
        float(modernbert_ll),

    "structured_prior_oof_log_loss":
        float(structured_ll),

    "tfidf_oof_log_loss":
        float(tfidf_ll),

    "raw_blend_oof_log_loss":
        float(raw_blend_ll),

    "raw_blend_oof_roc_auc":
        float(raw_blend_auc),

    "blend_weights": {
        "modernbert":
            float(w_modernbert),

        "structured_prior":
            float(w_structured),

        "tfidf":
            float(w_tfidf),
    },

    "calibration_decision": {
        "raw_blend_cf_log_loss":
            float(cf_raw_ll),

        "platt_cf_log_loss":
            float(cf_platt_ll),

        "isotonic_cf_log_loss":
            float(cf_isotonic_ll),

        "selected":
            "raw_blend",

        "reason":
            (
                "Cross-fitted Platt improvement was "
                "negligible; isotonic was worse. "
                "Raw blend retained for simplicity and "
                "slightly better ranking."
            ),
    },

    "new_training":
        False,

    "modal_required":
        False,

    "final_oof_ready":
        True,
}


with open(
    FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_manifest,
        f,
        indent=2,
    )

assert FINAL_MANIFEST_PATH.exists()

print(
    "Final manifest : PASS"
)


# ==============================================================================
# 19. FINAL STATUS
# ==============================================================================

FINAL_MODEL_LOCKED = True

print("\n" + "=" * 100)
print(
    "TRACE THE RACE — CELL 14 FINAL STATUS"
)
print("=" * 100)

print(
    "ModernBERT OOF             : PASS"
)

print(
    "Structured + Prior OOF     : PASS"
)

print(
    "TF-IDF OOF                 : PASS"
)

print(
    "Exact response alignment   : PASS"
)

print(
    "Exact fold alignment       : PASS"
)

print(
    "Blend weight integrity     : PASS"
)

print(
    "Blend recomputation        : PASS"
)

print(
    "Cross-fitted calibration   : PASS"
)

print(
    "Production method lock     : PASS"
)

print(
    "Final OOF artifact         : PASS"
)

print(
    "Final metrics              : PASS"
)

print(
    "Final manifest             : PASS"
)

print("\n" + "-" * 100)

print(
    "FINAL PRODUCTION METHOD:",
    FINAL_PRODUCTION_METHOD,
)

print(
    "FINAL OOF LOG LOSS:",
    f"{FINAL_PRODUCTION_LL:.12f}",
)

print(
    "FINAL OOF ROC-AUC:",
    f"{FINAL_PRODUCTION_AUC:.12f}",
)

print(
    "FINAL OOF ROWS:",
    f"{len(final_oof):,}",
)

print("-" * 100)

print(
    "\nFinal OOF:",
    FINAL_OOF_PATH,
)

print(
    "Final metrics:",
    FINAL_METRICS_PATH,
)

print(
    "Final manifest:",
    FINAL_MANIFEST_PATH,
)

print(
    "\nCELL 14 COMPLETE — PASS"
)


# ==============================================================================
# 20. MEMORY CLEANUP
# ==============================================================================

del final_reload
del final_oof
del blend_oof
del cross_fitted_oof

del modernbert_prediction
del structured_prediction
del tfidf_prediction
del saved_blend
del recomputed_blend

del cf_raw
del cf_platt
del cf_isotonic

del y
del blend_weights
del cf_metrics

gc.collect()

print(
    "Cell 14 memory cleanup : PASS"
)

TRACE THE RACE — CELL 14 REPAIR
FINAL MODEL / BLEND LOCK

Cell 11 dependency : PASS
Cell 13 dependency : PASS

Required artifacts : PASS

Blend OOF schema : PASS
Rows: 35,072
Cross-fitted OOF schema : PASS

Exact response alignment : PASS
Exact session alignment  : PASS
Exact fold alignment     : PASS
Exact target alignment   : PASS

Prediction numerical contract : PASS

Blend weights : PASS
ModernBERT     : 0.41900000
Structured     : 0.35100000
TF-IDF         : 0.23000000
Weight sum     : 1.000000000000

Blend recomputation : PASS
Maximum difference: 0.000e+00

RECOMPUTED OOF METRICS
ModernBERT       LL: 0.547551451901 | AUC: 0.716413904401
Structured+Prior LL: 0.549264028594 | AUC: 0.713097493283
TF-IDF           LL: 0.558696844276 | AUC: 0.698604824952
Raw Blend        LL: 0.543293053387 | AUC: 0.721930083655

CROSS-FITTED CALIBRATION
Raw blend       LL: 0.543293053387 | AUC: 0.721930083655
Platt           LL: 0.543286106288 | AUC: 0.721860181274
Isotonic        LL: 0.544229056994 

In [23]:
# ==============================================================================
# TRACE THE RACE — CELL 15
# FINAL TEST / SUBMISSION POPULATION DISCOVERY
#
# Purpose:
#   Discover and lock the final inference population after Cell 14.
#
# This cell:
#   - searches project artifacts for test/submission candidates
#   - inspects parquet/csv schemas
#   - identifies response/session identity candidates
#   - checks whether a submission template exists
#   - DOES NOT train anything
#   - DOES NOT run ModernBERT
#   - DOES NOT fit TF-IDF
#   - DOES NOT generate predictions
#
# Important:
#   Ambiguous candidates are NOT silently selected.
# ==============================================================================

from pathlib import Path
import gc
import json

import pandas as pd


print("=" * 100)
print(
    "TRACE THE RACE — CELL 15"
)
print(
    "FINAL TEST / SUBMISSION POPULATION DISCOVERY"
)
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    globals().get(
        "FINAL_MODEL_LOCKED",
        False,
    )
    is True
), (
    "Cell 14 dependency failed. "
    "Run Cell 14 successfully before Cell 15."
)

print(
    "\nCell 14 dependency : PASS"
)


# ==============================================================================
# 2. PROJECT ROOT
# ==============================================================================

assert (
    "PROJECT_ROOT" in globals()
), (
    "PROJECT_ROOT is missing."
)

PROJECT_ROOT = Path(
    PROJECT_ROOT
).resolve()

assert PROJECT_ROOT.exists(), (
    f"Project root does not exist:\n"
    f"{PROJECT_ROOT}"
)

print(
    "Project root : PASS"
)

print(
    "Project root:",
    PROJECT_ROOT,
)


# ==============================================================================
# 3. FINAL MODEL ROOT
# ==============================================================================

FINAL_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "final"
)

assert FINAL_ROOT.exists(), (
    f"Final model root missing:\n"
    f"{FINAL_ROOT}"
)

print(
    "\nFinal model root : PASS"
)


# ==============================================================================
# 4. SEARCH ROOTS
#
# We deliberately search source/raw/data areas, while excluding:
#
#   - modernbert outputs
#   - 09C model artifacts
#   - caches
#   - checkpoints
#
# This prevents accidentally treating an OOF artifact as test data.
# ==============================================================================

SEARCH_ROOTS = [
    PROJECT_ROOT,
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "scratch_mastery_outputs",
]


EXCLUDED_DIR_NAMES = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
    "checkpoints",
    "modernbert_outputs",
    "final",
}


EXCLUDED_PATH_PARTS = {
    "09C",
    "modernbert_outputs",
}


def should_skip_path(
    path: Path,
) -> bool:

    parts_lower = {
        part.lower()
        for part in path.parts
    }

    if (
        parts_lower
        &
        {
            x.lower()
            for x in EXCLUDED_DIR_NAMES
        }
    ):
        return True

    if (
        parts_lower
        &
        {
            x.lower()
            for x in EXCLUDED_PATH_PARTS
        }
    ):
        return True

    return False


# ==============================================================================
# 5. DISCOVER TABULAR FILES
# ==============================================================================

candidate_paths = []

seen_paths = set()

for search_root in SEARCH_ROOTS:

    if not search_root.exists():
        continue

    for path in search_root.rglob("*"):

        if not path.is_file():
            continue

        if should_skip_path(
            path
        ):
            continue

        suffix = (
            path.suffix.lower()
        )

        if suffix not in {
            ".parquet",
            ".csv",
        }:
            continue

        resolved = str(
            path.resolve()
        )

        if resolved in seen_paths:
            continue

        seen_paths.add(
            resolved
        )

        candidate_paths.append(
            path.resolve()
        )


candidate_paths = sorted(
    candidate_paths,
    key=lambda p: (
        str(p).lower()
    ),
)


print("\n" + "=" * 100)
print(
    "TABULAR CANDIDATE DISCOVERY"
)
print("=" * 100)

print(
    "Candidate files:",
    len(candidate_paths),
)


# ==============================================================================
# 6. CANDIDATE INSPECTION
# ==============================================================================

candidate_records = []


IDENTITY_HINTS = {
    "response_id",
    "session_id",
    "target",
    "fold",
    "objective_uid",
}


TEST_NAME_HINTS = (
    "test",
    "submission",
    "submit",
    "inference",
    "predict",
    "holdout",
    "public",
    "private",
)


TRAIN_NAME_HINTS = (
    "train",
    "training",
    "oof",
    "fold",
    "evidence",
    "prior",
    "feature",
    "structured",
    "tfidf",
    "modernbert",
)


for path in candidate_paths:

    record = {
        "path":
            str(path),

        "suffix":
            path.suffix.lower(),

        "rows":
            None,

        "columns":
            None,

        "column_names":
            [],

        "has_response_id":
            False,

        "has_session_id":
            False,

        "has_target":
            False,

        "has_fold":
            False,

        "name_test_score":
            0,

        "name_train_score":
            0,

        "read_status":
            "NOT_READ",

        "read_error":
            None,
    }

    try:

        if (
            path.suffix.lower()
            ==
            ".parquet"
        ):

            table = pd.read_parquet(
                path
            )

        else:

            table = pd.read_csv(
                path,
                nrows=5_000,
            )


        columns = [
            str(c)
            for c in table.columns
        ]

        record[
            "column_names"
        ] = columns

        record[
            "columns"
        ] = len(
            columns
        )

        if (
            path.suffix.lower()
            ==
            ".parquet"
        ):

            record[
                "rows"
            ] = int(
                len(
                    pd.read_parquet(
                        path,
                        columns=[
                            columns[0]
                        ],
                    )
                )
            )

        else:

            record[
                "rows"
            ] = int(
                len(
                    table
                )
            )


        normalized_columns = {
            c.lower()
            for c in columns
        }


        record[
            "has_response_id"
        ] = (
            "response_id"
            in normalized_columns
        )

        record[
            "has_session_id"
        ] = (
            "session_id"
            in normalized_columns
        )

        record[
            "has_target"
        ] = (
            "target"
            in normalized_columns
        )

        record[
            "has_fold"
        ] = (
            "fold"
            in normalized_columns
        )


        name_lower = (
            path.name.lower()
        )

        record[
            "name_test_score"
        ] = sum(
            hint in name_lower
            for hint in TEST_NAME_HINTS
        )

        record[
            "name_train_score"
        ] = sum(
            hint in name_lower
            for hint in TRAIN_NAME_HINTS
        )

        record[
            "read_status"
        ] = "PASS"


    except Exception as exc:

        record[
            "read_status"
        ] = "ERROR"

        record[
            "read_error"
        ] = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )


    candidate_records.append(
        record
    )


candidate_df = pd.DataFrame(
    candidate_records
)


assert (
    len(candidate_df)
    ==
    len(candidate_paths)
)

print(
    "Candidate inspection : PASS"
)


# ==============================================================================
# 7. REMOVE READ ERRORS
# ==============================================================================

readable_candidates = (
    candidate_df[
        candidate_df[
            "read_status"
        ]
        ==
        "PASS"
    ]
    .copy()
)

print(
    "Readable candidates:",
    len(
        readable_candidates
    ),
)


# ==============================================================================
# 8. IDENTITY-BASED CANDIDATES
# ==============================================================================

identity_candidates = (
    readable_candidates[
        (
            readable_candidates[
                "has_response_id"
            ]
        )
        |
        (
            readable_candidates[
                "has_session_id"
            ]
        )
    ]
    .copy()
)


print("\n" + "=" * 100)
print(
    "IDENTITY CANDIDATES"
)
print("=" * 100)

print(
    "Identity candidates:",
    len(
        identity_candidates
    ),
)


# ==============================================================================
# 9. LIKELY TEST CANDIDATES
#
# Strong signal:
#   response_id/session_id
#   NO target
#   NO fold
#
# This is intentionally stricter than filename matching.
# ==============================================================================

likely_test = (
    readable_candidates[
        (
            readable_candidates[
                "has_response_id"
            ]
        )
        &
        (
            ~readable_candidates[
                "has_target"
            ]
        )
        &
        (
            ~readable_candidates[
                "has_fold"
            ]
        )
    ]
    .copy()
)


# Name hints are secondary only.
likely_test[
    "test_candidate_score"
] = (
    likely_test[
        "name_test_score"
    ]
    +
    (
        likely_test[
            "has_response_id"
        ]
        .astype(int)
        * 2
    )
    +
    (
        likely_test[
            "has_session_id"
        ]
        .astype(int)
    )
    -
    (
        likely_test[
            "name_train_score"
        ]
    )
)


likely_test = (
    likely_test
    .sort_values(
        [
            "test_candidate_score",
            "rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "Likely test candidates:",
    len(
        likely_test
    ),
)


# ==============================================================================
# 10. PRINT CANDIDATES
# ==============================================================================

if len(
    likely_test
) > 0:

    display_columns = [
        "path",
        "rows",
        "columns",
        "has_response_id",
        "has_session_id",
        "has_target",
        "has_fold",
        "test_candidate_score",
    ]

    print(
        "\nTop test candidates:"
    )

    print(
        likely_test[
            display_columns
        ]
        .head(30)
        .to_string(
            index=False
        )
    )

else:

    print(
        "\nNo automatic test candidate found."
    )


# ==============================================================================
# 11. SUBMISSION TEMPLATE DISCOVERY
# ==============================================================================

submission_candidates = (
    readable_candidates[
        readable_candidates[
            "path"
        ]
        .str.lower()
        .str.contains(
            "submission"
            "|submit"
            "|sample"
            "|format",
            regex=True,
        )
    ]
    .copy()
)


print("\n" + "=" * 100)
print(
    "SUBMISSION TEMPLATE DISCOVERY"
)
print("=" * 100)

print(
    "Submission-like files:",
    len(
        submission_candidates
    ),
)


if len(
    submission_candidates
) > 0:

    print(
        submission_candidates[
            [
                "path",
                "rows",
                "columns",
                "column_names",
            ]
        ]
        .head(30)
        .to_string(
            index=False
        )
    )

else:

    print(
        "No submission template detected by filename."
    )


# ==============================================================================
# 12. TRAINING-LIKE CANDIDATE GUARD
# ==============================================================================

if len(
    likely_test
) > 0:

    # A likely test candidate must not have target/fold.
    assert not (
        likely_test[
            "has_target"
        ]
        .any()
    )

    assert not (
        likely_test[
            "has_fold"
        ]
        .any()
    )

    print(
        "\nTest candidate target/fold guard : PASS"
    )


# ==============================================================================
# 13. SAVE DISCOVERY AUDIT
# ==============================================================================

CELL15_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "final_test_discovery"
)

CELL15_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DISCOVERY_AUDIT_PATH = (
    CELL15_ROOT
    / "cell15_tabular_discovery.parquet"
)

DISCOVERY_JSON_PATH = (
    CELL15_ROOT
    / "cell15_discovery_summary.json"
)


candidate_df.to_parquet(
    DISCOVERY_AUDIT_PATH,
    index=False,
)

assert (
    DISCOVERY_AUDIT_PATH.exists()
)

print(
    "\nDiscovery audit write : PASS"
)


# ==============================================================================
# 14. IMPORTANT: DO NOT AUTO-LOCK AMBIGUOUS TEST DATA
# ==============================================================================

if len(
    likely_test
) == 0:

    TEST_DISCOVERY_STATUS = (
        "NO_CANDIDATE"
    )

elif len(
    likely_test
) == 1:

    TEST_DISCOVERY_STATUS = (
        "SINGLE_CANDIDATE"
    )

else:

    TEST_DISCOVERY_STATUS = (
        "MULTIPLE_CANDIDATES"
    )


summary = {
    "status":
        "PASS",

    "candidate_files":
        int(
            len(
                candidate_df
            )
        ),

    "readable_candidates":
        int(
            len(
                readable_candidates
            )
        ),

    "identity_candidates":
        int(
            len(
                identity_candidates
            )
        ),

    "likely_test_candidates":
        int(
            len(
                likely_test
            )
        ),

    "submission_like_candidates":
        int(
            len(
                submission_candidates
            )
        ),

    "test_discovery_status":
        TEST_DISCOVERY_STATUS,

    "auto_lock_test":
        False,

    "training_started":
        False,

    "prediction_started":
        False,
}


with open(
    DISCOVERY_JSON_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
    )

assert (
    DISCOVERY_JSON_PATH.exists()
)

print(
    "Discovery summary : PASS"
)


# ==============================================================================
# 15. FINAL STATUS
# ==============================================================================

TEST_DISCOVERY_READY = True

print("\n" + "=" * 100)
print(
    "TRACE THE RACE — CELL 15 FINAL STATUS"
)
print("=" * 100)

print(
    "Cell 14 dependency       : PASS"
)

print(
    "Tabular discovery        : PASS"
)

print(
    "Artifact inspection      : PASS"
)

print(
    "Identity candidate audit : PASS"
)

print(
    "Submission discovery     : PASS"
)

print(
    "Training started         : NO"
)

print(
    "Prediction started       : NO"
)

print(
    "Test discovery status    :",
    TEST_DISCOVERY_STATUS,
)

print(
    "\nDiscovery audit:",
    DISCOVERY_AUDIT_PATH,
)

print(
    "Discovery summary:",
    DISCOVERY_JSON_PATH,
)

print(
    "\nCELL 15 COMPLETE — PASS"
)


# ==============================================================================
# 16. MEMORY CLEANUP
# ==============================================================================

del candidate_df
del readable_candidates
del identity_candidates
del likely_test
del submission_candidates
del candidate_records

gc.collect()

print(
    "Cell 15 memory cleanup : PASS"
)

TRACE THE RACE — CELL 15
FINAL TEST / SUBMISSION POPULATION DISCOVERY

Cell 14 dependency : PASS
Project root : PASS
Project root: D:\Competition\Trace-the-race-local

Final model root : PASS

TABULAR CANDIDATE DISCOVERY
Candidate files: 23221
Candidate inspection : PASS
Readable candidates: 23221

IDENTITY CANDIDATES
Identity candidates: 23206
Likely test candidates: 8

Top test candidates:
                                                                                                                                  path  rows  columns  has_response_id  has_session_id  has_target  has_fold  test_candidate_score
         D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\diagnostics\r3_response_diagnostics.parquet 35072        9             True            True       False     False                     3
D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\diagnostics\r3_response_forensic_diagnostics.parquet 35072       23  

In [24]:
# ==============================================================================
# TRACE THE RACE — CELL 16
# TEST POPULATION FORENSIC DISCOVERY + LOCK
#
# Cell 15 found multiple "test-like" candidates.
#
# This cell does NOT guess.
#
# It determines whether a candidate is actually a new inference population
# by checking:
#
#   1. response_id uniqueness
#   2. overlap with canonical response population
#   3. target/fold presence
#   4. row population
#   5. Dataset/ directory artifacts
#   6. filename semantics
#
# NO MODEL TRAINING
# NO MODERNBERT INFERENCE
# NO TF-IDF FITTING
# NO PREDICTION
# ==============================================================================

from pathlib import Path
import gc
import json
import re

import numpy as np
import pandas as pd


print("=" * 100)
print(
    "TRACE THE RACE — CELL 16"
)
print(
    "TEST POPULATION FORENSIC DISCOVERY + LOCK"
)
print("=" * 100)


# ==============================================================================
# 1. DEPENDENCY
# ==============================================================================

assert (
    globals().get(
        "TEST_DISCOVERY_READY",
        False,
    )
    is True
), (
    "Cell 15 dependency failed. "
    "Run Cell 15 successfully first."
)

assert (
    globals().get(
        "FINAL_MODEL_LOCKED",
        False,
    )
    is True
), (
    "Cell 14 final model is not locked."
)

print(
    "\nCell 14 dependency : PASS"
)

print(
    "Cell 15 dependency : PASS"
)


# ==============================================================================
# 2. PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    PROJECT_ROOT
).resolve()

RESPONSES_PATH = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "01_data_foundation"
    / "03_integrity"
    / "canonical"
    / "responses.parquet"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Dataset"
)

CELL16_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "final_test_discovery"
)

CELL16_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


assert RESPONSES_PATH.exists(), (
    f"Canonical responses missing:\n{RESPONSES_PATH}"
)

assert DATASET_ROOT.exists(), (
    f"Dataset directory missing:\n{DATASET_ROOT}"
)

print(
    "\nCanonical responses : PASS"
)

print(
    "Dataset directory   : PASS"
)


# ==============================================================================
# 3. LOAD CANONICAL RESPONSE IDs
# ==============================================================================

canonical_responses = pd.read_parquet(
    RESPONSES_PATH,
    columns=[
        "response_id",
    ],
)

assert (
    "response_id"
    in canonical_responses.columns
)

canonical_response_ids = set(
    canonical_responses[
        "response_id"
    ]
    .dropna()
    .astype(str)
)

assert (
    len(canonical_response_ids)
    ==
    len(canonical_responses)
)

CANONICAL_RESPONSE_COUNT = (
    len(canonical_response_ids)
)

print("\n" + "=" * 100)
print(
    "CANONICAL RESPONSE POPULATION"
)
print("=" * 100)

print(
    "Canonical rows:",
    f"{CANONICAL_RESPONSE_COUNT:,}",
)

print(
    "Unique response IDs:",
    f"{len(canonical_response_ids):,}",
)

assert (
    CANONICAL_RESPONSE_COUNT
    ==
    35_072
)

print(
    "Canonical population contract : PASS"
)


# ==============================================================================
# 4. DATASET DIRECTORY INVENTORY
# ==============================================================================

dataset_files = []

for path in DATASET_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in {
        ".parquet",
        ".csv",
        ".json",
        ".jsonl",
    }:
        continue

    dataset_files.append(
        path.resolve()
    )

dataset_files = sorted(
    dataset_files,
    key=lambda p: str(p).lower(),
)


print("\n" + "=" * 100)
print(
    "DATASET DIRECTORY INVENTORY"
)
print("=" * 100)

print(
    "Tabular/JSON files:",
    len(dataset_files),
)

for path in dataset_files:
    print(
        " ",
        path.relative_to(
            PROJECT_ROOT
        )
    )


# ==============================================================================
# 5. DATASET FILE INSPECTION
# ==============================================================================

dataset_records = []


def read_small_table(
    path,
):
    suffix = (
        path.suffix.lower()
    )

    if suffix == ".parquet":

        return pd.read_parquet(
            path
        )

    if suffix == ".csv":

        return pd.read_csv(
            path
        )

    if suffix == ".jsonl":

        return pd.read_json(
            path,
            lines=True,
        )

    if suffix == ".json":

        return pd.read_json(
            path
        )

    raise ValueError(
        f"Unsupported file: {path}"
    )


for path in dataset_files:

    record = {
        "path":
            str(path),

        "relative_path":
            str(
                path.relative_to(
                    PROJECT_ROOT
                )
            ),

        "rows":
            None,

        "columns":
            None,

        "column_names":
            [],

        "has_response_id":
            False,

        "has_session_id":
            False,

        "has_target":
            False,

        "has_fold":
            False,

        "has_prediction":
            False,

        "response_id_unique":
            False,

        "canonical_overlap":
            None,

        "canonical_overlap_rate":
            None,

        "name_train":
            False,

        "name_test":
            False,

        "name_submission":
            False,

        "read_status":
            "NOT_READ",

        "error":
            None,
    }

    try:

        table = read_small_table(
            path
        )

        columns = [
            str(c)
            for c in table.columns
        ]

        normalized = {
            c.lower()
            for c in columns
        }

        record[
            "rows"
        ] = int(
            len(table)
        )

        record[
            "columns"
        ] = int(
            len(columns)
        )

        record[
            "column_names"
        ] = columns

        record[
            "has_response_id"
        ] = (
            "response_id"
            in normalized
        )

        record[
            "has_session_id"
        ] = (
            "session_id"
            in normalized
        )

        record[
            "has_target"
        ] = (
            "target"
            in normalized
        )

        record[
            "has_fold"
        ] = (
            "fold"
            in normalized
        )

        record[
            "has_prediction"
        ] = (
            "prediction"
            in normalized
        )

        if (
            record[
                "has_response_id"
            ]
        ):

            response_values = (
                table[
                    [
                        c
                        for c in table.columns
                        if str(c).lower()
                        ==
                        "response_id"
                    ][0]
                ]
                .dropna()
                .astype(str)
            )

            record[
                "response_id_unique"
            ] = bool(
                response_values.is_unique
            )

            candidate_ids = set(
                response_values
            )

            overlap = len(
                candidate_ids
                &
                canonical_response_ids
            )

            record[
                "canonical_overlap"
            ] = int(
                overlap
            )

            if len(
                candidate_ids
            ) > 0:

                record[
                    "canonical_overlap_rate"
                ] = float(
                    overlap
                    /
                    len(candidate_ids)
                )

        name_lower = (
            path.name.lower()
        )

        record[
            "name_train"
        ] = (
            "train"
            in name_lower
        )

        record[
            "name_test"
        ] = (
            "test"
            in name_lower
        )

        record[
            "name_submission"
        ] = (
            "submission"
            in name_lower
            or
            "submit"
            in name_lower
        )

        record[
            "read_status"
        ] = "PASS"

    except Exception as exc:

        record[
            "read_status"
        ] = "ERROR"

        record[
            "error"
        ] = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )

    dataset_records.append(
        record
    )


dataset_df = pd.DataFrame(
    dataset_records
)


assert (
    len(dataset_df)
    ==
    len(dataset_files)
)

print(
    "\nDataset inspection : PASS"
)


# ==============================================================================
# 6. PRINT DATASET SUMMARY
# ==============================================================================

summary_columns = [
    "relative_path",
    "rows",
    "columns",
    "has_response_id",
    "has_session_id",
    "has_target",
    "has_fold",
    "has_prediction",
    "response_id_unique",
    "canonical_overlap",
    "canonical_overlap_rate",
    "name_train",
    "name_test",
    "name_submission",
]


print("\n" + "=" * 100)
print(
    "DATASET ARTIFACT SUMMARY"
)
print("=" * 100)

print(
    dataset_df[
        summary_columns
    ]
    .to_string(
        index=False
    )
)


# ==============================================================================
# 7. STRONG TEST CANDIDATE DEFINITION
#
# A real inference population should normally satisfy:
#
#   response_id exists
#   response_id unique
#   no target
#   no fold
#   not simply the canonical training population
#
# We also reject artifacts that are 100% contained in canonical response IDs.
# ==============================================================================

strong_test_candidates = (
    dataset_df[
        (
            dataset_df[
                "read_status"
            ]
            ==
            "PASS"
        )
        &
        (
            dataset_df[
                "has_response_id"
            ]
        )
        &
        (
            dataset_df[
                "response_id_unique"
            ]
        )
        &
        (
            ~dataset_df[
                "has_target"
            ]
        )
        &
        (
            ~dataset_df[
                "has_fold"
            ]
        )
        &
        (
            dataset_df[
                "canonical_overlap_rate"
            ]
            <
            0.999999
        )
    ]
    .copy()
)


print("\n" + "=" * 100)
print(
    "STRONG TEST CANDIDATES"
)
print("=" * 100)

print(
    "Candidates:",
    len(
        strong_test_candidates
    ),
)

if len(
    strong_test_candidates
) > 0:

    print(
        strong_test_candidates[
            summary_columns
        ]
        .to_string(
            index=False
        )
    )

else:

    print(
        "No strong test population found in Dataset/."
    )


# ==============================================================================
# 8. TRAIN FILE GUARD
# ==============================================================================

train_candidates = (
    dataset_df[
        dataset_df[
            "name_train"
        ]
    ]
    .copy()
)


print("\n" + "=" * 100)
print(
    "TRAIN-NAMED ARTIFACT GUARD"
)
print("=" * 100)

print(
    "Train-named files:",
    len(
        train_candidates
    ),
)

if len(
    train_candidates
) > 0:

    print(
        train_candidates[
            [
                "relative_path",
                "rows",
                "columns",
                "has_response_id",
                "has_target",
                "has_fold",
                "canonical_overlap",
                "canonical_overlap_rate",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ==============================================================================
# 9. EXPLICIT TEST-NAMED ARTIFACTS
# ==============================================================================

test_named_candidates = (
    dataset_df[
        dataset_df[
            "name_test"
        ]
    ]
    .copy()
)


print("\n" + "=" * 100)
print(
    "TEST-NAMED ARTIFACTS"
)
print("=" * 100)

print(
    "Test-named files:",
    len(
        test_named_candidates
    ),
)

if len(
    test_named_candidates
) > 0:

    print(
        test_named_candidates[
            summary_columns
        ]
        .to_string(
            index=False
        )
    )

else:

    print(
        "No file with 'test' in filename."
    )


# ==============================================================================
# 10. SUBMISSION-LIKE ARTIFACTS
# ==============================================================================

submission_candidates = (
    dataset_df[
        dataset_df[
            "name_submission"
        ]
    ]
    .copy()
)


print("\n" + "=" * 100)
print(
    "SUBMISSION-LIKE ARTIFACTS"
)
print("=" * 100)

print(
    "Submission-like files:",
    len(
        submission_candidates
    ),
)

if len(
    submission_candidates
) > 0:

    print(
        submission_candidates[
            summary_columns
        ]
        .to_string(
            index=False
        )
    )

else:

    print(
        "No submission-like artifact in Dataset/."
    )


# ==============================================================================
# 11. CANONICAL OVERLAP INTERPRETATION
# ==============================================================================

print("\n" + "=" * 100)
print(
    "CANONICAL OVERLAP INTERPRETATION"
)
print("=" * 100)

print(
    "Canonical training response population:",
    f"{CANONICAL_RESPONSE_COUNT:,}",
)

print(
    "\nInterpretation:"
)

print(
    "  overlap_rate = 1.0"
)

print(
    "      -> artifact is entirely composed of known "
    "canonical responses."
)

print(
    "  overlap_rate < 1.0"
)

print(
    "      -> artifact contains response IDs not present "
    "in canonical training responses."
)

print(
    "A genuine held-out inference population should "
    "contain response IDs outside the canonical "
    "training population."
)


# ==============================================================================
# 12. LOCK DECISION
#
# NEVER silently select an arbitrary artifact.
# ==============================================================================

if len(
    strong_test_candidates
) == 1:

    TEST_POPULATION_STATUS = (
        "SINGLE_STRONG_CANDIDATE"
    )

elif len(
    strong_test_candidates
) == 0:

    TEST_POPULATION_STATUS = (
        "NO_STRONG_CANDIDATE"
    )

else:

    TEST_POPULATION_STATUS = (
        "MULTIPLE_STRONG_CANDIDATES"
    )


print("\n" + "=" * 100)
print(
    "TEST POPULATION LOCK DECISION"
)
print("=" * 100)

print(
    "Status:",
    TEST_POPULATION_STATUS
)


# ==============================================================================
# 13. SAVE FORENSIC AUDIT
# ==============================================================================

DATASET_AUDIT_PATH = (
    CELL16_ROOT
    / "cell16_dataset_forensic.parquet"
)

DATASET_SUMMARY_PATH = (
    CELL16_ROOT
    / "cell16_dataset_forensic.json"
)


dataset_df.to_parquet(
    DATASET_AUDIT_PATH,
    index=False,
)

assert (
    DATASET_AUDIT_PATH.exists()
)

print(
    "\nDataset forensic audit : PASS"
)


cell16_summary = {
    "status":
        "PASS",

    "canonical_response_count":
        int(
            CANONICAL_RESPONSE_COUNT
        ),

    "dataset_file_count":
        int(
            len(dataset_df)
        ),

    "strong_test_candidate_count":
        int(
            len(
                strong_test_candidates
            )
        ),

    "test_population_status":
        TEST_POPULATION_STATUS,

    "training_started":
        False,

    "prediction_started":
        False,

    "auto_locked":
        False,
}


with open(
    DATASET_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell16_summary,
        f,
        indent=2,
    )

assert (
    DATASET_SUMMARY_PATH.exists()
)

print(
    "Dataset forensic summary : PASS"
)


# ==============================================================================
# 14. FINAL STATUS
# ==============================================================================

TEST_FORENSIC_READY = True

print("\n" + "=" * 100)
print(
    "TRACE THE RACE — CELL 16 FINAL STATUS"
)
print("=" * 100)

print(
    "Cell 14 dependency           : PASS"
)

print(
    "Cell 15 dependency           : PASS"
)

print(
    "Canonical response audit     : PASS"
)

print(
    "Dataset inventory            : PASS"
)

print(
    "Response overlap analysis    : PASS"
)

print(
    "Target/fold guard            : PASS"
)

print(
    "Train/test filename audit    : PASS"
)

print(
    "Test population auto-lock    : NO"
)

print(
    "Training started             : NO"
)

print(
    "Prediction started           : NO"
)

print(
    "\nTest population status:",
    TEST_POPULATION_STATUS,
)

print(
    "\nForensic audit:",
    DATASET_AUDIT_PATH,
)

print(
    "Forensic summary:",
    DATASET_SUMMARY_PATH,
)

print(
    "\nCELL 16 COMPLETE — PASS"
)


# ==============================================================================
# 15. MEMORY CLEANUP
# ==============================================================================

del dataset_df
del dataset_records
del strong_test_candidates
del train_candidates
del test_named_candidates
del submission_candidates
del canonical_responses

gc.collect()

print(
    "Cell 16 memory cleanup : PASS"
)

TRACE THE RACE — CELL 16
TEST POPULATION FORENSIC DISCOVERY + LOCK

Cell 14 dependency : PASS
Cell 15 dependency : PASS

Canonical responses : PASS
Dataset directory   : PASS

CANONICAL RESPONSE POPULATION
Canonical rows: 35,072
Unique response IDs: 35,072
Canonical population contract : PASS

DATASET DIRECTORY INVENTORY
Tabular/JSON files: 22823
  Dataset\train_features_TMQTWsB.csv
  Dataset\train_labels_44ujmj2.csv
  Dataset\train_transcripts\aaaedit.csv
  Dataset\train_transcripts\aaaptjd.csv
  Dataset\train_transcripts\aabkeov.csv
  Dataset\train_transcripts\aacggvb.csv
  Dataset\train_transcripts\aadexbc.csv
  Dataset\train_transcripts\aadinwu.csv
  Dataset\train_transcripts\aadljmq.csv
  Dataset\train_transcripts\aadmino.csv
  Dataset\train_transcripts\aadsgow.csv
  Dataset\train_transcripts\aadylxv.csv
  Dataset\train_transcripts\aaeovaj.csv
  Dataset\train_transcripts\aaeqxpf.csv
  Dataset\train_transcripts\aafcpue.csv
  Dataset\train_transcripts\aafitff.csv
  Dataset\train_tra